In [1]:
# Cell 1 — Install retrieval experiment dependencies

!pip -q install -U sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 47.2 MB/s eta 0:00:00


In [ ]:
# Cell 2 — Verify experimental environment

import sys
import torch
import numpy as np
import pandas as pd
import sentence_transformers
import faiss

print("EXPERIMENT ENVIRONMENT")
print("=" * 60)

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("FAISS:", faiss.__version__)

print("\nHARDWARE")
print("-" * 60)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    DEVICE = "cuda"
else:
    print("GPU: None")
    DEVICE = "cpu"

print("Selected device:", DEVICE)

print("\nSTATUS")
print("=" * 60)
print("PASS — Retrieval experiment environment is ready.")

EXPERIMENT ENVIRONMENT
Python: 3.13.15
PyTorch: 2.11.0+cpu
Sentence Transformers: 6.1.0
NumPy: 2.1.3
Pandas: 2.2.3
FAISS: 1.15.1

HARDWARE
------------------------------------------------------------
CUDA available: False
GPU: None
Selected device: cpu

STATUS
PASS — Retrieval experiment environment is ready.


In [ ]:
# Cell 3 — Define the frozen embedding model registry

EMBEDDING_MODELS = {
    "E1": {
        "model_name": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
        "query_prefix": "",
        "passage_prefix": ""
    },
    "E2": {
        "model_name": "intfloat/multilingual-e5-base",
        "query_prefix": "query: ",
        "passage_prefix": "passage: "
    },
    "E3": {
        "model_name": "BAAI/bge-m3",
        "query_prefix": "",
        "passage_prefix": ""
    }
}

print("FROZEN EMBEDDING MODEL REGISTRY")
print("=" * 70)

for model_id, config in EMBEDDING_MODELS.items():
    print(f"{model_id}: {config['model_name']}")
    print(f"   Query prefix: {repr(config['query_prefix'])}")
    print(f"   Passage prefix: {repr(config['passage_prefix'])}")

print("\nSTATUS")
print("=" * 70)
print("PASS — E1, E2 and E3 are registered for the experiment.")

FROZEN EMBEDDING MODEL REGISTRY
E1: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
   Query prefix: ''
   Passage prefix: ''
E2: intfloat/multilingual-e5-base
   Query prefix: 'query: '
   Passage prefix: 'passage: '
E3: BAAI/bge-m3
   Query prefix: ''
   Passage prefix: ''

STATUS
PASS — E1, E2 and E3 are registered for the experiment.


In [ ]:
# Cell 4 — Load and validate E1 only

from sentence_transformers import SentenceTransformer
import numpy as np

E1_ID = "E1"
E1_NAME = EMBEDDING_MODELS[E1_ID]["model_name"]

print("E1 MODEL VALIDATION")
print("=" * 70)

print("Model ID:", E1_ID)
print("Model:", E1_NAME)
print("Device:", DEVICE)

# Load model
e1_model = SentenceTransformer(
    E1_NAME,
    device=DEVICE
)

print("\nMODEL LOADED")
print("-" * 70)

print(
    "Embedding dimension:",
    e1_model.get_sentence_embedding_dimension()
)

print(
    "Max sequence length:",
    e1_model.max_seq_length
)

# ---------------------------------------------------------
# Small Arabic encoding test
# ---------------------------------------------------------

test_texts = [
    "ماذا يحدث للمادة الوراثية DNA قبل انقسام الخلية؟",
    "تتضاعف مادة الوراثة DNA قبل البدء بعملية الانقسام."
]

test_embeddings = e1_model.encode(
    test_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False
)

print("\nENCODING TEST")
print("-" * 70)

print("Input texts:", len(test_texts))
print("Embedding matrix shape:", test_embeddings.shape)
print("Data type:", test_embeddings.dtype)

finite_check = np.isfinite(test_embeddings).all()

norms = np.linalg.norm(
    test_embeddings,
    axis=1
)

print("All values finite:", finite_check)
print(
    "Embedding norms:",
    np.round(norms, 6).tolist()
)

# ---------------------------------------------------------
# Final validation
# ---------------------------------------------------------

expected_dimension = (
    e1_model.get_sentence_embedding_dimension()
)

valid = (
    test_embeddings.shape ==
    (len(test_texts), expected_dimension)
    and finite_check
    and np.allclose(norms, 1.0, atol=1e-5)
)

print("\nFINAL E1 VALIDATION")
print("=" * 70)

if valid:
    print(
        "PASS — E1 loaded successfully and produced "
        "valid normalized Arabic embeddings."
    )
else:
    print(
        "FAIL — E1 validation requires review before "
        "continuing."
    )

E1 MODEL VALIDATION
Model ID: E1
Model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Device: cpu


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


MODEL LOADED
----------------------------------------------------------------------
Embedding dimension: 384
Max sequence length: 128


/tmp/ipykernel_2416/1266202845.py:27: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  e1_model.get_sentence_embedding_dimension()



ENCODING TEST
----------------------------------------------------------------------
Input texts: 2
Embedding matrix shape: (2, 384)
Data type: float32
All values finite: True
Embedding norms: [1.0, 1.0]

FINAL E1 VALIDATION
PASS — E1 loaded successfully and produced valid normalized Arabic embeddings.


/tmp/ipykernel_2416/1266202845.py:76: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  e1_model.get_sentence_embedding_dimension()


In [ ]:
# Cell 5 — Load and validate E2 with asymmetric retrieval prefixes

from sentence_transformers import SentenceTransformer
import numpy as np

E2_ID = "E2"
E2_NAME = EMBEDDING_MODELS[E2_ID]["model_name"]

QUERY_PREFIX = EMBEDDING_MODELS[E2_ID]["query_prefix"]
PASSAGE_PREFIX = EMBEDDING_MODELS[E2_ID]["passage_prefix"]

print("E2 MODEL VALIDATION")
print("=" * 70)

print("Model ID:", E2_ID)
print("Model:", E2_NAME)
print("Device:", DEVICE)

print("\nENCODING POLICY")
print("-" * 70)
print("Query prefix:", repr(QUERY_PREFIX))
print("Passage prefix:", repr(PASSAGE_PREFIX))

# ---------------------------------------------------------
# Load model
# ---------------------------------------------------------

e2_model = SentenceTransformer(
    E2_NAME,
    device=DEVICE
)

print("\nMODEL LOADED")
print("-" * 70)

print(
    "Embedding dimension:",
    e2_model.get_embedding_dimension()
)

print(
    "Max sequence length:",
    e2_model.max_seq_length
)

# ---------------------------------------------------------
# Technical Arabic smoke test
# ---------------------------------------------------------

raw_query = (
    "ماذا يحدث للمادة الوراثية DNA قبل انقسام الخلية؟"
)

raw_passage = (
    "تتضاعف مادة الوراثة DNA قبل البدء بعملية الانقسام."
)

formatted_query = QUERY_PREFIX + raw_query
formatted_passage = PASSAGE_PREFIX + raw_passage

print("\nFORMATTED INPUT CHECK")
print("-" * 70)

print("Formatted query:")
print(formatted_query)

print("\nFormatted passage:")
print(formatted_passage)

# ---------------------------------------------------------
# Encode
# ---------------------------------------------------------

query_embedding = e2_model.encode(
    [formatted_query],
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False
)

passage_embedding = e2_model.encode(
    [formatted_passage],
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False
)

print("\nENCODING TEST")
print("-" * 70)

print(
    "Query embedding shape:",
    query_embedding.shape
)

print(
    "Passage embedding shape:",
    passage_embedding.shape
)

print(
    "Query values finite:",
    np.isfinite(query_embedding).all()
)

print(
    "Passage values finite:",
    np.isfinite(passage_embedding).all()
)

query_norm = np.linalg.norm(
    query_embedding,
    axis=1
)

passage_norm = np.linalg.norm(
    passage_embedding,
    axis=1
)

print(
    "Query norm:",
    np.round(query_norm, 6).tolist()
)

print(
    "Passage norm:",
    np.round(passage_norm, 6).tolist()
)

# ---------------------------------------------------------
# Diagnostic similarity only
# ---------------------------------------------------------

diagnostic_similarity = float(
    query_embedding @ passage_embedding.T
)

print("\nDIAGNOSTIC CHECK")
print("-" * 70)

print(
    "Query-passage cosine similarity:",
    round(diagnostic_similarity, 6)
)

print(
    "Note: this value is diagnostic only and is not "
    "used to evaluate or select the embedding model."
)

# ---------------------------------------------------------
# Final validation
# ---------------------------------------------------------

expected_dimension = (
    e2_model.get_embedding_dimension()
)

valid = (
    formatted_query.startswith("query: ")
    and formatted_passage.startswith("passage: ")
    and query_embedding.shape == (1, expected_dimension)
    and passage_embedding.shape == (1, expected_dimension)
    and np.isfinite(query_embedding).all()
    and np.isfinite(passage_embedding).all()
    and np.allclose(query_norm, 1.0, atol=1e-5)
    and np.allclose(passage_norm, 1.0, atol=1e-5)
)

print("\nFINAL E2 VALIDATION")
print("=" * 70)

if valid:
    print(
        "PASS — E2 loaded successfully and produced "
        "valid normalized Arabic query and passage embeddings "
        "using the frozen E5 prefixes."
    )
else:
    print(
        "FAIL — E2 validation requires review before continuing."
    )

E2 MODEL VALIDATION
Model ID: E2
Model: intfloat/multilingual-e5-base
Device: cpu

ENCODING POLICY
----------------------------------------------------------------------
Query prefix: 'query: '
Passage prefix: 'passage: '


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]


MODEL LOADED
----------------------------------------------------------------------
Embedding dimension: 768
Max sequence length: 512

FORMATTED INPUT CHECK
----------------------------------------------------------------------
Formatted query:
query: ماذا يحدث للمادة الوراثية DNA قبل انقسام الخلية؟

Formatted passage:
passage: تتضاعف مادة الوراثة DNA قبل البدء بعملية الانقسام.

ENCODING TEST
----------------------------------------------------------------------
Query embedding shape: (1, 768)
Passage embedding shape: (1, 768)
Query values finite: True
Passage values finite: True
Query norm: [1.0]
Passage norm: [1.0]

DIAGNOSTIC CHECK
----------------------------------------------------------------------
Query-passage cosine similarity: 0.879468
Note: this value is diagnostic only and is not used to evaluate or select the embedding model.

FINAL E2 VALIDATION
PASS — E2 loaded successfully and produced valid normalized Arabic query and passage embeddings using the frozen E5 prefixes.


/tmp/ipykernel_2416/3907418029.py:135: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  diagnostic_similarity = float(


In [ ]:
# Cell 6 — Load and validate E3 (BGE-M3)

from sentence_transformers import SentenceTransformer
import numpy as np

E3_ID = "E3"
E3_NAME = EMBEDDING_MODELS[E3_ID]["model_name"]

print("E3 MODEL VALIDATION")
print("=" * 70)

print("Model ID:", E3_ID)
print("Model:", E3_NAME)
print("Device:", DEVICE)

# ---------------------------------------------------------
# Load model
# ---------------------------------------------------------

e3_model = SentenceTransformer(
    E3_NAME,
    device=DEVICE
)

print("\nMODEL LOADED")
print("-" * 70)

print(
    "Embedding dimension:",
    e3_model.get_embedding_dimension()
)

print(
    "Max sequence length:",
    e3_model.max_seq_length
)

# ---------------------------------------------------------
# Technical Arabic smoke test
# ---------------------------------------------------------

test_query = (
    "ماذا يحدث للمادة الوراثية DNA قبل انقسام الخلية؟"
)

test_passage = (
    "تتضاعف مادة الوراثة DNA قبل البدء بعملية الانقسام."
)

print("\nINPUT CHECK")
print("-" * 70)

print("Query:")
print(test_query)

print("\nPassage:")
print(test_passage)

# ---------------------------------------------------------
# Encode
# ---------------------------------------------------------

query_embedding = e3_model.encode(
    [test_query],
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False
)

passage_embedding = e3_model.encode(
    [test_passage],
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False
)

print("\nENCODING TEST")
print("-" * 70)

print(
    "Query embedding shape:",
    query_embedding.shape
)

print(
    "Passage embedding shape:",
    passage_embedding.shape
)

print(
    "Query values finite:",
    np.isfinite(query_embedding).all()
)

print(
    "Passage values finite:",
    np.isfinite(passage_embedding).all()
)

query_norm = np.linalg.norm(
    query_embedding,
    axis=1
)

passage_norm = np.linalg.norm(
    passage_embedding,
    axis=1
)

print(
    "Query norm:",
    np.round(query_norm, 6).tolist()
)

print(
    "Passage norm:",
    np.round(passage_norm, 6).tolist()
)

# ---------------------------------------------------------
# Diagnostic similarity only
# ---------------------------------------------------------

diagnostic_similarity = (
    query_embedding @ passage_embedding.T
)[0, 0]

print("\nDIAGNOSTIC CHECK")
print("-" * 70)

print(
    "Query-passage cosine similarity:",
    round(float(diagnostic_similarity), 6)
)

print(
    "Note: this value is diagnostic only and is not "
    "used to evaluate or select the embedding model."
)

# ---------------------------------------------------------
# Final validation
# ---------------------------------------------------------

expected_dimension = (
    e3_model.get_embedding_dimension()
)

valid = (
    query_embedding.shape == (1, expected_dimension)
    and passage_embedding.shape == (1, expected_dimension)
    and np.isfinite(query_embedding).all()
    and np.isfinite(passage_embedding).all()
    and np.allclose(query_norm, 1.0, atol=1e-5)
    and np.allclose(passage_norm, 1.0, atol=1e-5)
)

print("\nFINAL E3 VALIDATION")
print("=" * 70)

if valid:
    print(
        "PASS — E3 loaded successfully and produced "
        "valid normalized Arabic embeddings."
    )
else:
    print(
        "FAIL — E3 validation requires review before continuing."
    )

E3 MODEL VALIDATION
Model ID: E3
Model: BAAI/bge-m3
Device: cpu


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]


MODEL LOADED
----------------------------------------------------------------------
Embedding dimension: 1024
Max sequence length: 8192

INPUT CHECK
----------------------------------------------------------------------
Query:
ماذا يحدث للمادة الوراثية DNA قبل انقسام الخلية؟

Passage:
تتضاعف مادة الوراثة DNA قبل البدء بعملية الانقسام.

ENCODING TEST
----------------------------------------------------------------------
Query embedding shape: (1, 1024)
Passage embedding shape: (1, 1024)
Query values finite: True
Passage values finite: True
Query norm: [1.0]
Passage norm: [1.0]

DIAGNOSTIC CHECK
----------------------------------------------------------------------
Query-passage cosine similarity: 0.726325
Note: this value is diagnostic only and is not used to evaluate or select the embedding model.

FINAL E3 VALIDATION
PASS — E3 loaded successfully and produced valid normalized Arabic embeddings.


In [ ]:
# Cell 7 — Validate the frozen experimental inputs before retrieval

import json
import pandas as pd
from pathlib import Path

CHUNKS_FILE = Path(
    "/content/experimental_chunk_representations.csv"
)

QUERIES_FILE = Path(
    "/content/retrieval_queries.json"
)

print("PRE-EXECUTION INPUT VALIDATION")
print("=" * 75)

# =========================================================
# 1. Check files
# =========================================================

print("Chunks file exists:", CHUNKS_FILE.exists())
print("Queries file exists:", QUERIES_FILE.exists())

if not CHUNKS_FILE.exists():
    raise FileNotFoundError(CHUNKS_FILE)

if not QUERIES_FILE.exists():
    raise FileNotFoundError(QUERIES_FILE)


# =========================================================
# 2. Load frozen inputs
# =========================================================

chunks_df = pd.read_csv(CHUNKS_FILE)

with open(
    QUERIES_FILE,
    "r",
    encoding="utf-8"
) as f:
    query_data = json.load(f)

queries = query_data["queries"]


# =========================================================
# 3. Validate required chunk columns
# =========================================================

required_chunk_columns = {
    "configuration",
    "overlap_setting",
    "chunk_id",
    "title",
    "core_source_unit_ids",
    "overlap_source_unit_ids",
    "retrieval_source_unit_ids",
    "core_word_count",
    "retrieval_word_count",
    "core_text",
    "retrieval_text"
}

missing_columns = (
    required_chunk_columns -
    set(chunks_df.columns)
)

print("\nCHUNK FILE STRUCTURE")
print("-" * 75)

print("Rows:", len(chunks_df))
print("Columns:", list(chunks_df.columns))

print(
    "Missing required columns:",
    sorted(missing_columns)
    if missing_columns
    else "None"
)


# =========================================================
# 4. Validate configurations and overlap settings
# =========================================================

print("\nCONFIGURATION × OVERLAP COUNTS")
print("-" * 75)

counts = (
    chunks_df
    .groupby(
        ["configuration", "overlap_setting"]
    )
    .size()
)

print(counts)

expected_counts = {
    ("A_Fine", "O0"): 17,
    ("A_Fine", "O1"): 17,

    ("B_Baseline", "O0"): 11,
    ("B_Baseline", "O1"): 11,

    ("C_Broad", "O0"): 9,
    ("C_Broad", "O1"): 9
}

count_check = True

for key, expected in expected_counts.items():

    actual = counts.get(key, 0)

    if actual != expected:
        count_check = False
        print(
            f"Mismatch {key}: "
            f"expected {expected}, got {actual}"
        )


# =========================================================
# 5. Validate retrieval text
# =========================================================

missing_retrieval_text = (
    chunks_df["retrieval_text"]
    .isna()
    .sum()
)

empty_retrieval_text = (
    chunks_df["retrieval_text"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

print("\nRETRIEVAL TEXT CHECK")
print("-" * 75)

print(
    "Missing retrieval texts:",
    missing_retrieval_text
)

print(
    "Empty retrieval texts:",
    empty_retrieval_text
)


# =========================================================
# 6. Validate unique experimental representations
# =========================================================

duplicate_representation_keys = (
    chunks_df.duplicated(
        subset=[
            "configuration",
            "overlap_setting",
            "chunk_id"
        ]
    ).sum()
)

print("\nREPRESENTATION KEY CHECK")
print("-" * 75)

print(
    "Duplicate configuration/overlap/chunk keys:",
    duplicate_representation_keys
)


# =========================================================
# 7. Validate query set
# =========================================================

query_ids = [
    q["query_id"]
    for q in queries
]

expected_query_ids = [
    f"Q{i:02d}"
    for i in range(1, 13)
]

print("\nQUERY SET CHECK")
print("-" * 75)

print("Number of queries:", len(queries))
print("Query IDs:", query_ids)


# =========================================================
# 8. Calculate frozen experimental matrix
# =========================================================

number_of_chunk_overlap_conditions = 6
number_of_embedding_models = 3
number_of_queries = 12
top_k = 3

retrieval_conditions = (
    number_of_chunk_overlap_conditions
    * number_of_embedding_models
)

query_condition_runs = (
    retrieval_conditions
    * number_of_queries
)

ranked_records = (
    query_condition_runs
    * top_k
)

print("\nFROZEN EXPERIMENTAL MATRIX")
print("-" * 75)

print(
    "Chunk × overlap conditions:",
    number_of_chunk_overlap_conditions
)

print(
    "Embedding models:",
    number_of_embedding_models
)

print(
    "Total retrieval conditions:",
    retrieval_conditions
)

print(
    "Queries per condition:",
    number_of_queries
)

print(
    "Total query-condition runs:",
    query_condition_runs
)

print(
    "Top-k:",
    top_k
)

print(
    "Expected ranked retrieval records:",
    ranked_records
)


# =========================================================
# 9. Final validation
# =========================================================

valid = (
    len(chunks_df) == 74
    and not missing_columns
    and count_check
    and missing_retrieval_text == 0
    and empty_retrieval_text == 0
    and duplicate_representation_keys == 0
    and len(queries) == 12
    and query_ids == expected_query_ids
    and retrieval_conditions == 18
    and query_condition_runs == 216
    and ranked_records == 648
)

print("\nFINAL PRE-EXECUTION VALIDATION")
print("=" * 75)

if valid:
    print(
        "PASS — Frozen experimental inputs are internally "
        "consistent and ready for retrieval execution."
    )
else:
    print(
        "FAIL — Resolve the input validation issues before "
        "running retrieval."
    )

PRE-EXECUTION INPUT VALIDATION
Chunks file exists: True
Queries file exists: True

CHUNK FILE STRUCTURE
---------------------------------------------------------------------------
Rows: 74
Columns: ['configuration', 'overlap_setting', 'chunk_id', 'title', 'core_source_unit_ids', 'overlap_source_unit_ids', 'retrieval_source_unit_ids', 'core_word_count', 'retrieval_word_count', 'core_text', 'retrieval_text']
Missing required columns: None

CONFIGURATION × OVERLAP COUNTS
---------------------------------------------------------------------------
configuration  overlap_setting
A_Fine         O0                 17
               O1                 17
B_Baseline     O0                 11
               O1                 11
C_Broad        O0                  9
               O1                  9
dtype: int64

RETRIEVAL TEXT CHECK
---------------------------------------------------------------------------
Missing retrieval texts: 0
Empty retrieval texts: 0

REPRESENTATION KEY CHECK
---------

In [ ]:
# Cell 8 — Define retrieval execution functions

import gc
import time
import numpy as np
import pandas as pd
import faiss
import torch
from sentence_transformers import SentenceTransformer


TOP_K = 3


def format_texts_for_model(
    model_id,
    texts,
    text_type
):
    """
    Apply the frozen model-specific input formatting.

    text_type:
        "query"   -> retrieval query
        "passage" -> curriculum chunk
    """

    config = EMBEDDING_MODELS[model_id]

    if text_type == "query":
        prefix = config["query_prefix"]
    elif text_type == "passage":
        prefix = config["passage_prefix"]
    else:
        raise ValueError(
            "text_type must be 'query' or 'passage'"
        )

    return [
        prefix + str(text)
        for text in texts
    ]


def build_faiss_index(embeddings):
    """
    Build an inner-product FAISS index.

    Because embeddings are L2-normalized,
    inner product corresponds to cosine similarity.
    """

    embeddings = np.asarray(
        embeddings,
        dtype="float32"
    )

    dimension = embeddings.shape[1]

    index = faiss.IndexFlatIP(dimension)
    index.add(embeddings)

    return index


def run_single_condition(
    model_id,
    model,
    condition_df,
    queries,
    configuration,
    overlap_setting,
    top_k=3
):
    """
    Run retrieval for one:
    configuration × overlap × embedding model condition.
    """

    condition_df = (
        condition_df
        .reset_index(drop=True)
        .copy()
    )

    passage_texts = (
        condition_df["retrieval_text"]
        .astype(str)
        .tolist()
    )

    formatted_passages = format_texts_for_model(
        model_id=model_id,
        texts=passage_texts,
        text_type="passage"
    )

    # Encode curriculum retrieval representations
    passage_embeddings = model.encode(
        formatted_passages,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    passage_embeddings = np.asarray(
        passage_embeddings,
        dtype="float32"
    )

    index = build_faiss_index(
        passage_embeddings
    )

    raw_records = []

    for query in queries:

        formatted_query = format_texts_for_model(
            model_id=model_id,
            texts=[query["query_ar"]],
            text_type="query"
        )

        query_embedding = model.encode(
            formatted_query,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False
        )

        query_embedding = np.asarray(
            query_embedding,
            dtype="float32"
        )

        scores, indices = index.search(
            query_embedding,
            top_k
        )

        for rank_position in range(top_k):

            retrieved_index = int(
                indices[0, rank_position]
            )

            similarity_score = float(
                scores[0, rank_position]
            )

            retrieved_row = condition_df.iloc[
                retrieved_index
            ]

            raw_records.append({
                "model_id": model_id,
                "model_name": (
                    EMBEDDING_MODELS[
                        model_id
                    ]["model_name"]
                ),
                "configuration": configuration,
                "overlap_setting": overlap_setting,
                "query_id": query["query_id"],
                "query_type": query["query_type"],
                "query_ar": query["query_ar"],
                "rank": rank_position + 1,
                "chunk_id": retrieved_row["chunk_id"],
                "chunk_title": retrieved_row["title"],
                "similarity_score": similarity_score,
                "core_source_unit_ids": (
                    retrieved_row[
                        "core_source_unit_ids"
                    ]
                ),
                "overlap_source_unit_ids": (
                    retrieved_row[
                        "overlap_source_unit_ids"
                    ]
                ),
                "retrieval_source_unit_ids": (
                    retrieved_row[
                        "retrieval_source_unit_ids"
                    ]
                ),
                "retrieval_text": (
                    retrieved_row[
                        "retrieval_text"
                    ]
                )
            })

    return raw_records

In [ ]:
# Cell 9 — Execute the full frozen retrieval experiment

from pathlib import Path

RAW_RESULTS_FILE = Path(
    "/content/raw_retrieval_results.csv"
)

RUN_SUMMARY_FILE = Path(
    "/content/retrieval_run_summary.csv"
)

all_raw_records = []
run_summaries = []

condition_order = [
    ("A_Fine", "O0"),
    ("A_Fine", "O1"),
    ("B_Baseline", "O0"),
    ("B_Baseline", "O1"),
    ("C_Broad", "O0"),
    ("C_Broad", "O1")
]

model_order = [
    "E1",
    "E2",
    "E3"
]

print("FULL RETRIEVAL EXPERIMENT")
print("=" * 80)

print("Models:", model_order)
print("Chunk × overlap conditions:", len(condition_order))
print("Queries:", len(queries))
print("Top-k:", TOP_K)
print(
    "Expected ranked records:",
    18 * len(queries) * TOP_K
)

experiment_start = time.time()


# =========================================================
# Execute one model at a time
# =========================================================

for model_id in model_order:

    model_name = (
        EMBEDDING_MODELS[
            model_id
        ]["model_name"]
    )

    print("\n" + "=" * 80)
    print(
        f"LOADING {model_id}: {model_name}"
    )
    print("=" * 80)

    model_start = time.time()

    # Reuse already-loaded models when available
    if model_id == "E1" and "e1_model" in globals():
        model = e1_model

    elif model_id == "E2" and "e2_model" in globals():
        model = e2_model

    elif model_id == "E3" and "e3_model" in globals():
        model = e3_model

    else:
        model = SentenceTransformer(
            model_name,
            device=DEVICE
        )

    print(
        "Embedding dimension:",
        model.get_embedding_dimension()
    )


    # =====================================================
    # Six chunk × overlap conditions
    # =====================================================

    for configuration, overlap_setting in condition_order:

        condition_start = time.time()

        condition_df = chunks_df[
            (
                chunks_df["configuration"]
                == configuration
            )
            &
            (
                chunks_df["overlap_setting"]
                == overlap_setting
            )
        ].copy()

        print(
            f"\nRunning "
            f"{configuration} × "
            f"{overlap_setting} × "
            f"{model_id}"
        )

        print(
            "Chunks:",
            len(condition_df)
        )

        condition_records = run_single_condition(
            model_id=model_id,
            model=model,
            condition_df=condition_df,
            queries=queries,
            configuration=configuration,
            overlap_setting=overlap_setting,
            top_k=TOP_K
        )

        all_raw_records.extend(
            condition_records
        )

        condition_seconds = (
            time.time() -
            condition_start
        )

        run_summaries.append({
            "model_id": model_id,
            "model_name": model_name,
            "configuration": configuration,
            "overlap_setting": overlap_setting,
            "number_of_chunks": len(condition_df),
            "number_of_queries": len(queries),
            "top_k": TOP_K,
            "ranked_records": len(condition_records),
            "runtime_seconds": round(
                condition_seconds,
                3
            )
        })

        print(
            "Ranked records:",
            len(condition_records)
        )

        print(
            "Runtime seconds:",
            round(condition_seconds, 2)
        )


    model_seconds = (
        time.time() -
        model_start
    )

    print(
        f"\n{model_id} completed in "
        f"{round(model_seconds, 2)} seconds."
    )

    # Save progress after every model.
    pd.DataFrame(
        all_raw_records
    ).to_csv(
        RAW_RESULTS_FILE,
        index=False,
        encoding="utf-8-sig"
    )

    pd.DataFrame(
        run_summaries
    ).to_csv(
        RUN_SUMMARY_FILE,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        "Progress saved after",
        model_id
    )


# =========================================================
# Final save
# =========================================================

raw_results_df = pd.DataFrame(
    all_raw_records
)

run_summary_df = pd.DataFrame(
    run_summaries
)

raw_results_df.to_csv(
    RAW_RESULTS_FILE,
    index=False,
    encoding="utf-8-sig"
)

run_summary_df.to_csv(
    RUN_SUMMARY_FILE,
    index=False,
    encoding="utf-8-sig"
)

experiment_seconds = (
    time.time() -
    experiment_start
)


# =========================================================
# Raw execution validation
# =========================================================

expected_conditions = 18
expected_query_runs = 216
expected_records = 648

actual_conditions = len(
    run_summary_df
)

actual_query_runs = (
    raw_results_df[
        [
            "model_id",
            "configuration",
            "overlap_setting",
            "query_id"
        ]
    ]
    .drop_duplicates()
    .shape[0]
)

actual_records = len(
    raw_results_df
)

duplicate_rank_keys = (
    raw_results_df
    .duplicated(
        subset=[
            "model_id",
            "configuration",
            "overlap_setting",
            "query_id",
            "rank"
        ]
    )
    .sum()
)

print("\n" + "=" * 80)
print("RAW RETRIEVAL EXECUTION VALIDATION")
print("=" * 80)

print(
    "Completed conditions:",
    actual_conditions,
    "/",
    expected_conditions
)

print(
    "Completed query-condition runs:",
    actual_query_runs,
    "/",
    expected_query_runs
)

print(
    "Ranked retrieval records:",
    actual_records,
    "/",
    expected_records
)

print(
    "Duplicate rank keys:",
    duplicate_rank_keys
)

print(
    "Total runtime seconds:",
    round(experiment_seconds, 2)
)

print("\nSaved:")
print(RAW_RESULTS_FILE)
print(RUN_SUMMARY_FILE)


valid_execution = (
    actual_conditions == expected_conditions
    and actual_query_runs == expected_query_runs
    and actual_records == expected_records
    and duplicate_rank_keys == 0
)

print("\nFINAL EXECUTION STATUS")
print("=" * 80)

if valid_execution:
    print(
        "PASS — All frozen retrieval conditions were "
        "executed and the raw ranked results were preserved."
    )
else:
    print(
        "FAIL — Review execution completeness before "
        "evaluating retrieval quality."
    )

FULL RETRIEVAL EXPERIMENT
Models: ['E1', 'E2', 'E3']
Chunk × overlap conditions: 6
Queries: 12
Top-k: 3
Expected ranked records: 648

LOADING E1: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Embedding dimension: 384

Running A_Fine × O0 × E1
Chunks: 17
Ranked records: 36
Runtime seconds: 3.92

Running A_Fine × O1 × E1
Chunks: 17
Ranked records: 36
Runtime seconds: 1.66

Running B_Baseline × O0 × E1
Chunks: 11
Ranked records: 36
Runtime seconds: 1.18

Running B_Baseline × O1 × E1
Chunks: 11
Ranked records: 36
Runtime seconds: 1.15

Running C_Broad × O0 × E1
Chunks: 9
Ranked records: 36
Runtime seconds: 1.02

Running C_Broad × O1 × E1
Chunks: 9
Ranked records: 36
Runtime seconds: 1.0

E1 completed in 9.93 seconds.
Progress saved after E1

LOADING E2: intfloat/multilingual-e5-base
Embedding dimension: 768

Running A_Fine × O0 × E2
Chunks: 17
Ranked records: 36
Runtime seconds: 12.63

Running A_Fine × O1 × E2
Chunks: 17
Ranked records: 36
Runtime seconds: 15.05

Running B_Ba

In [3]:
# Cell 10 — Build the retrieval annotation dataset

import json
import pandas as pd
from pathlib import Path

RAW_FILE = Path(
    "/content/raw_retrieval_results.csv"
)

QUERY_FILE = Path(
    "/content/retrieval_queries.json"
)

ANNOTATION_FILE = Path(
    "/content/retrieval_annotation_dataset.csv"
)

print("BUILD RETRIEVAL ANNOTATION DATASET")
print("=" * 75)

# ---------------------------------------------------------
# 1. Load frozen raw results
# ---------------------------------------------------------

raw_df = pd.read_csv(RAW_FILE)

with open(
    QUERY_FILE,
    "r",
    encoding="utf-8"
) as f:
    query_data = json.load(f)

queries = query_data["queries"]

query_lookup = {
    q["query_id"]: q
    for q in queries
}

print("Raw retrieval records:", len(raw_df))
print("Queries:", len(queries))


# ---------------------------------------------------------
# 2. Add frozen expected curriculum content
# ---------------------------------------------------------

annotation_df = raw_df.copy()

annotation_df[
    "expected_curriculum_content"
] = annotation_df["query_id"].map(
    lambda qid: " || ".join(
        query_lookup[qid][
            "expected_curriculum_content"
        ]
    )
)

annotation_df[
    "expected_element_count"
] = annotation_df["query_id"].map(
    lambda qid: len(
        query_lookup[qid][
            "expected_curriculum_content"
        ]
    )
)


# ---------------------------------------------------------
# 3. Add empty annotation fields
# ---------------------------------------------------------

annotation_df[
    "curriculum_relevance"
] = ""

annotation_df[
    "recovered_element_count"
] = ""

annotation_df[
    "annotation_notes"
] = ""


# ---------------------------------------------------------
# 4. Validate annotation dataset
# ---------------------------------------------------------

expected_rows = 648

missing_expected_content = (
    annotation_df[
        "expected_curriculum_content"
    ]
    .isna()
    .sum()
)

duplicate_rank_keys = (
    annotation_df
    .duplicated(
        subset=[
            "model_id",
            "configuration",
            "overlap_setting",
            "query_id",
            "rank"
        ]
    )
    .sum()
)

print("\nANNOTATION DATASET CHECK")
print("-" * 75)

print("Rows:", len(annotation_df))

print(
    "Missing expected content:",
    missing_expected_content
)

print(
    "Duplicate rank keys:",
    duplicate_rank_keys
)

print(
    "Annotation fields added:",
    [
        "curriculum_relevance",
        "recovered_element_count",
        "annotation_notes"
    ]
)


# ---------------------------------------------------------
# 5. Save separate annotation dataset
# ---------------------------------------------------------

annotation_df.to_csv(
    ANNOTATION_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved:")
print(ANNOTATION_FILE)


# ---------------------------------------------------------
# 6. Final validation
# ---------------------------------------------------------

valid = (
    len(annotation_df) == expected_rows
    and missing_expected_content == 0
    and duplicate_rank_keys == 0
)

print("\nFINAL ANNOTATION DATASET STATUS")
print("=" * 75)

if valid:
    print(
        "PASS — A separate annotation dataset was created "
        "without modifying the preserved raw retrieval results."
    )
else:
    print(
        "FAIL — Review the annotation dataset before "
        "starting retrieval-quality annotation."
    )

BUILD RETRIEVAL ANNOTATION DATASET
Raw retrieval records: 648
Queries: 12

ANNOTATION DATASET CHECK
---------------------------------------------------------------------------
Rows: 648
Missing expected content: 0
Duplicate rank keys: 0
Annotation fields added: ['curriculum_relevance', 'recovered_element_count', 'annotation_notes']

Saved:
/content/retrieval_annotation_dataset.csv

FINAL ANNOTATION DATASET STATUS
PASS — A separate annotation dataset was created without modifying the preserved raw retrieval results.


In [4]:
# Cell 11 — Add element-level recovery annotation field

import pandas as pd
from pathlib import Path

ANNOTATION_FILE = Path(
    "/content/retrieval_annotation_dataset.csv"
)

UPDATED_ANNOTATION_FILE = Path(
    "/content/retrieval_annotation_dataset.csv"
)

print("ANNOTATION STRUCTURE REFINEMENT")
print("=" * 75)

# ---------------------------------------------------------
# 1. Load annotation dataset
# ---------------------------------------------------------

annotation_df = pd.read_csv(
    ANNOTATION_FILE,
    keep_default_na=False
)

print("Rows loaded:", len(annotation_df))


# ---------------------------------------------------------
# 2. Add recovered_element_ids
# ---------------------------------------------------------

if "recovered_element_ids" not in annotation_df.columns:

    insert_position = (
        annotation_df.columns.get_loc(
            "recovered_element_count"
        )
    )

    annotation_df.insert(
        insert_position,
        "recovered_element_ids",
        ""
    )

    print(
        "Added field: recovered_element_ids"
    )

else:
    print(
        "Field already exists: recovered_element_ids"
    )


# ---------------------------------------------------------
# 3. Validate annotation fields
# ---------------------------------------------------------

required_annotation_fields = [
    "expected_curriculum_content",
    "expected_element_count",
    "curriculum_relevance",
    "recovered_element_ids",
    "recovered_element_count",
    "annotation_notes"
]

missing_fields = [
    field
    for field in required_annotation_fields
    if field not in annotation_df.columns
]

print("\nANNOTATION FIELDS")
print("-" * 75)

for field in required_annotation_fields:
    print(field)

print(
    "\nMissing fields:",
    missing_fields if missing_fields else "None"
)


# ---------------------------------------------------------
# 4. Confirm annotation fields are still empty
# ---------------------------------------------------------

fields_to_be_empty = [
    "curriculum_relevance",
    "recovered_element_ids",
    "recovered_element_count",
    "annotation_notes"
]

print("\nPRE-ANNOTATION EMPTY-FIELD CHECK")
print("-" * 75)

non_empty_counts = {}

for field in fields_to_be_empty:

    count = (
        annotation_df[field]
        .astype(str)
        .str.strip()
        .ne("")
        .sum()
    )

    non_empty_counts[field] = int(count)

    print(
        f"{field}: {count} non-empty values"
    )


# ---------------------------------------------------------
# 5. Validate frozen record structure
# ---------------------------------------------------------

duplicate_rank_keys = (
    annotation_df
    .duplicated(
        subset=[
            "model_id",
            "configuration",
            "overlap_setting",
            "query_id",
            "rank"
        ]
    )
    .sum()
)

print("\nRECORD CHECK")
print("-" * 75)

print("Rows:", len(annotation_df))

print(
    "Duplicate rank keys:",
    duplicate_rank_keys
)


# ---------------------------------------------------------
# 6. Save updated dataset
# ---------------------------------------------------------

annotation_df.to_csv(
    UPDATED_ANNOTATION_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved:")
print(UPDATED_ANNOTATION_FILE)


# ---------------------------------------------------------
# 7. Final validation
# ---------------------------------------------------------

valid = (
    len(annotation_df) == 648
    and not missing_fields
    and all(
        count == 0
        for count in non_empty_counts.values()
    )
    and duplicate_rank_keys == 0
)

print("\nFINAL ANNOTATION STRUCTURE STATUS")
print("=" * 75)

if valid:
    print(
        "PASS — The annotation dataset now supports "
        "element-level recovery tracking and remains "
        "unannotated."
    )
else:
    print(
        "FAIL — Review the annotation structure before "
        "starting annotation."
    )

ANNOTATION STRUCTURE REFINEMENT
Rows loaded: 648
Added field: recovered_element_ids

ANNOTATION FIELDS
---------------------------------------------------------------------------
expected_curriculum_content
expected_element_count
curriculum_relevance
recovered_element_ids
recovered_element_count
annotation_notes

Missing fields: None

PRE-ANNOTATION EMPTY-FIELD CHECK
---------------------------------------------------------------------------
curriculum_relevance: 0 non-empty values
recovered_element_ids: 0 non-empty values
recovered_element_count: 0 non-empty values
annotation_notes: 0 non-empty values

RECORD CHECK
---------------------------------------------------------------------------
Rows: 648
Duplicate rank keys: 0

Saved:
/content/retrieval_annotation_dataset.csv

FINAL ANNOTATION STRUCTURE STATUS
PASS — The annotation dataset now supports element-level recovery tracking and remains unannotated.


In [5]:
# Cell 12 — Create and freeze expected-element IDs for Q01–Q12

import json
from pathlib import Path

QUERY_FILE = Path(
    "/content/retrieval_queries.json"
)

ELEMENT_FILE = Path(
    "/content/retrieval_expected_elements.json"
)

print("EXPECTED ELEMENT ID DEFINITION")
print("=" * 75)

# ---------------------------------------------------------
# 1. Load frozen query set
# ---------------------------------------------------------

with open(
    QUERY_FILE,
    "r",
    encoding="utf-8"
) as f:
    query_data = json.load(f)

queries = query_data["queries"]

print("Queries loaded:", len(queries))


# ---------------------------------------------------------
# 2. Assign stable element IDs within each query
# ---------------------------------------------------------

expected_elements = []

for query in queries:

    query_elements = []

    for index, element_text in enumerate(
        query["expected_curriculum_content"],
        start=1
    ):

        element_id = f"E{index:02d}"

        query_elements.append({
            "element_id": element_id,
            "element_text": element_text
        })

    expected_elements.append({
        "query_id": query["query_id"],
        "query_ar": query["query_ar"],
        "query_type": query["query_type"],
        "expected_element_count": len(
            query_elements
        ),
        "expected_elements": query_elements
    })


# ---------------------------------------------------------
# 3. Save frozen element definition
# ---------------------------------------------------------

element_data = {
    "definition": (
        "Expected-element IDs provide stable identifiers "
        "for the frozen Expected Curriculum Content used "
        "during retrieval annotation. IDs are local to "
        "each query."
    ),
    "id_scope": (
        "Element IDs restart at E01 for each query and "
        "must always be interpreted together with query_id."
    ),
    "queries": expected_elements
}

with open(
    ELEMENT_FILE,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        element_data,
        f,
        ensure_ascii=False,
        indent=2
    )


# ---------------------------------------------------------
# 4. Display definitions
# ---------------------------------------------------------

print("\nFROZEN EXPECTED ELEMENTS")
print("-" * 75)

total_elements = 0

for query in expected_elements:

    print(
        f'\n{query["query_id"]} | '
        f'{query["query_type"]}'
    )

    for element in query["expected_elements"]:

        print(
            f'  {element["element_id"]}: '
            f'{element["element_text"]}'
        )

        total_elements += 1


# ---------------------------------------------------------
# 5. Validate IDs
# ---------------------------------------------------------

validation_errors = []

for query in expected_elements:

    actual_ids = [
        element["element_id"]
        for element in query["expected_elements"]
    ]

    expected_ids = [
        f"E{i:02d}"
        for i in range(
            1,
            query["expected_element_count"] + 1
        )
    ]

    if actual_ids != expected_ids:
        validation_errors.append(
            query["query_id"]
        )


print("\nVALIDATION")
print("-" * 75)

print(
    "Total expected elements:",
    total_elements
)

print(
    "Queries with invalid element IDs:",
    validation_errors
    if validation_errors
    else "None"
)

print("Saved:")
print(ELEMENT_FILE)


# ---------------------------------------------------------
# 6. Final status
# ---------------------------------------------------------

valid = (
    len(expected_elements) == 12
    and not validation_errors
    and total_elements > 0
)

print("\nFINAL EXPECTED-ELEMENT STATUS")
print("=" * 75)

if valid:
    print(
        "PASS — Expected Curriculum Content now has "
        "stable query-specific element IDs for annotation."
    )
else:
    print(
        "FAIL — Review expected-element definitions "
        "before annotation."
    )

EXPECTED ELEMENT ID DEFINITION
Queries loaded: 12

FROZEN EXPECTED ELEMENTS
---------------------------------------------------------------------------

Q01 | Direct/Narrow
  E01: تتضاعف مادة الوراثة DNA قبل البدء بعملية الانقسام.

Q02 | Direct/Narrow
  E01: يتكون الكروموسوم المتضاعف من كروماتيدين.
  E02: يرتبط الكروماتيدان بنقطة تسمى السنترومير.

Q03 | Direct/Relational
  E01: تنقسم الخلايا الجسمية بطريقة الانقسام المتساوي.
  E02: ينتج عن الانقسام المتساوي خليتان ابنتان.
  E03: تحتوي الخليتان الناتجتان على نفس عدد كروموسومات الخلية الأم.

Q04 | Direct/Narrow
  E01: تنمو الخلية.
  E02: يزداد حجم الخلية.
  E03: تتضاعف كمية المادة الوراثية DNA.

Q05 | Direct/Narrow
  E01: تتباعد الكروماتيدات الشقيقة.
  E02: يحدث التباعد بفعل انكماش خيوط المغزل.
  E03: يتجه كل منها إلى قطب.

Q06 | Sequence/Multi-part
  E01: الدور التمهيدي.
  E02: الدور الاستوائي.
  E03: الدور الانفصالي.
  E04: الدور النهائي.
  E05: انقسام السيتوبلازم.
  E06: تنتج خليتان متماثلتان.
  E07: تحتوي كل خلية ناتجة على نفس عدد كر

In [6]:
# Cell 13 — Build unique query–retrieved-text annotation units

import json
import pandas as pd
from pathlib import Path

ANNOTATION_FILE = Path(
    "/content/retrieval_annotation_dataset.csv"
)

ELEMENT_FILE = Path(
    "/content/retrieval_expected_elements.json"
)

UNIQUE_FILE = Path(
    "/content/unique_annotation_units.csv"
)

print("BUILD UNIQUE ANNOTATION UNITS")
print("=" * 75)

# ---------------------------------------------------------
# 1. Load annotation dataset
# ---------------------------------------------------------

annotation_df = pd.read_csv(
    ANNOTATION_FILE,
    keep_default_na=False
)

with open(
    ELEMENT_FILE,
    "r",
    encoding="utf-8"
) as f:
    element_data = json.load(f)

print(
    "Original retrieval records:",
    len(annotation_df)
)


# ---------------------------------------------------------
# 2. Build readable expected-element definitions
# ---------------------------------------------------------

element_lookup = {}

for query in element_data["queries"]:

    element_lookup[
        query["query_id"]
    ] = " || ".join(
        [
            f'{element["element_id"]}: '
            f'{element["element_text"]}'
            for element in query[
                "expected_elements"
            ]
        ]
    )


# ---------------------------------------------------------
# 3. Define unique annotation unit
# ---------------------------------------------------------

unique_df = (
    annotation_df[
        [
            "query_id",
            "query_type",
            "query_ar",
            "retrieval_text"
        ]
    ]
    .drop_duplicates(
        subset=[
            "query_id",
            "retrieval_text"
        ]
    )
    .reset_index(drop=True)
)

unique_df.insert(
    0,
    "annotation_unit_id",
    [
        f"AU{i:03d}"
        for i in range(
            1,
            len(unique_df) + 1
        )
    ]
)


# ---------------------------------------------------------
# 4. Add expected elements
# ---------------------------------------------------------

unique_df[
    "expected_elements"
] = unique_df["query_id"].map(
    element_lookup
)


# ---------------------------------------------------------
# 5. Add blank annotation fields
# ---------------------------------------------------------

unique_df[
    "curriculum_relevance"
] = ""

unique_df[
    "recovered_element_ids"
] = ""

unique_df[
    "recovered_element_count"
] = ""

unique_df[
    "annotation_notes"
] = ""


# ---------------------------------------------------------
# 6. Count how often each unit occurs in raw results
# ---------------------------------------------------------

occurrence_counts = (
    annotation_df
    .groupby(
        [
            "query_id",
            "retrieval_text"
        ]
    )
    .size()
    .reset_index(
        name="occurrence_count"
    )
)

unique_df = unique_df.merge(
    occurrence_counts,
    on=[
        "query_id",
        "retrieval_text"
    ],
    how="left"
)


# ---------------------------------------------------------
# 7. Validation
# ---------------------------------------------------------

duplicate_units = (
    unique_df
    .duplicated(
        subset=[
            "query_id",
            "retrieval_text"
        ]
    )
    .sum()
)

missing_elements = (
    unique_df[
        "expected_elements"
    ]
    .isna()
    .sum()
)

reconstructed_records = (
    unique_df[
        "occurrence_count"
    ]
    .sum()
)

print("\nUNIQUE ANNOTATION SET")
print("-" * 75)

print(
    "Unique query–text pairs:",
    len(unique_df)
)

print(
    "Duplicate annotation units:",
    duplicate_units
)

print(
    "Missing expected-element definitions:",
    missing_elements
)

print(
    "Occurrences represented:",
    reconstructed_records
)

print(
    "Original records represented:",
    len(annotation_df)
)


# ---------------------------------------------------------
# 8. Summary by query
# ---------------------------------------------------------

print("\nUNIQUE UNITS BY QUERY")
print("-" * 75)

query_counts = (
    unique_df
    .groupby("query_id")
    .size()
)

print(query_counts)


# ---------------------------------------------------------
# 9. Save
# ---------------------------------------------------------

unique_df.to_csv(
    UNIQUE_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved:")
print(UNIQUE_FILE)


# ---------------------------------------------------------
# 10. Final status
# ---------------------------------------------------------

valid = (
    duplicate_units == 0
    and missing_elements == 0
    and reconstructed_records
        == len(annotation_df)
)

print("\nFINAL UNIQUE-UNIT STATUS")
print("=" * 75)

if valid:
    print(
        "PASS — Unique annotation units were created "
        "without losing any retrieval-record occurrences."
    )
else:
    print(
        "FAIL — Review unique-unit construction before "
        "annotation."
    )

BUILD UNIQUE ANNOTATION UNITS
Original retrieval records: 648

UNIQUE ANNOTATION SET
---------------------------------------------------------------------------
Unique query–text pairs: 220
Duplicate annotation units: 0
Missing expected-element definitions: 0
Occurrences represented: 648
Original records represented: 648

UNIQUE UNITS BY QUERY
---------------------------------------------------------------------------
query_id
Q01    20
Q02    20
Q03    21
Q04    19
Q05    17
Q06    20
Q07    14
Q08    19
Q09    15
Q10    22
Q11    19
Q12    14
dtype: int64

Saved:
/content/unique_annotation_units.csv

FINAL UNIQUE-UNIT STATUS
PASS — Unique annotation units were created without losing any retrieval-record occurrences.


In [7]:
# Cell 14 — Build element-level annotation sheet

import json
import pandas as pd
from pathlib import Path

UNIQUE_FILE = Path(
    "/content/unique_annotation_units.csv"
)

ELEMENT_FILE = Path(
    "/content/retrieval_expected_elements.json"
)

ELEMENT_ANNOTATION_FILE = Path(
    "/content/element_level_annotation.csv"
)

print("BUILD ELEMENT-LEVEL ANNOTATION SHEET")
print("=" * 75)

# ---------------------------------------------------------
# 1. Load data
# ---------------------------------------------------------

unique_df = pd.read_csv(
    UNIQUE_FILE,
    keep_default_na=False
)

with open(
    ELEMENT_FILE,
    "r",
    encoding="utf-8"
) as f:
    element_data = json.load(f)

print(
    "Unique annotation units:",
    len(unique_df)
)


# ---------------------------------------------------------
# 2. Build expected-element lookup
# ---------------------------------------------------------

element_lookup = {}

for query in element_data["queries"]:

    element_lookup[
        query["query_id"]
    ] = query["expected_elements"]


# ---------------------------------------------------------
# 3. Expand each annotation unit by its expected elements
# ---------------------------------------------------------

rows = []

for _, unit in unique_df.iterrows():

    query_id = unit["query_id"]

    for element in element_lookup[query_id]:

        rows.append({
            "annotation_unit_id":
                unit["annotation_unit_id"],

            "query_id":
                query_id,

            "query_type":
                unit["query_type"],

            "query_ar":
                unit["query_ar"],

            "element_id":
                element["element_id"],

            "expected_element":
                element["element_text"],

            "retrieval_text":
                unit["retrieval_text"],

            # Human annotation field:
            # Yes = element directly supported
            # No  = element not directly supported
            "element_recovered":
                "",

            "element_annotation_notes":
                ""
        })


element_annotation_df = pd.DataFrame(rows)


# ---------------------------------------------------------
# 4. Validate
# ---------------------------------------------------------

duplicate_keys = (
    element_annotation_df
    .duplicated(
        subset=[
            "annotation_unit_id",
            "element_id"
        ]
    )
    .sum()
)

missing_expected_elements = (
    element_annotation_df[
        "expected_element"
    ]
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

non_empty_annotations = (
    element_annotation_df[
        "element_recovered"
    ]
    .astype(str)
    .str.strip()
    .ne("")
    .sum()
)


print("\nELEMENT-LEVEL DATASET")
print("-" * 75)

print(
    "Rows:",
    len(element_annotation_df)
)

print(
    "Duplicate unit-element keys:",
    duplicate_keys
)

print(
    "Missing expected elements:",
    missing_expected_elements
)

print(
    "Pre-filled element judgments:",
    non_empty_annotations
)


# ---------------------------------------------------------
# 5. Summary by query
# ---------------------------------------------------------

print("\nELEMENT-LEVEL ROWS BY QUERY")
print("-" * 75)

print(
    element_annotation_df
    .groupby("query_id")
    .size()
)


# ---------------------------------------------------------
# 6. Save
# ---------------------------------------------------------

element_annotation_df.to_csv(
    ELEMENT_ANNOTATION_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved:")
print(ELEMENT_ANNOTATION_FILE)


# ---------------------------------------------------------
# 7. Final validation
# ---------------------------------------------------------

valid = (
    duplicate_keys == 0
    and missing_expected_elements == 0
    and non_empty_annotations == 0
)

print("\nFINAL ELEMENT-LEVEL STATUS")
print("=" * 75)

if valid:
    print(
        "PASS — Element-level annotation sheet was "
        "created and remains fully unannotated."
    )
else:
    print(
        "FAIL — Review the element-level sheet before "
        "starting annotation."
    )

BUILD ELEMENT-LEVEL ANNOTATION SHEET
Unique annotation units: 220

ELEMENT-LEVEL DATASET
---------------------------------------------------------------------------
Rows: 674
Duplicate unit-element keys: 0
Missing expected elements: 0
Pre-filled element judgments: 0

ELEMENT-LEVEL ROWS BY QUERY
---------------------------------------------------------------------------
query_id
Q01     20
Q02     40
Q03     63
Q04     57
Q05     51
Q06    140
Q07     28
Q08     38
Q09     45
Q10     88
Q11     76
Q12     28
dtype: int64

Saved:
/content/element_level_annotation.csv

FINAL ELEMENT-LEVEL STATUS
PASS — Element-level annotation sheet was created and remains fully unannotated.


In [8]:
# Cell 15 — Create the human retrieval annotation workbook

import pandas as pd
from pathlib import Path
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, PatternFill
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.utils import get_column_letter


UNIQUE_FILE = Path(
    "/content/unique_annotation_units.csv"
)

ELEMENT_FILE = Path(
    "/content/element_level_annotation.csv"
)

WORKBOOK_FILE = Path(
    "/content/retrieval_annotation_workbook.xlsx"
)


print("CREATE HUMAN ANNOTATION WORKBOOK")
print("=" * 75)


# =========================================================
# 1. Load working datasets
# =========================================================

unique_df = pd.read_csv(
    UNIQUE_FILE,
    keep_default_na=False
)

element_df = pd.read_csv(
    ELEMENT_FILE,
    keep_default_na=False
)

print("Unique annotation units:", len(unique_df))
print("Element-level rows:", len(element_df))


# =========================================================
# 2. Prepare Unit-Level Annotation sheet
# =========================================================

unit_columns = [
    "annotation_unit_id",
    "query_id",
    "query_type",
    "query_ar",
    "expected_elements",
    "retrieval_text",
    "occurrence_count",
    "curriculum_relevance",
    "annotation_notes"
]

unit_sheet = unique_df[
    unit_columns
].copy()


# =========================================================
# 3. Prepare Element-Level Annotation sheet
# =========================================================

element_columns = [
    "annotation_unit_id",
    "query_id",
    "query_type",
    "query_ar",
    "element_id",
    "expected_element",
    "retrieval_text",
    "element_recovered",
    "element_annotation_notes"
]

element_sheet = element_df[
    element_columns
].copy()


# =========================================================
# 4. Create Instructions sheet
# =========================================================

instructions = pd.DataFrame({
    "Section": [
        "Purpose",
        "Unit-Level Relevance",
        "Relevant",
        "Partially Relevant",
        "Not Relevant",
        "Element-Level Recovery",
        "Yes",
        "No",
        "Important Rule",
        "Source Boundary"
    ],

    "Instruction": [
        (
            "Use this workbook for human annotation of "
            "the preliminary retrieval experiment."
        ),
        (
            "Assign one curriculum relevance judgment "
            "to each unique query–retrieved-text pair."
        ),
        (
            "The retrieved text directly contains curriculum "
            "information needed to answer the query or directly "
            "contributes one or more required expected-content elements."
        ),
        (
            "The retrieved text is related to the queried curriculum "
            "concept and provides useful context, but does not directly "
            "provide the required answer or expected-content element(s)."
        ),
        (
            "The retrieved text does not directly answer or meaningfully "
            "support the query."
        ),
        (
            "For every expected element, decide whether the retrieved "
            "text directly supports that curriculum meaning."
        ),
        (
            "Use Yes when the expected element is directly supported "
            "by the retrieved curriculum text."
        ),
        (
            "Use No when the expected element is not directly supported "
            "by the retrieved curriculum text."
        ),
        (
            "Exact wording is not required, but do not infer missing "
            "information from external scientific knowledge."
        ),
        (
            "Judge only against the selected official Palestinian "
            "Grade 8 Science curriculum used in the experiment."
        )
    ]
})


# =========================================================
# 5. Write workbook
# =========================================================

with pd.ExcelWriter(
    WORKBOOK_FILE,
    engine="openpyxl"
) as writer:

    instructions.to_excel(
        writer,
        sheet_name="Instructions",
        index=False
    )

    unit_sheet.to_excel(
        writer,
        sheet_name="Unit Annotation",
        index=False
    )

    element_sheet.to_excel(
        writer,
        sheet_name="Element Annotation",
        index=False
    )


# =========================================================
# 6. Open workbook for formatting
# =========================================================

wb = load_workbook(WORKBOOK_FILE)

instruction_ws = wb["Instructions"]
unit_ws = wb["Unit Annotation"]
element_ws = wb["Element Annotation"]


# =========================================================
# 7. General formatting
# =========================================================

for ws in [
    instruction_ws,
    unit_ws,
    element_ws
]:

    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

    for cell in ws[1]:

        cell.font = Font(
            bold=True
        )

        cell.fill = PatternFill(
            fill_type="solid",
            fgColor="D9EAF7"
        )

        cell.alignment = Alignment(
            horizontal="center",
            vertical="center",
            wrap_text=True
        )

    for row in ws.iter_rows(
        min_row=2
    ):

        for cell in row:

            cell.alignment = Alignment(
                vertical="top",
                wrap_text=True
            )


# =========================================================
# 8. Column widths
# =========================================================

instruction_widths = {
    "A": 24,
    "B": 100
}

unit_widths = {
    "A": 18,
    "B": 10,
    "C": 24,
    "D": 55,
    "E": 85,
    "F": 100,
    "G": 18,
    "H": 24,
    "I": 55
}

element_widths = {
    "A": 18,
    "B": 10,
    "C": 24,
    "D": 55,
    "E": 12,
    "F": 75,
    "G": 100,
    "H": 20,
    "I": 55
}

for column, width in instruction_widths.items():
    instruction_ws.column_dimensions[column].width = width

for column, width in unit_widths.items():
    unit_ws.column_dimensions[column].width = width

for column, width in element_widths.items():
    element_ws.column_dimensions[column].width = width


# =========================================================
# 9. Add relevance dropdown
# =========================================================

relevance_validation = DataValidation(
    type="list",
    formula1=(
        '"Relevant,Partially Relevant,Not Relevant"'
    ),
    allow_blank=True
)

unit_ws.add_data_validation(
    relevance_validation
)

relevance_validation.add(
    f"H2:H{unit_ws.max_row}"
)


# =========================================================
# 10. Add Yes/No dropdown
# =========================================================

element_validation = DataValidation(
    type="list",
    formula1='"Yes,No"',
    allow_blank=True
)

element_ws.add_data_validation(
    element_validation
)

element_validation.add(
    f"H2:H{element_ws.max_row}"
)


# =========================================================
# 11. Highlight human-input columns
# =========================================================

input_fill = PatternFill(
    fill_type="solid",
    fgColor="FFF2CC"
)

for row in range(
    2,
    unit_ws.max_row + 1
):
    unit_ws[f"H{row}"].fill = input_fill
    unit_ws[f"I{row}"].fill = input_fill

for row in range(
    2,
    element_ws.max_row + 1
):
    element_ws[f"H{row}"].fill = input_fill
    element_ws[f"I{row}"].fill = input_fill


# =========================================================
# 12. Save workbook
# =========================================================

wb.save(WORKBOOK_FILE)


# =========================================================
# 13. Validate workbook
# =========================================================

validation_wb = load_workbook(
    WORKBOOK_FILE,
    read_only=False
)

sheet_names = validation_wb.sheetnames

unit_rows = (
    validation_wb["Unit Annotation"].max_row - 1
)

element_rows = (
    validation_wb["Element Annotation"].max_row - 1
)


print("\nWORKBOOK VALIDATION")
print("-" * 75)

print("Sheets:", sheet_names)
print("Unit annotation rows:", unit_rows)
print("Element annotation rows:", element_rows)

print("\nHuman-input fields:")
print(
    "Unit Annotation → "
    "curriculum_relevance, annotation_notes"
)
print(
    "Element Annotation → "
    "element_recovered, element_annotation_notes"
)


valid = (
    sheet_names == [
        "Instructions",
        "Unit Annotation",
        "Element Annotation"
    ]
    and unit_rows == 220
    and element_rows == 674
)


print("\nSaved:")
print(WORKBOOK_FILE)

print("\nFINAL WORKBOOK STATUS")
print("=" * 75)

if valid:
    print(
        "PASS — Human annotation workbook was created "
        "with 220 unit-level judgments and 674 "
        "element-level judgments."
    )
else:
    print(
        "FAIL — Review workbook construction before annotation."
    )

CREATE HUMAN ANNOTATION WORKBOOK
Unique annotation units: 220
Element-level rows: 674

WORKBOOK VALIDATION
---------------------------------------------------------------------------
Sheets: ['Instructions', 'Unit Annotation', 'Element Annotation']
Unit annotation rows: 220
Element annotation rows: 674

Human-input fields:
Unit Annotation → curriculum_relevance, annotation_notes
Element Annotation → element_recovered, element_annotation_notes

Saved:
/content/retrieval_annotation_workbook.xlsx

FINAL WORKBOOK STATUS
PASS — Human annotation workbook was created with 220 unit-level judgments and 674 element-level judgments.


In [9]:
# Cell 16 — Display first three annotation units for pilot review

import pandas as pd
from pathlib import Path

UNIQUE_FILE = Path(
    "/content/unique_annotation_units.csv"
)

ELEMENT_FILE = Path(
    "/content/element_level_annotation.csv"
)

unique_df = pd.read_csv(
    UNIQUE_FILE,
    keep_default_na=False
)

element_df = pd.read_csv(
    ELEMENT_FILE,
    keep_default_na=False
)

pilot_ids = (
    unique_df["annotation_unit_id"]
    .head(3)
    .tolist()
)

print("PILOT ANNOTATION REVIEW")
print("=" * 75)

for unit_id in pilot_ids:

    unit = unique_df[
        unique_df["annotation_unit_id"] == unit_id
    ].iloc[0]

    elements = element_df[
        element_df["annotation_unit_id"] == unit_id
    ]

    print("\n" + "=" * 75)
    print("ANNOTATION UNIT:", unit_id)
    print("QUERY ID:", unit["query_id"])
    print("QUERY TYPE:", unit["query_type"])

    print("\nQUERY:")
    print(unit["query_ar"])

    print("\nEXPECTED ELEMENTS:")
    for _, element in elements.iterrows():
        print(
            f'{element["element_id"]}: '
            f'{element["expected_element"]}'
        )

    print("\nRETRIEVED TEXT:")
    print(unit["retrieval_text"])

    print("\nOCCURRENCE COUNT:")
    print(unit["occurrence_count"])

print("\n" + "=" * 75)
print(
    "Pilot units displayed. "
    "No annotation values were written."
)

PILOT ANNOTATION REVIEW

ANNOTATION UNIT: AU001
QUERY ID: Q01
QUERY TYPE: Direct/Narrow

QUERY:
ماذا يحدث للمادة الوراثية DNA قبل انقسام الخلية؟

EXPECTED ELEMENTS:
E01: تتضاعف مادة الوراثة DNA قبل البدء بعملية الانقسام.

RETRIEVED TEXT:
كثير من الخلايا في أجسامنا تنقسم وتتكاثر، فتنتج خلايا جديدة، قد تكون مطابقة للخلايا الأصلية، أو غير مطابقة. وقبل انقسام الخلايا تكون الكروموسومات مفردة، وعلى شكل خيوط طويلة رفيعة ومتشابكة داخل النواة في الخلايا، تسمى شبكة كروماتينية.
ولا يمكن مشاهدتها على هيئة كروموسومات مستقلة بوضوح في حال عدم انقسام الخلية. يحدث تضاعف لمادة الوراثة (DNA) والعضيات، قبل البدء بعملية الانقسام.
يوضح الشكل مراحل تضاعف الكروموسومات في نواة الخلية: قبل التضاعف، تضاعف DNA، ثم ظهور نسخة أصلية ونسخة جديدة.

OCCURRENCE COUNT:
3

ANNOTATION UNIT: AU002
QUERY ID: Q01
QUERY TYPE: Direct/Narrow

QUERY:
ماذا يحدث للمادة الوراثية DNA قبل انقسام الخلية؟

EXPECTED ELEMENTS:
E01: تتضاعف مادة الوراثة DNA قبل البدء بعملية الانقسام.

RETRIEVED TEXT:
انقسام الخلايا
قبل دخول الخلية في عملية 

In [10]:
# Cell 17 — Display multi-element pilot annotation units

import pandas as pd
from pathlib import Path

UNIQUE_FILE = Path("/content/unique_annotation_units.csv")
ELEMENT_FILE = Path("/content/element_level_annotation.csv")

unique_df = pd.read_csv(
    UNIQUE_FILE,
    keep_default_na=False
)

element_df = pd.read_csv(
    ELEMENT_FILE,
    keep_default_na=False
)

# Predefined multi-element queries
pilot_queries = ["Q06", "Q10", "Q11"]

# Select first two annotation units for each query
pilot_units = []

for query_id in pilot_queries:
    selected = (
        unique_df[
            unique_df["query_id"] == query_id
        ]
        .head(2)
        ["annotation_unit_id"]
        .tolist()
    )

    pilot_units.extend(selected)


print("MULTI-ELEMENT PILOT REVIEW")
print("=" * 75)

print("Pilot queries:", pilot_queries)
print("Selected units:", pilot_units)
print("Number of units:", len(pilot_units))


for unit_id in pilot_units:

    unit = unique_df[
        unique_df["annotation_unit_id"] == unit_id
    ].iloc[0]

    elements = element_df[
        element_df["annotation_unit_id"] == unit_id
    ]

    print("\n" + "=" * 75)
    print("ANNOTATION UNIT:", unit_id)
    print("QUERY ID:", unit["query_id"])
    print("QUERY TYPE:", unit["query_type"])

    print("\nQUERY:")
    print(unit["query_ar"])

    print("\nEXPECTED ELEMENTS:")

    for _, element in elements.iterrows():
        print(
            f'{element["element_id"]}: '
            f'{element["expected_element"]}'
        )

    print("\nRETRIEVED TEXT:")
    print(unit["retrieval_text"])

    print("\nOCCURRENCE COUNT:")
    print(unit["occurrence_count"])


print("\n" + "=" * 75)
print(
    "Multi-element pilot units displayed. "
    "No annotation values were written."
)

MULTI-ELEMENT PILOT REVIEW
Pilot queries: ['Q06', 'Q10', 'Q11']
Selected units: ['AU016', 'AU017', 'AU028', 'AU029', 'AU031', 'AU032']
Number of units: 6

ANNOTATION UNIT: AU016
QUERY ID: Q06
QUERY TYPE: Sequence/Multi-part

QUERY:
ما مراحل الانقسام المتساوي وما الذي يحدث خلالها حتى تنتج خليتان؟

EXPECTED ELEMENTS:
E01: الدور التمهيدي.
E02: الدور الاستوائي.
E03: الدور الانفصالي.
E04: الدور النهائي.
E05: انقسام السيتوبلازم.
E06: تنتج خليتان متماثلتان.
E07: تحتوي كل خلية ناتجة على نفس عدد كروموسومات الخلية الأم.

RETRIEVED TEXT:
ادرس مراحل الانقسام المتساوي للخلية في الشكل أدناه، ثم أجب عن الأسئلة التي تليه.
الدور التمهيدي: تظهر كروموسومات قصيرة وغليظة متضاعفة، يختفي الغلاف النووي، وتختفي النوية.

OCCURRENCE COUNT:
3

ANNOTATION UNIT: AU017
QUERY ID: Q06
QUERY TYPE: Sequence/Multi-part

QUERY:
ما مراحل الانقسام المتساوي وما الذي يحدث خلالها حتى تنتج خليتان؟

EXPECTED ELEMENTS:
E01: الدور التمهيدي.
E02: الدور الاستوائي.
E03: الدور الانفصالي.
E04: الدور النهائي.
E05: انقسام السيتوبلازم.
E0

In [17]:
# Cell 18 — Validate saved pilot annotations

import pandas as pd
from pathlib import Path

WORKBOOK_FILE = Path(
    "/content/retrieval_annotation_workbook.xlsx"
)

print("VALIDATE PILOT ANNOTATIONS")
print("=" * 75)

# ---------------------------------------------------------
# 1. Load workbook sheets
# ---------------------------------------------------------

unit_df = pd.read_excel(
    WORKBOOK_FILE,
    sheet_name="Unit Annotation",
    keep_default_na=False
)

element_df = pd.read_excel(
    WORKBOOK_FILE,
    sheet_name="Element Annotation",
    keep_default_na=False
)

print("Unit rows loaded:", len(unit_df))
print("Element rows loaded:", len(element_df))


# ---------------------------------------------------------
# 2. Pilot units that should currently be annotated
# ---------------------------------------------------------

pilot_units = [
    "AU001",
    "AU002",
    "AU003",
    "AU016",
    "AU017",
    "AU028",
    "AU029",
    "AU031",
    "AU032"
]


# ---------------------------------------------------------
# 3. Expected unit-level pilot judgments
# ---------------------------------------------------------

expected_unit = {
    "AU001": "Relevant",
    "AU002": "Relevant",
    "AU003": "Not Relevant",
    "AU016": "Relevant",
    "AU017": "Partially Relevant",
    "AU028": "Relevant",
    "AU029": "Not Relevant",
    "AU031": "Relevant",
    "AU032": "Partially Relevant"
}


# ---------------------------------------------------------
# 4. Expected element-level pilot judgments
# ---------------------------------------------------------

expected_element = {
    ("AU001", "E01"): "Yes",

    ("AU002", "E01"): "Yes",

    ("AU003", "E01"): "No",

    ("AU016", "E01"): "Yes",
    ("AU016", "E02"): "No",
    ("AU016", "E03"): "No",
    ("AU016", "E04"): "No",
    ("AU016", "E05"): "No",
    ("AU016", "E06"): "No",
    ("AU016", "E07"): "No",

    ("AU017", "E01"): "No",
    ("AU017", "E02"): "No",
    ("AU017", "E03"): "No",
    ("AU017", "E04"): "No",
    ("AU017", "E05"): "No",
    ("AU017", "E06"): "No",
    ("AU017", "E07"): "No",

    ("AU028", "E01"): "Yes",
    ("AU028", "E02"): "Yes",
    ("AU028", "E03"): "Yes",
    ("AU028", "E04"): "Yes",

    ("AU029", "E01"): "No",
    ("AU029", "E02"): "No",
    ("AU029", "E03"): "No",
    ("AU029", "E04"): "No",

    ("AU031", "E01"): "Yes",
    ("AU031", "E02"): "Yes",
    ("AU031", "E03"): "No",
    ("AU031", "E04"): "No",

    ("AU032", "E01"): "No",
    ("AU032", "E02"): "No",
    ("AU032", "E03"): "No",
    ("AU032", "E04"): "No"
}


# ---------------------------------------------------------
# 5. Validate unit-level judgments
# ---------------------------------------------------------

unit_errors = []

print("\nUNIT-LEVEL PILOT CHECK")
print("-" * 75)

for unit_id, expected_value in expected_unit.items():

    rows = unit_df[
        unit_df["annotation_unit_id"] == unit_id
    ]

    if len(rows) != 1:
        unit_errors.append(
            f"{unit_id}: expected 1 row, found {len(rows)}"
        )
        continue

    actual_value = str(
        rows.iloc[0]["curriculum_relevance"]
    ).strip()

    status = (
        "PASS"
        if actual_value == expected_value
        else "FAIL"
    )

    print(
        f"{unit_id}: "
        f"{actual_value!r} | "
        f"Expected: {expected_value!r} "
        f"→ {status}"
    )

    if actual_value != expected_value:
        unit_errors.append(
            f"{unit_id}: {actual_value!r} "
            f"!= {expected_value!r}"
        )


# ---------------------------------------------------------
# 6. Validate element-level judgments
# ---------------------------------------------------------

element_errors = []

print("\nELEMENT-LEVEL PILOT CHECK")
print("-" * 75)

for key, expected_value in expected_element.items():

    unit_id, element_id = key

    rows = element_df[
        (element_df["annotation_unit_id"] == unit_id)
        &
        (element_df["element_id"] == element_id)
    ]

    if len(rows) != 1:
        element_errors.append(
            f"{unit_id}/{element_id}: "
            f"expected 1 row, found {len(rows)}"
        )
        continue

    actual_value = str(
        rows.iloc[0]["element_recovered"]
    ).strip()

    status = (
        "PASS"
        if actual_value == expected_value
        else "FAIL"
    )

    print(
        f"{unit_id}/{element_id}: "
        f"{actual_value!r} | "
        f"Expected: {expected_value!r} "
        f"→ {status}"
    )

    if actual_value != expected_value:
        element_errors.append(
            f"{unit_id}/{element_id}: "
            f"{actual_value!r} "
            f"!= {expected_value!r}"
        )


# ---------------------------------------------------------
# 7. Check accidental annotations outside pilot
# ---------------------------------------------------------

nonpilot_unit_annotations = unit_df[
    (~unit_df["annotation_unit_id"].isin(pilot_units))
    &
    (
        unit_df["curriculum_relevance"]
        .astype(str)
        .str.strip()
        .ne("")
    )
]

nonpilot_element_annotations = element_df[
    (~element_df["annotation_unit_id"].isin(pilot_units))
    &
    (
        element_df["element_recovered"]
        .astype(str)
        .str.strip()
        .ne("")
    )
]

print("\nACCIDENTAL-ENTRY CHECK")
print("-" * 75)

print(
    "Annotated unit rows outside pilot:",
    len(nonpilot_unit_annotations)
)

print(
    "Annotated element rows outside pilot:",
    len(nonpilot_element_annotations)
)


# ---------------------------------------------------------
# 8. Summary
# ---------------------------------------------------------

print("\nPILOT SUMMARY")
print("-" * 75)

print(
    "Expected annotated units:",
    len(expected_unit)
)

print(
    "Expected element judgments:",
    len(expected_element)
)

print(
    "Unit-level errors:",
    len(unit_errors)
)

print(
    "Element-level errors:",
    len(element_errors)
)


# ---------------------------------------------------------
# 9. Final status
# ---------------------------------------------------------

valid = (
    len(unit_errors) == 0
    and len(element_errors) == 0
    and len(nonpilot_unit_annotations) == 0
    and len(nonpilot_element_annotations) == 0
)

print("\nFINAL PILOT VALIDATION STATUS")
print("=" * 75)

if valid:
    print(
        "PASS — All pilot annotations were saved correctly "
        "and no annotations were entered outside the pilot set."
    )
else:
    print(
        "FAIL — Review the reported annotation entries "
        "before proceeding to full annotation."
    )

VALIDATE PILOT ANNOTATIONS
Unit rows loaded: 220
Element rows loaded: 674

UNIT-LEVEL PILOT CHECK
---------------------------------------------------------------------------
AU001: 'Relevant' | Expected: 'Relevant' → PASS
AU002: 'Relevant' | Expected: 'Relevant' → PASS
AU003: 'Not Relevant' | Expected: 'Not Relevant' → PASS
AU016: 'Relevant' | Expected: 'Relevant' → PASS
AU017: 'Partially Relevant' | Expected: 'Partially Relevant' → PASS
AU028: 'Relevant' | Expected: 'Relevant' → PASS
AU029: 'Not Relevant' | Expected: 'Not Relevant' → PASS
AU031: 'Relevant' | Expected: 'Relevant' → PASS
AU032: 'Partially Relevant' | Expected: 'Partially Relevant' → PASS

ELEMENT-LEVEL PILOT CHECK
---------------------------------------------------------------------------
AU001/E01: 'Yes' | Expected: 'Yes' → PASS
AU002/E01: 'Yes' | Expected: 'Yes' → PASS
AU003/E01: 'No' | Expected: 'No' → PASS
AU016/E01: 'Yes' | Expected: 'Yes' → PASS
AU016/E02: 'No' | Expected: 'No' → PASS
AU016/E03: 'No' | Expected: '

In [18]:
# ============================================================
# CELL 19 — DERIVE RELEVANCE FROM ELEMENT-LEVEL ANNOTATIONS
# ============================================================

import pandas as pd
import numpy as np

WORKBOOK_PATH = "/content/retrieval_annotation_workbook.xlsx"

# Load current workbook sheets
unit_df = pd.read_excel(WORKBOOK_PATH, sheet_name="Unit Annotation")
element_df = pd.read_excel(WORKBOOK_PATH, sheet_name="Element Annotation")

print("DERIVE RELEVANCE FROM EXPECTED-ELEMENT RECOVERY")
print("=" * 75)

# ------------------------------------------------------------
# 1. Normalize element judgments
# ------------------------------------------------------------

element_df["element_recovered"] = (
    element_df["element_recovered"]
    .astype("string")
    .str.strip()
)

valid_element_values = {"Yes", "No"}

invalid_element_values = sorted(
    set(
        element_df.loc[
            element_df["element_recovered"].notna(),
            "element_recovered"
        ].tolist()
    ) - valid_element_values
)

print("Invalid element judgment values:", invalid_element_values)

# ------------------------------------------------------------
# 2. Calculate annotation progress per unique unit
# ------------------------------------------------------------

def summarize_element_annotation(group):
    total = len(group)

    answered = group["element_recovered"].isin(["Yes", "No"]).sum()
    recovered = (group["element_recovered"] == "Yes").sum()

    return pd.Series({
        "expected_element_count": total,
        "annotated_element_count": answered,
        "recovered_element_count": recovered
    })

element_summary = (
    element_df
    .groupby("annotation_unit_id", as_index=False)
    .apply(
        summarize_element_annotation,
        include_groups=False
    )
    .reset_index()
)

# Handle pandas versions where groupby/apply adds an extra index column
if "level_0" in element_summary.columns:
    element_summary = element_summary.drop(columns=["level_0"])

if "index" in element_summary.columns:
    element_summary = element_summary.drop(columns=["index"])

# ------------------------------------------------------------
# 3. Derive completion status
# ------------------------------------------------------------

element_summary["annotation_complete"] = (
    element_summary["annotated_element_count"]
    == element_summary["expected_element_count"]
)

# ------------------------------------------------------------
# 4. Derive curriculum relevance ONLY for completed units
# ------------------------------------------------------------

def derive_relevance(row):
    if not row["annotation_complete"]:
        return pd.NA

    recovered = int(row["recovered_element_count"])
    expected = int(row["expected_element_count"])

    if recovered == 0:
        return "Not Relevant"

    if recovered == expected:
        return "Relevant"

    return "Partially Relevant"


element_summary["derived_curriculum_relevance"] = (
    element_summary.apply(derive_relevance, axis=1)
)

# ------------------------------------------------------------
# 5. Merge with unit-level information
# ------------------------------------------------------------

columns_to_keep = [
    col for col in [
        "annotation_unit_id",
        "query_id",
        "query_text",
        "expected_curriculum_content",
        "retrieval_text"
    ]
    if col in unit_df.columns
]

derived_df = (
    unit_df[columns_to_keep]
    .merge(
        element_summary,
        on="annotation_unit_id",
        how="left"
    )
)

# ------------------------------------------------------------
# 6. Validation
# ------------------------------------------------------------

print("\nVALIDATION")
print("-" * 75)

print("Unique annotation units:", len(derived_df))

print(
    "Completed units:",
    int(derived_df["annotation_complete"].fillna(False).sum())
)

print(
    "Incomplete units:",
    int((~derived_df["annotation_complete"].fillna(False)).sum())
)

print(
    "Units with derived relevance:",
    int(derived_df["derived_curriculum_relevance"].notna().sum())
)

print(
    "Units without derived relevance:",
    int(derived_df["derived_curriculum_relevance"].isna().sum())
)

# Completed units must always have a relevance label
completed_without_label = derived_df[
    derived_df["annotation_complete"].fillna(False)
    & derived_df["derived_curriculum_relevance"].isna()
]

# Incomplete units must NOT receive a relevance label
incomplete_with_label = derived_df[
    (~derived_df["annotation_complete"].fillna(False))
    & derived_df["derived_curriculum_relevance"].notna()
]

print(
    "Completed units without relevance:",
    len(completed_without_label)
)

print(
    "Incomplete units with relevance:",
    len(incomplete_with_label)
)

# ------------------------------------------------------------
# 7. Show completed pilot units
# ------------------------------------------------------------

pilot_completed = derived_df[
    derived_df["annotation_complete"].fillna(False)
][
    [
        "annotation_unit_id",
        "query_id",
        "expected_element_count",
        "recovered_element_count",
        "derived_curriculum_relevance"
    ]
]

print("\nCURRENT COMPLETED ANNOTATION UNITS")
print("-" * 75)

print(
    pilot_completed.to_string(index=False)
    if len(pilot_completed)
    else "None"
)

# ------------------------------------------------------------
# 8. Save derived working file
# ------------------------------------------------------------

OUTPUT_PATH = "/content/derived_annotation_status.csv"

derived_df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved:", OUTPUT_PATH)

# ------------------------------------------------------------
# 9. Final status
# ------------------------------------------------------------

if (
    len(invalid_element_values) == 0
    and len(completed_without_label) == 0
    and len(incomplete_with_label) == 0
):
    print("\nSTATUS: PASS")
else:
    print("\nSTATUS: REVIEW REQUIRED")

DERIVE RELEVANCE FROM EXPECTED-ELEMENT RECOVERY
Invalid element judgment values: []

VALIDATION
---------------------------------------------------------------------------
Unique annotation units: 220
Completed units: 9
Incomplete units: 211
Units with derived relevance: 9
Units without derived relevance: 211
Completed units without relevance: 0
Incomplete units with relevance: 0

CURRENT COMPLETED ANNOTATION UNITS
---------------------------------------------------------------------------
annotation_unit_id query_id  expected_element_count  recovered_element_count derived_curriculum_relevance
             AU001      Q01                       1                        1                     Relevant
             AU002      Q01                       1                        1                     Relevant
             AU003      Q01                       1                        0                 Not Relevant
             AU016      Q06                       7                        1     

In [21]:
# ============================================================
# CELL 20 — DISPLAY REMAINING Q01 ANNOTATION UNITS
# ============================================================

import pandas as pd

WORKBOOK_PATH = "/content/retrieval_annotation_workbook.xlsx"

element_df = pd.read_excel(
    WORKBOOK_PATH,
    sheet_name="Element Annotation"
)

# Normalize existing annotation values
element_df["element_recovered"] = (
    element_df["element_recovered"]
    .astype("string")
    .str.strip()
)

# Select only Q01 judgments that have not yet been completed
q01_remaining = element_df[
    (element_df["query_id"] == "Q01")
    & (~element_df["element_recovered"].isin(["Yes", "No"]))
].copy()

print("Q01 — REMAINING HUMAN ANNOTATION")
print("=" * 80)
print("Remaining judgments:", len(q01_remaining))
print()

for _, row in q01_remaining.iterrows():

    print("=" * 80)
    print(
        f'{row["annotation_unit_id"]} | '
        f'{row["element_id"]}'
    )

    print("\nQUERY:")
    print(row["query_ar"])

    print("\nEXPECTED ELEMENT:")
    print(row["expected_element"])

    print("\nRETRIEVED TEXT:")
    print(row["retrieval_text"])

    print()

Q01 — REMAINING HUMAN ANNOTATION
Remaining judgments: 17

AU037 | E01

QUERY:
ماذا يحدث للمادة الوراثية DNA قبل انقسام الخلية؟

EXPECTED ELEMENT:
تتضاعف مادة الوراثة DNA قبل البدء بعملية الانقسام.

RETRIEVED TEXT:
قبل دخول الخلية في عملية الانقسام الخلوي تمر بمرحلة تحضيرية، تسمى الطور البيني، تنمو فيه الخلية ويزداد حجمها، وتتضاعف كمية المادة الوراثية (DNA).
الانقسام المتساوي: يحدث في جميع خلايا الكائنات الحية وحيدة الخلية لتتكاثر مثل البراميسيوم، وعديدة الخلايا مثل الإنسان؛ للنمو، وتعويض الخلايا التالفة.

AU038 | E01

QUERY:
ماذا يحدث للمادة الوراثية DNA قبل انقسام الخلية؟

EXPECTED ELEMENT:
تتضاعف مادة الوراثة DNA قبل البدء بعملية الانقسام.

RETRIEVED TEXT:
هل يستمر انقسام الخلايا بالسرعة نفسها مع التقدم في العمر؟
كثير من الخلايا في أجسامنا تنقسم وتتكاثر، فتنتج خلايا جديدة، قد تكون مطابقة للخلايا الأصلية، أو غير مطابقة. وقبل انقسام الخلايا تكون الكروموسومات مفردة، وعلى شكل خيوط طويلة رفيعة ومتشابكة داخل النواة في الخلايا، تسمى شبكة كروماتينية.
ولا يمكن مشاهدتها على هيئة كروموسومات مست

In [22]:
# ============================================================
# CELL 21 — WRITE REVIEWED Q01 JUDGMENTS + VALIDATE
# ============================================================

import pandas as pd
from openpyxl import load_workbook

WORKBOOK_PATH = "/content/retrieval_annotation_workbook.xlsx"

# Reviewed Q01 judgments
q01_judgments = {
    "AU001": "Yes",   # already annotated in pilot
    "AU002": "Yes",   # already annotated in pilot
    "AU003": "No",    # already annotated in pilot

    "AU037": "Yes",
    "AU038": "Yes",
    "AU039": "No",
    "AU073": "Yes",
    "AU074": "No",
    "AU075": "No",
    "AU095": "No",
    "AU096": "Yes",
    "AU097": "No",
    "AU118": "Yes",
    "AU119": "No",
    "AU134": "Yes",
    "AU157": "Yes",
    "AU170": "Yes",
    "AU179": "Yes",
    "AU180": "Yes",
    "AU181": "No",
}

print("WRITE REVIEWED Q01 JUDGMENTS")
print("=" * 75)

# ------------------------------------------------------------
# 1. Load workbook
# ------------------------------------------------------------

wb = load_workbook(WORKBOOK_PATH)
ws = wb["Element Annotation"]

# Locate required columns
headers = {
    cell.value: cell.column
    for cell in ws[1]
}

required_columns = [
    "annotation_unit_id",
    "query_id",
    "element_id",
    "element_recovered",
]

missing_columns = [
    col for col in required_columns
    if col not in headers
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

# ------------------------------------------------------------
# 2. Write judgments ONLY to Q01 / E01
# ------------------------------------------------------------

updated = []
not_found = []
unexpected_existing = []

for annotation_unit_id, judgment in q01_judgments.items():

    found = False

    for row in range(2, ws.max_row + 1):

        au = ws.cell(
            row=row,
            column=headers["annotation_unit_id"]
        ).value

        query_id = ws.cell(
            row=row,
            column=headers["query_id"]
        ).value

        element_id = ws.cell(
            row=row,
            column=headers["element_id"]
        ).value

        if (
            au == annotation_unit_id
            and query_id == "Q01"
            and element_id == "E01"
        ):
            found = True

            target = ws.cell(
                row=row,
                column=headers["element_recovered"]
            )

            existing = (
                str(target.value).strip()
                if target.value is not None
                else ""
            )

            # Protect against silently overwriting a conflicting judgment
            if existing in ["Yes", "No"] and existing != judgment:
                unexpected_existing.append(
                    (
                        annotation_unit_id,
                        existing,
                        judgment
                    )
                )
            else:
                target.value = judgment
                updated.append(annotation_unit_id)

            break

    if not found:
        not_found.append(annotation_unit_id)

# Do not save if a conflict was detected
if unexpected_existing:
    print("\nCONFLICTS DETECTED:")
    for item in unexpected_existing:
        print(
            f"{item[0]}: existing={item[1]}, "
            f"reviewed={item[2]}"
        )

    raise ValueError(
        "Workbook was NOT saved because conflicting "
        "existing judgments were detected."
    )

wb.save(WORKBOOK_PATH)

print("Rows written/confirmed:", len(updated))
print("Judgment IDs not found:", not_found)

# ------------------------------------------------------------
# 3. Reload and validate Q01
# ------------------------------------------------------------

element_df = pd.read_excel(
    WORKBOOK_PATH,
    sheet_name="Element Annotation"
)

element_df["element_recovered"] = (
    element_df["element_recovered"]
    .astype("string")
    .str.strip()
)

q01 = element_df[
    element_df["query_id"] == "Q01"
].copy()

valid_mask = q01["element_recovered"].isin(
    ["Yes", "No"]
)

yes_count = (
    q01["element_recovered"] == "Yes"
).sum()

no_count = (
    q01["element_recovered"] == "No"
).sum()

print("\nQ01 VALIDATION")
print("-" * 75)

print("Total Q01 annotation units:", len(q01))
print("Completed judgments:", int(valid_mask.sum()))
print("Remaining judgments:", int((~valid_mask).sum()))
print("Yes:", int(yes_count))
print("No:", int(no_count))

# Check exact reviewed values
errors = []

for annotation_unit_id, expected in q01_judgments.items():

    match = q01[
        q01["annotation_unit_id"]
        == annotation_unit_id
    ]

    if len(match) != 1:
        errors.append(
            f"{annotation_unit_id}: expected exactly one row, "
            f"found {len(match)}"
        )
        continue

    actual = match.iloc[0]["element_recovered"]

    if actual != expected:
        errors.append(
            f"{annotation_unit_id}: "
            f"actual={actual}, expected={expected}"
        )

print("Validation errors:", len(errors))

if errors:
    for error in errors:
        print(" -", error)

# ------------------------------------------------------------
# 4. Final status
# ------------------------------------------------------------

if (
    len(q01) == 20
    and valid_mask.sum() == 20
    and len(errors) == 0
    and len(not_found) == 0
):
    print("\nQ01 STATUS: PASS — FULLY ANNOTATED")
else:
    print("\nQ01 STATUS: REVIEW REQUIRED")

WRITE REVIEWED Q01 JUDGMENTS
Rows written/confirmed: 20
Judgment IDs not found: []

Q01 VALIDATION
---------------------------------------------------------------------------
Total Q01 annotation units: 20
Completed judgments: 20
Remaining judgments: 0
Yes: 12
No: 8
Validation errors: 0

Q01 STATUS: PASS — FULLY ANNOTATED


In [23]:
# ============================================================
# CELL 22 — DISPLAY REMAINING Q02 ANNOTATION UNITS
# ============================================================

import pandas as pd

WORKBOOK_PATH = "/content/retrieval_annotation_workbook.xlsx"

element_df = pd.read_excel(
    WORKBOOK_PATH,
    sheet_name="Element Annotation"
)

element_df["element_recovered"] = (
    element_df["element_recovered"]
    .astype("string")
    .str.strip()
)

q02_remaining = element_df[
    (element_df["query_id"] == "Q02")
    & (~element_df["element_recovered"].isin(["Yes", "No"]))
].copy()

print("Q02 — REMAINING HUMAN ANNOTATION")
print("=" * 80)

print("Remaining element judgments:", len(q02_remaining))
print(
    "Unique annotation units:",
    q02_remaining["annotation_unit_id"].nunique()
)
print()

for annotation_unit_id, group in q02_remaining.groupby(
    "annotation_unit_id",
    sort=False
):

    first = group.iloc[0]

    print("=" * 80)
    print(annotation_unit_id)

    print("\nQUERY:")
    print(first["query_ar"])

    print("\nEXPECTED ELEMENTS:")

    for _, row in group.iterrows():
        print(
            f'{row["element_id"]}: '
            f'{row["expected_element"]}'
        )

    print("\nRETRIEVED TEXT:")
    print(first["retrieval_text"])

    print()

Q02 — REMAINING HUMAN ANNOTATION
Remaining element judgments: 40
Unique annotation units: 20

AU004

QUERY:
مم يتكون الكروموسوم المتضاعف وكيف يرتبط مكوناه؟

EXPECTED ELEMENTS:
E01: يتكون الكروموسوم المتضاعف من كروماتيدين.
E02: يرتبط الكروماتيدان بنقطة تسمى السنترومير.

RETRIEVED TEXT:
الدور الاستوائي: تترتب الكروموسومات المتضاعفة فرادى في منتصف الخلية، وترتبط خيوط المغزل بسنترومير كل كروموسوم من القطبين.

AU005

QUERY:
مم يتكون الكروموسوم المتضاعف وكيف يرتبط مكوناه؟

EXPECTED ELEMENTS:
E01: يتكون الكروموسوم المتضاعف من كروماتيدين.
E02: يرتبط الكروماتيدان بنقطة تسمى السنترومير.

RETRIEVED TEXT:
حيث يظهر كل كروموسوم متضاعف تحت المجهر الإلكتروني مكوناً من كروماتيدين (كروماتيد أصلي وكروماتيد جديد). يرتبط كل كروماتيدين بنقطة تسمى السنترومير.
يوضح الشكل كروموسومات غير متضاعفة ثم كروموسومات متكاثرة متضاعفة، مع السنترومير والكروماتيدات الشقيقة.
أنواع انقسام الخلايا
المواد والأدوات: لوح كرتون أو خشبي رقيق (50 سم × 50 سم)، ومقص، وخيوط صوف ملونة، ومادة لاصقة.
خطوات العمل: 1- قص خيوط الصوف بأطوال 

In [24]:
# ============================================================
# CELL 23 — WRITE REVIEWED Q02 JUDGMENTS + VALIDATE
# ============================================================

import pandas as pd
from openpyxl import load_workbook

WORKBOOK_PATH = "/content/retrieval_annotation_workbook.xlsx"

q02_judgments = {
    "AU004": {"E01": "No",  "E02": "No"},
    "AU005": {"E01": "Yes", "E02": "Yes"},
    "AU006": {"E01": "No",  "E02": "No"},
    "AU040": {"E01": "Yes", "E02": "Yes"},
    "AU041": {"E01": "No",  "E02": "No"},
    "AU042": {"E01": "No",  "E02": "No"},
    "AU076": {"E01": "No",  "E02": "No"},
    "AU077": {"E01": "No",  "E02": "No"},
    "AU098": {"E01": "No",  "E02": "No"},
    "AU099": {"E01": "No",  "E02": "No"},
    "AU120": {"E01": "No",  "E02": "No"},
    "AU135": {"E01": "No",  "E02": "No"},
    "AU146": {"E01": "No",  "E02": "No"},
    "AU147": {"E01": "No",  "E02": "No"},
    "AU158": {"E01": "No",  "E02": "No"},
    "AU171": {"E01": "No",  "E02": "No"},
    "AU172": {"E01": "No",  "E02": "No"},
    "AU182": {"E01": "No",  "E02": "No"},
    "AU190": {"E01": "No",  "E02": "No"},
    "AU195": {"E01": "No",  "E02": "No"},
}

print("WRITE REVIEWED Q02 JUDGMENTS")
print("=" * 75)

wb = load_workbook(WORKBOOK_PATH)
ws = wb["Element Annotation"]

headers = {cell.value: cell.column for cell in ws[1]}

required = [
    "annotation_unit_id",
    "query_id",
    "element_id",
    "element_recovered"
]

missing = [c for c in required if c not in headers]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

updated = 0
not_found = []
conflicts = []

for au, elements in q02_judgments.items():
    for element_id, judgment in elements.items():

        found = False

        for row in range(2, ws.max_row + 1):

            if (
                ws.cell(row, headers["annotation_unit_id"]).value == au
                and ws.cell(row, headers["query_id"]).value == "Q02"
                and ws.cell(row, headers["element_id"]).value == element_id
            ):
                found = True
                target = ws.cell(
                    row,
                    headers["element_recovered"]
                )

                existing = (
                    str(target.value).strip()
                    if target.value is not None
                    else ""
                )

                if existing in ["Yes", "No"] and existing != judgment:
                    conflicts.append(
                        (au, element_id, existing, judgment)
                    )
                else:
                    target.value = judgment
                    updated += 1

                break

        if not found:
            not_found.append((au, element_id))

if conflicts:
    print("CONFLICTS:", conflicts)
    raise ValueError(
        "Workbook NOT saved because conflicts were detected."
    )

wb.save(WORKBOOK_PATH)

print("Rows written/confirmed:", updated)
print("Judgments not found:", not_found)

# ---------------- Validation ----------------

df = pd.read_excel(
    WORKBOOK_PATH,
    sheet_name="Element Annotation"
)

df["element_recovered"] = (
    df["element_recovered"]
    .astype("string")
    .str.strip()
)

q02 = df[df["query_id"] == "Q02"].copy()

complete = q02["element_recovered"].isin(["Yes", "No"])

errors = []

for au, elements in q02_judgments.items():
    for element_id, expected in elements.items():

        match = q02[
            (q02["annotation_unit_id"] == au)
            & (q02["element_id"] == element_id)
        ]

        if len(match) != 1:
            errors.append(
                f"{au}/{element_id}: found {len(match)} rows"
            )
            continue

        actual = match.iloc[0]["element_recovered"]

        if actual != expected:
            errors.append(
                f"{au}/{element_id}: "
                f"actual={actual}, expected={expected}"
            )

print("\nQ02 VALIDATION")
print("-" * 75)

print("Total element judgments:", len(q02))
print("Completed judgments:", int(complete.sum()))
print("Remaining judgments:", int((~complete).sum()))
print(
    "Yes:",
    int((q02["element_recovered"] == "Yes").sum())
)
print(
    "No:",
    int((q02["element_recovered"] == "No").sum())
)
print("Validation errors:", len(errors))

if errors:
    for error in errors:
        print(" -", error)

if (
    len(q02) == 40
    and complete.sum() == 40
    and len(errors) == 0
    and len(not_found) == 0
):
    print("\nQ02 STATUS: PASS — FULLY ANNOTATED")
else:
    print("\nQ02 STATUS: REVIEW REQUIRED")

WRITE REVIEWED Q02 JUDGMENTS
Rows written/confirmed: 40
Judgments not found: []

Q02 VALIDATION
---------------------------------------------------------------------------
Total element judgments: 40
Completed judgments: 40
Remaining judgments: 0
Yes: 4
No: 36
Validation errors: 0

Q02 STATUS: PASS — FULLY ANNOTATED


In [25]:
# ============================================================
# CELL 24 — DISPLAY REMAINING Q03 ANNOTATION UNITS
# ============================================================

import pandas as pd

WORKBOOK_PATH = "/content/retrieval_annotation_workbook.xlsx"

element_df = pd.read_excel(
    WORKBOOK_PATH,
    sheet_name="Element Annotation"
)

element_df["element_recovered"] = (
    element_df["element_recovered"]
    .astype("string")
    .str.strip()
)

q03_remaining = element_df[
    (element_df["query_id"] == "Q03")
    & (~element_df["element_recovered"].isin(["Yes", "No"]))
].copy()

print("Q03 — REMAINING HUMAN ANNOTATION")
print("=" * 80)

print(
    "Remaining element judgments:",
    len(q03_remaining)
)

print(
    "Unique annotation units:",
    q03_remaining["annotation_unit_id"].nunique()
)

print()

for annotation_unit_id, group in q03_remaining.groupby(
    "annotation_unit_id",
    sort=False
):

    first = group.iloc[0]

    print("=" * 80)
    print(annotation_unit_id)

    print("\nQUERY:")
    print(first["query_ar"])

    print("\nEXPECTED ELEMENTS:")

    for _, row in group.iterrows():
        print(
            f'{row["element_id"]}: '
            f'{row["expected_element"]}'
        )

    print("\nRETRIEVED TEXT:")
    print(first["retrieval_text"])

    print()

Q03 — REMAINING HUMAN ANNOTATION
Remaining element judgments: 63
Unique annotation units: 21

AU007

QUERY:
ما نوع الانقسام المرتبط بالخلايا الجسمية وما نتيجته؟

EXPECTED ELEMENTS:
E01: تنقسم الخلايا الجسمية بطريقة الانقسام المتساوي.
E02: ينتج عن الانقسام المتساوي خليتان ابنتان.
E03: تحتوي الخليتان الناتجتان على نفس عدد كروموسومات الخلية الأم.

RETRIEVED TEXT:
الانقسام المتساوي: يحدث في جميع خلايا الكائنات الحية وحيدة الخلية لتتكاثر مثل البراميسيوم، وعديدة الخلايا مثل الإنسان؛ للنمو، وتعويض الخلايا التالفة.

AU008

QUERY:
ما نوع الانقسام المرتبط بالخلايا الجسمية وما نتيجته؟

EXPECTED ELEMENTS:
E01: تنقسم الخلايا الجسمية بطريقة الانقسام المتساوي.
E02: ينتج عن الانقسام المتساوي خليتان ابنتان.
E03: تحتوي الخليتان الناتجتان على نفس عدد كروموسومات الخلية الأم.

RETRIEVED TEXT:
تتكون أجسام الكائنات الحية عديدة الخلايا التي تتكاثر جنسياً من نوعين رئيسين من الخلايا، هما:
الخلايا الجسمية: تشمل خلايا جسم الكائنات الحية الراقية، وتنقسم النواة بطريقة الانقسام المتساوي، وينتج عنه خليتان (ابنتان)، ت

In [26]:
# ============================================================
# CELL 25 — WRITE REVIEWED Q03 JUDGMENTS + VALIDATE
# ============================================================

import pandas as pd
from openpyxl import load_workbook

WORKBOOK_PATH = "/content/retrieval_annotation_workbook.xlsx"

q03_judgments = {
    "AU007": {"E01": "No",  "E02": "No",  "E03": "No"},
    "AU008": {"E01": "Yes", "E02": "Yes", "E03": "Yes"},
    "AU009": {"E01": "No",  "E02": "No",  "E03": "No"},
    "AU043": {"E01": "No",  "E02": "No",  "E03": "No"},
    "AU044": {"E01": "Yes", "E02": "Yes", "E03": "Yes"},
    "AU045": {"E01": "No",  "E02": "Yes", "E03": "Yes"},
    "AU078": {"E01": "No",  "E02": "No",  "E03": "No"},
    "AU100": {"E01": "No",  "E02": "Yes", "E03": "Yes"},
    "AU101": {"E01": "No",  "E02": "No",  "E03": "No"},
    "AU121": {"E01": "No",  "E02": "Yes", "E03": "Yes"},
    "AU136": {"E01": "No",  "E02": "Yes", "E03": "Yes"},
    "AU148": {"E01": "No",  "E02": "No",  "E03": "No"},
    "AU149": {"E01": "No",  "E02": "No",  "E03": "No"},
    "AU159": {"E01": "No",  "E02": "No",  "E03": "No"},
    "AU160": {"E01": "No",  "E02": "No",  "E03": "No"},
    "AU173": {"E01": "No",  "E02": "No",  "E03": "No"},
    "AU191": {"E01": "No",  "E02": "No",  "E03": "No"},
    "AU196": {"E01": "No",  "E02": "No",  "E03": "No"},
    "AU203": {"E01": "No",  "E02": "No",  "E03": "No"},
    "AU208": {"E01": "No",  "E02": "No",  "E03": "No"},
    "AU211": {"E01": "No",  "E02": "Yes", "E03": "Yes"},
}

print("WRITE REVIEWED Q03 JUDGMENTS")
print("=" * 75)

wb = load_workbook(WORKBOOK_PATH)
ws = wb["Element Annotation"]

headers = {cell.value: cell.column for cell in ws[1]}

required = [
    "annotation_unit_id",
    "query_id",
    "element_id",
    "element_recovered"
]

missing = [c for c in required if c not in headers]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

updated = 0
not_found = []
conflicts = []

for au, elements in q03_judgments.items():
    for element_id, judgment in elements.items():

        found = False

        for row in range(2, ws.max_row + 1):

            if (
                ws.cell(row, headers["annotation_unit_id"]).value == au
                and ws.cell(row, headers["query_id"]).value == "Q03"
                and ws.cell(row, headers["element_id"]).value == element_id
            ):
                found = True

                target = ws.cell(
                    row,
                    headers["element_recovered"]
                )

                existing = (
                    str(target.value).strip()
                    if target.value is not None
                    else ""
                )

                if (
                    existing in ["Yes", "No"]
                    and existing != judgment
                ):
                    conflicts.append(
                        (au, element_id, existing, judgment)
                    )
                else:
                    target.value = judgment
                    updated += 1

                break

        if not found:
            not_found.append((au, element_id))

if conflicts:
    print("CONFLICTS:", conflicts)
    raise ValueError(
        "Workbook NOT saved because conflicts were detected."
    )

wb.save(WORKBOOK_PATH)

print("Rows written/confirmed:", updated)
print("Judgments not found:", not_found)

# ============================================================
# VALIDATION
# ============================================================

df = pd.read_excel(
    WORKBOOK_PATH,
    sheet_name="Element Annotation"
)

df["element_recovered"] = (
    df["element_recovered"]
    .astype("string")
    .str.strip()
)

q03 = df[df["query_id"] == "Q03"].copy()

complete = q03["element_recovered"].isin(
    ["Yes", "No"]
)

errors = []

for au, elements in q03_judgments.items():
    for element_id, expected in elements.items():

        match = q03[
            (q03["annotation_unit_id"] == au)
            & (q03["element_id"] == element_id)
        ]

        if len(match) != 1:
            errors.append(
                f"{au}/{element_id}: found {len(match)} rows"
            )
            continue

        actual = match.iloc[0]["element_recovered"]

        if actual != expected:
            errors.append(
                f"{au}/{element_id}: "
                f"actual={actual}, expected={expected}"
            )

print("\nQ03 VALIDATION")
print("-" * 75)

print("Total element judgments:", len(q03))
print("Completed judgments:", int(complete.sum()))
print("Remaining judgments:", int((~complete).sum()))
print(
    "Yes:",
    int((q03["element_recovered"] == "Yes").sum())
)
print(
    "No:",
    int((q03["element_recovered"] == "No").sum())
)
print("Validation errors:", len(errors))

if errors:
    for error in errors:
        print(" -", error)

if (
    len(q03) == 63
    and complete.sum() == 63
    and len(errors) == 0
    and len(not_found) == 0
):
    print("\nQ03 STATUS: PASS — FULLY ANNOTATED")
else:
    print("\nQ03 STATUS: REVIEW REQUIRED")

WRITE REVIEWED Q03 JUDGMENTS
Rows written/confirmed: 63
Judgments not found: []

Q03 VALIDATION
---------------------------------------------------------------------------
Total element judgments: 63
Completed judgments: 63
Remaining judgments: 0
Yes: 16
No: 47
Validation errors: 0

Q03 STATUS: PASS — FULLY ANNOTATED


In [27]:
# ============================================================
# CELL 26 — DISPLAY REMAINING Q04 ANNOTATION UNITS
# ============================================================

import pandas as pd

WORKBOOK_PATH = "/content/retrieval_annotation_workbook.xlsx"

element_df = pd.read_excel(
    WORKBOOK_PATH,
    sheet_name="Element Annotation"
)

element_df["element_recovered"] = (
    element_df["element_recovered"]
    .astype("string")
    .str.strip()
)

q04_remaining = element_df[
    (element_df["query_id"] == "Q04")
    & (~element_df["element_recovered"].isin(["Yes", "No"]))
].copy()

print("Q04 — REMAINING HUMAN ANNOTATION")
print("=" * 80)

print("Remaining element judgments:", len(q04_remaining))
print(
    "Unique annotation units:",
    q04_remaining["annotation_unit_id"].nunique()
)
print()

for annotation_unit_id, group in q04_remaining.groupby(
    "annotation_unit_id",
    sort=False
):

    first = group.iloc[0]

    print("=" * 80)
    print(annotation_unit_id)

    print("\nQUERY:")
    print(first["query_ar"])

    print("\nEXPECTED ELEMENTS:")

    for _, row in group.iterrows():
        print(
            f'{row["element_id"]}: '
            f'{row["expected_element"]}'
        )

    print("\nRETRIEVED TEXT:")
    print(first["retrieval_text"])

    print()

Q04 — REMAINING HUMAN ANNOTATION
Remaining element judgments: 57
Unique annotation units: 19

AU010

QUERY:
ماذا يحدث للخلية في الطور البيني؟

EXPECTED ELEMENTS:
E01: تنمو الخلية.
E02: يزداد حجم الخلية.
E03: تتضاعف كمية المادة الوراثية DNA.

RETRIEVED TEXT:
ماذا يحدث إذا لم تمر الخلية في الطور البيني؟
من خلال الشكل أعلاه، حدد الأدوار التي يمر بها الانقسام المتساوي؟
ما أهمية المريكزات، وخيوط المغزل في عملية الانقسام المتساوي؟
لماذا تترتب الكروموسومات فرادى في منتصف الخلية في الدور الاستوائي؟
هل الخلية المنقسمة حيوانية أم نباتية؟ فسر إجابتك.
ما عدد الخلايا الناتجة عن الانقسام؟
ما عدد الكروموسومات في الخلايا الناتجة؟

AU011

QUERY:
ماذا يحدث للخلية في الطور البيني؟

EXPECTED ELEMENTS:
E01: تنمو الخلية.
E02: يزداد حجم الخلية.
E03: تتضاعف كمية المادة الوراثية DNA.

RETRIEVED TEXT:
ادرس مراحل الانقسام المتساوي للخلية في الشكل أدناه، ثم أجب عن الأسئلة التي تليه.
الدور التمهيدي: تظهر كروموسومات قصيرة وغليظة متضاعفة، يختفي الغلاف النووي، وتختفي النوية.

AU012

QUERY:
ماذا يحدث للخلية في الطور ا

In [28]:
# ============================================================
# CELL 27 — WRITE REVIEWED Q04 JUDGMENTS + VALIDATE
# ============================================================

import pandas as pd
from openpyxl import load_workbook

WORKBOOK_PATH = "/content/retrieval_annotation_workbook.xlsx"

yes_units = {
    "AU048", "AU080", "AU122", "AU150",
    "AU161", "AU183", "AU197"
}

all_units = {
    "AU010", "AU011", "AU012", "AU046", "AU047",
    "AU048", "AU079", "AU080", "AU102", "AU103",
    "AU122", "AU123", "AU137", "AU150", "AU151",
    "AU161", "AU183", "AU184", "AU197"
}

q04_judgments = {
    au: {
        "E01": "Yes" if au in yes_units else "No",
        "E02": "Yes" if au in yes_units else "No",
        "E03": "Yes" if au in yes_units else "No",
    }
    for au in all_units
}

print("WRITE REVIEWED Q04 JUDGMENTS")
print("=" * 75)

wb = load_workbook(WORKBOOK_PATH)
ws = wb["Element Annotation"]

headers = {cell.value: cell.column for cell in ws[1]}

required = [
    "annotation_unit_id",
    "query_id",
    "element_id",
    "element_recovered"
]

missing = [c for c in required if c not in headers]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

updated = 0
not_found = []
conflicts = []

for au, elements in q04_judgments.items():
    for element_id, judgment in elements.items():

        found = False

        for row in range(2, ws.max_row + 1):

            if (
                ws.cell(row, headers["annotation_unit_id"]).value == au
                and ws.cell(row, headers["query_id"]).value == "Q04"
                and ws.cell(row, headers["element_id"]).value == element_id
            ):
                found = True

                target = ws.cell(
                    row,
                    headers["element_recovered"]
                )

                existing = (
                    str(target.value).strip()
                    if target.value is not None
                    else ""
                )

                if existing in ["Yes", "No"] and existing != judgment:
                    conflicts.append(
                        (au, element_id, existing, judgment)
                    )
                else:
                    target.value = judgment
                    updated += 1

                break

        if not found:
            not_found.append((au, element_id))

if conflicts:
    print("CONFLICTS:", conflicts)
    raise ValueError(
        "Workbook NOT saved because conflicts were detected."
    )

wb.save(WORKBOOK_PATH)

print("Rows written/confirmed:", updated)
print("Judgments not found:", not_found)

# ============================================================
# VALIDATION
# ============================================================

df = pd.read_excel(
    WORKBOOK_PATH,
    sheet_name="Element Annotation"
)

df["element_recovered"] = (
    df["element_recovered"]
    .astype("string")
    .str.strip()
)

q04 = df[df["query_id"] == "Q04"].copy()

complete = q04["element_recovered"].isin(["Yes", "No"])

errors = []

for au, elements in q04_judgments.items():
    for element_id, expected in elements.items():

        match = q04[
            (q04["annotation_unit_id"] == au)
            & (q04["element_id"] == element_id)
        ]

        if len(match) != 1:
            errors.append(
                f"{au}/{element_id}: found {len(match)} rows"
            )
            continue

        actual = match.iloc[0]["element_recovered"]

        if actual != expected:
            errors.append(
                f"{au}/{element_id}: "
                f"actual={actual}, expected={expected}"
            )

print("\nQ04 VALIDATION")
print("-" * 75)
print("Total element judgments:", len(q04))
print("Completed judgments:", int(complete.sum()))
print("Remaining judgments:", int((~complete).sum()))
print("Yes:", int((q04["element_recovered"] == "Yes").sum()))
print("No:", int((q04["element_recovered"] == "No").sum()))
print("Validation errors:", len(errors))

if errors:
    for error in errors:
        print(" -", error)

if (
    len(q04) == 57
    and complete.sum() == 57
    and len(errors) == 0
    and len(not_found) == 0
):
    print("\nQ04 STATUS: PASS — FULLY ANNOTATED")
else:
    print("\nQ04 STATUS: REVIEW REQUIRED")

WRITE REVIEWED Q04 JUDGMENTS
Rows written/confirmed: 57
Judgments not found: []

Q04 VALIDATION
---------------------------------------------------------------------------
Total element judgments: 57
Completed judgments: 57
Remaining judgments: 0
Yes: 21
No: 36
Validation errors: 0

Q04 STATUS: PASS — FULLY ANNOTATED


In [29]:
# ============================================================
# CELL 28 — DISPLAY REMAINING Q05 ANNOTATION UNITS
# ============================================================

import pandas as pd

WORKBOOK_PATH = "/content/retrieval_annotation_workbook.xlsx"

element_df = pd.read_excel(
    WORKBOOK_PATH,
    sheet_name="Element Annotation"
)

element_df["element_recovered"] = (
    element_df["element_recovered"]
    .astype("string")
    .str.strip()
)

q05_remaining = element_df[
    (element_df["query_id"] == "Q05")
    & (~element_df["element_recovered"].isin(["Yes", "No"]))
].copy()

print("Q05 — REMAINING HUMAN ANNOTATION")
print("=" * 80)

print(
    "Remaining element judgments:",
    len(q05_remaining)
)

print(
    "Unique annotation units:",
    q05_remaining["annotation_unit_id"].nunique()
)

print()

for annotation_unit_id, group in q05_remaining.groupby(
    "annotation_unit_id",
    sort=False
):

    first = group.iloc[0]

    print("=" * 80)
    print(annotation_unit_id)

    print("\nQUERY:")
    print(first["query_ar"])

    print("\nEXPECTED ELEMENTS:")

    for _, row in group.iterrows():
        print(
            f'{row["element_id"]}: '
            f'{row["expected_element"]}'
        )

    print("\nRETRIEVED TEXT:")
    print(first["retrieval_text"])

    print()

Q05 — REMAINING HUMAN ANNOTATION
Remaining element judgments: 51
Unique annotation units: 17

AU013

QUERY:
ماذا يحدث للكروماتيدات الشقيقة في الدور الانفصالي؟

EXPECTED ELEMENTS:
E01: تتباعد الكروماتيدات الشقيقة.
E02: يحدث التباعد بفعل انكماش خيوط المغزل.
E03: يتجه كل منها إلى قطب.

RETRIEVED TEXT:
الدور الانفصالي: تتباعد الكروماتيدات الشقيقة بفعل انكماش خيوط المغزل كل إلى قطب.

AU014

QUERY:
ماذا يحدث للكروماتيدات الشقيقة في الدور الانفصالي؟

EXPECTED ELEMENTS:
E01: تتباعد الكروماتيدات الشقيقة.
E02: يحدث التباعد بفعل انكماش خيوط المغزل.
E03: يتجه كل منها إلى قطب.

RETRIEVED TEXT:
ادرس مراحل الانقسام المتساوي للخلية في الشكل أدناه، ثم أجب عن الأسئلة التي تليه.
الدور التمهيدي: تظهر كروموسومات قصيرة وغليظة متضاعفة، يختفي الغلاف النووي، وتختفي النوية.

AU015

QUERY:
ماذا يحدث للكروماتيدات الشقيقة في الدور الانفصالي؟

EXPECTED ELEMENTS:
E01: تتباعد الكروماتيدات الشقيقة.
E02: يحدث التباعد بفعل انكماش خيوط المغزل.
E03: يتجه كل منها إلى قطب.

RETRIEVED TEXT:
ماذا يحدث إذا لم تمر الخلية في الط

In [30]:
# ============================================================
# CELL 29 — WRITE REVIEWED Q05 JUDGMENTS + VALIDATE
# ============================================================

import pandas as pd
from openpyxl import load_workbook

WORKBOOK_PATH = "/content/retrieval_annotation_workbook.xlsx"

yes_units = {
    "AU013", "AU049", "AU081", "AU104",
    "AU124", "AU162", "AU198"
}

all_units = {
    "AU013", "AU014", "AU015", "AU049", "AU050",
    "AU051", "AU081", "AU082", "AU104", "AU105",
    "AU124", "AU125", "AU138", "AU162", "AU163",
    "AU198", "AU204"
}

q05_judgments = {
    au: {
        "E01": "Yes" if au in yes_units else "No",
        "E02": "Yes" if au in yes_units else "No",
        "E03": "Yes" if au in yes_units else "No",
    }
    for au in all_units
}

print("WRITE REVIEWED Q05 JUDGMENTS")
print("=" * 75)

wb = load_workbook(WORKBOOK_PATH)
ws = wb["Element Annotation"]

headers = {cell.value: cell.column for cell in ws[1]}

required = [
    "annotation_unit_id",
    "query_id",
    "element_id",
    "element_recovered"
]

missing = [c for c in required if c not in headers]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

updated = 0
not_found = []
conflicts = []

for au, elements in q05_judgments.items():
    for element_id, judgment in elements.items():

        found = False

        for row in range(2, ws.max_row + 1):

            if (
                ws.cell(row, headers["annotation_unit_id"]).value == au
                and ws.cell(row, headers["query_id"]).value == "Q05"
                and ws.cell(row, headers["element_id"]).value == element_id
            ):
                found = True

                target = ws.cell(
                    row,
                    headers["element_recovered"]
                )

                existing = (
                    str(target.value).strip()
                    if target.value is not None
                    else ""
                )

                if (
                    existing in ["Yes", "No"]
                    and existing != judgment
                ):
                    conflicts.append(
                        (au, element_id, existing, judgment)
                    )
                else:
                    target.value = judgment
                    updated += 1

                break

        if not found:
            not_found.append((au, element_id))

if conflicts:
    print("CONFLICTS:", conflicts)
    raise ValueError(
        "Workbook NOT saved because conflicts were detected."
    )

wb.save(WORKBOOK_PATH)

print("Rows written/confirmed:", updated)
print("Judgments not found:", not_found)

# ============================================================
# VALIDATION
# ============================================================

df = pd.read_excel(
    WORKBOOK_PATH,
    sheet_name="Element Annotation"
)

df["element_recovered"] = (
    df["element_recovered"]
    .astype("string")
    .str.strip()
)

q05 = df[df["query_id"] == "Q05"].copy()

complete = q05["element_recovered"].isin(["Yes", "No"])

errors = []

for au, elements in q05_judgments.items():
    for element_id, expected in elements.items():

        match = q05[
            (q05["annotation_unit_id"] == au)
            & (q05["element_id"] == element_id)
        ]

        if len(match) != 1:
            errors.append(
                f"{au}/{element_id}: found {len(match)} rows"
            )
            continue

        actual = match.iloc[0]["element_recovered"]

        if actual != expected:
            errors.append(
                f"{au}/{element_id}: "
                f"actual={actual}, expected={expected}"
            )

print("\nQ05 VALIDATION")
print("-" * 75)

print("Total element judgments:", len(q05))
print("Completed judgments:", int(complete.sum()))
print("Remaining judgments:", int((~complete).sum()))
print("Yes:", int((q05["element_recovered"] == "Yes").sum()))
print("No:", int((q05["element_recovered"] == "No").sum()))
print("Validation errors:", len(errors))

if errors:
    for error in errors:
        print(" -", error)

if (
    len(q05) == 51
    and complete.sum() == 51
    and len(errors) == 0
    and len(not_found) == 0
):
    print("\nQ05 STATUS: PASS — FULLY ANNOTATED")
else:
    print("\nQ05 STATUS: REVIEW REQUIRED")

WRITE REVIEWED Q05 JUDGMENTS
Rows written/confirmed: 51
Judgments not found: []

Q05 VALIDATION
---------------------------------------------------------------------------
Total element judgments: 51
Completed judgments: 51
Remaining judgments: 0
Yes: 21
No: 30
Validation errors: 0

Q05 STATUS: PASS — FULLY ANNOTATED


In [31]:
# ============================================================
# CELL 30 — DISPLAY REMAINING Q06 ANNOTATION UNITS
# ============================================================

import pandas as pd

WORKBOOK_PATH = "/content/retrieval_annotation_workbook.xlsx"

element_df = pd.read_excel(
    WORKBOOK_PATH,
    sheet_name="Element Annotation"
)

element_df["element_recovered"] = (
    element_df["element_recovered"]
    .astype("string")
    .str.strip()
)

q06_remaining = element_df[
    (element_df["query_id"] == "Q06")
    & (~element_df["element_recovered"].isin(["Yes", "No"]))
].copy()

print("Q06 — REMAINING HUMAN ANNOTATION")
print("=" * 80)

print(
    "Remaining element judgments:",
    len(q06_remaining)
)

print(
    "Unique annotation units:",
    q06_remaining["annotation_unit_id"].nunique()
)

print()

for annotation_unit_id, group in q06_remaining.groupby(
    "annotation_unit_id",
    sort=False
):

    first = group.iloc[0]

    print("=" * 80)
    print(annotation_unit_id)

    print("\nQUERY:")
    print(first["query_ar"])

    print("\nEXPECTED ELEMENTS:")

    for _, row in group.iterrows():
        print(
            f'{row["element_id"]}: '
            f'{row["expected_element"]}'
        )

    print("\nRETRIEVED TEXT:")
    print(first["retrieval_text"])

    print()

Q06 — REMAINING HUMAN ANNOTATION
Remaining element judgments: 126
Unique annotation units: 18

AU018

QUERY:
ما مراحل الانقسام المتساوي وما الذي يحدث خلالها حتى تنتج خليتان؟

EXPECTED ELEMENTS:
E01: الدور التمهيدي.
E02: الدور الاستوائي.
E03: الدور الانفصالي.
E04: الدور النهائي.
E05: انقسام السيتوبلازم.
E06: تنتج خليتان متماثلتان.
E07: تحتوي كل خلية ناتجة على نفس عدد كروموسومات الخلية الأم.

RETRIEVED TEXT:
الدور النهائي: تتجمع الكروموسومات الوليدة في منتصف كل خلية جديدة، ويبدأ الغلاف النووي بالتكون والسيتوبلازم بالانقسام.
انقسام السيتوبلازم: ينقسم السيتوبلازم وتنتج خليتان متماثلتان تحوي كل منهما على نفس عدد كروموسومات الخلية الأم.

AU052

QUERY:
ما مراحل الانقسام المتساوي وما الذي يحدث خلالها حتى تنتج خليتان؟

EXPECTED ELEMENTS:
E01: الدور التمهيدي.
E02: الدور الاستوائي.
E03: الدور الانفصالي.
E04: الدور النهائي.
E05: انقسام السيتوبلازم.
E06: تنتج خليتان متماثلتان.
E07: تحتوي كل خلية ناتجة على نفس عدد كروموسومات الخلية الأم.

RETRIEVED TEXT:
الانقسام المتساوي: يحدث في جميع خلايا الكائنا

In [32]:
# ============================================================
# CELL 31 — WRITE REVIEWED Q06 JUDGMENTS + VALIDATE
# ============================================================

import pandas as pd
from openpyxl import load_workbook

WORKBOOK_PATH = "/content/retrieval_annotation_workbook.xlsx"

q06_judgments = {
    "AU018": ["No","No","No","Yes","Yes","Yes","Yes"],
    "AU052": ["Yes","No","No","No","No","No","No"],
    "AU053": ["No","No","No","No","Yes","Yes","Yes"],
    "AU054": ["No","No","Yes","Yes","Yes","Yes","Yes"],
    "AU083": ["No","No","No","No","No","No","No"],
    "AU084": ["Yes","Yes","Yes","Yes","Yes","Yes","Yes"],
    "AU106": ["Yes","Yes","Yes","Yes","Yes","Yes","Yes"],
    "AU107": ["No","No","No","No","No","Yes","Yes"],
    "AU126": ["Yes","Yes","Yes","Yes","Yes","Yes","Yes"],
    "AU127": ["No","No","No","No","No","No","No"],
    "AU139": ["No","No","No","No","No","No","No"],
    "AU152": ["No","No","No","No","No","No","No"],
    "AU164": ["No","No","No","No","No","No","No"],
    "AU185": ["No","No","No","No","No","No","No"],
    "AU192": ["No","No","No","No","No","No","No"],
    "AU199": ["Yes","Yes","Yes","Yes","Yes","Yes","Yes"],
    "AU200": ["No","No","Yes","Yes","Yes","No","No"],
    "AU205": ["No","No","No","No","No","No","No"],
}

q06_judgments = {
    au: {
        f"E{i:02d}": value
        for i, value in enumerate(values, start=1)
    }
    for au, values in q06_judgments.items()
}

print("WRITE REVIEWED Q06 JUDGMENTS")
print("=" * 75)

wb = load_workbook(WORKBOOK_PATH)
ws = wb["Element Annotation"]

headers = {cell.value: cell.column for cell in ws[1]}

required = [
    "annotation_unit_id",
    "query_id",
    "element_id",
    "element_recovered"
]

missing = [c for c in required if c not in headers]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

updated = 0
not_found = []
conflicts = []

for au, elements in q06_judgments.items():
    for element_id, judgment in elements.items():

        found = False

        for row in range(2, ws.max_row + 1):

            if (
                ws.cell(row, headers["annotation_unit_id"]).value == au
                and ws.cell(row, headers["query_id"]).value == "Q06"
                and ws.cell(row, headers["element_id"]).value == element_id
            ):
                found = True

                target = ws.cell(
                    row,
                    headers["element_recovered"]
                )

                existing = (
                    str(target.value).strip()
                    if target.value is not None
                    else ""
                )

                if (
                    existing in ["Yes", "No"]
                    and existing != judgment
                ):
                    conflicts.append(
                        (au, element_id, existing, judgment)
                    )
                else:
                    target.value = judgment
                    updated += 1

                break

        if not found:
            not_found.append((au, element_id))

if conflicts:
    print("CONFLICTS:", conflicts)
    raise ValueError(
        "Workbook NOT saved because conflicts were detected."
    )

wb.save(WORKBOOK_PATH)

print("Rows written/confirmed:", updated)
print("Judgments not found:", not_found)

# ============================================================
# VALIDATION
# ============================================================

df = pd.read_excel(
    WORKBOOK_PATH,
    sheet_name="Element Annotation"
)

df["element_recovered"] = (
    df["element_recovered"]
    .astype("string")
    .str.strip()
)

q06 = df[df["query_id"] == "Q06"].copy()

complete = q06["element_recovered"].isin(["Yes", "No"])

errors = []

for au, elements in q06_judgments.items():
    for element_id, expected in elements.items():

        match = q06[
            (q06["annotation_unit_id"] == au)
            & (q06["element_id"] == element_id)
        ]

        if len(match) != 1:
            errors.append(
                f"{au}/{element_id}: found {len(match)} rows"
            )
            continue

        actual = match.iloc[0]["element_recovered"]

        if actual != expected:
            errors.append(
                f"{au}/{element_id}: "
                f"actual={actual}, expected={expected}"
            )

print("\nQ06 VALIDATION")
print("-" * 75)

print("Total element judgments:", len(q06))
print("Completed judgments:", int(complete.sum()))
print("Remaining judgments:", int((~complete).sum()))
print("Yes:", int((q06["element_recovered"] == "Yes").sum()))
print("No:", int((q06["element_recovered"] == "No").sum()))
print("Validation errors:", len(errors))

if errors:
    for error in errors:
        print(" -", error)

if (
    len(q06) == 126
    and complete.sum() == 126
    and len(errors) == 0
    and len(not_found) == 0
):
    print("\nQ06 STATUS: PASS — FULLY ANNOTATED")
else:
    print("\nQ06 STATUS: REVIEW REQUIRED")

WRITE REVIEWED Q06 JUDGMENTS
Rows written/confirmed: 126
Judgments not found: []

Q06 VALIDATION
---------------------------------------------------------------------------
Total element judgments: 140
Completed judgments: 140
Remaining judgments: 0
Yes: 47
No: 93
Validation errors: 0

Q06 STATUS: REVIEW REQUIRED


In [33]:
# ============================================================
# CELL 31B — FINAL Q06 COMPLETENESS VALIDATION
# ============================================================

import pandas as pd

WORKBOOK_PATH = "/content/retrieval_annotation_workbook.xlsx"

df = pd.read_excel(
    WORKBOOK_PATH,
    sheet_name="Element Annotation"
)

df["element_recovered"] = (
    df["element_recovered"]
    .astype("string")
    .str.strip()
)

q06 = df[df["query_id"] == "Q06"].copy()

valid = q06["element_recovered"].isin(["Yes", "No"])

print("Q06 FINAL VALIDATION")
print("=" * 75)

print("Total Q06 element judgments:", len(q06))
print("Completed judgments:", int(valid.sum()))
print("Remaining judgments:", int((~valid).sum()))

print(
    "Unique annotation units:",
    q06["annotation_unit_id"].nunique()
)

print(
    "Yes:",
    int((q06["element_recovered"] == "Yes").sum())
)

print(
    "No:",
    int((q06["element_recovered"] == "No").sum())
)

print("\nPILOT UNITS:")
pilot = q06[
    q06["annotation_unit_id"].isin(["AU016", "AU017"])
][
    [
        "annotation_unit_id",
        "element_id",
        "element_recovered"
    ]
]

print(pilot.to_string(index=False))

if (
    len(q06) == 140
    and valid.sum() == 140
    and q06["annotation_unit_id"].nunique() == 20
):
    print("\nQ06 STATUS: PASS — FULLY ANNOTATED")
else:
    print("\nQ06 STATUS: REVIEW REQUIRED")

Q06 FINAL VALIDATION
Total Q06 element judgments: 140
Completed judgments: 140
Remaining judgments: 0
Unique annotation units: 20
Yes: 47
No: 93

PILOT UNITS:
annotation_unit_id element_id element_recovered
             AU016        E01               Yes
             AU016        E02                No
             AU016        E03                No
             AU016        E04                No
             AU016        E05                No
             AU016        E06                No
             AU016        E07                No
             AU017        E01                No
             AU017        E02                No
             AU017        E03                No
             AU017        E04                No
             AU017        E05                No
             AU017        E06                No
             AU017        E07                No

Q06 STATUS: PASS — FULLY ANNOTATED


In [34]:
# ============================================================
# CELL 32 — DISPLAY REMAINING Q07 ANNOTATION UNITS
# ============================================================

import pandas as pd

WORKBOOK_PATH = "/content/retrieval_annotation_workbook.xlsx"

element_df = pd.read_excel(
    WORKBOOK_PATH,
    sheet_name="Element Annotation"
)

element_df["element_recovered"] = (
    element_df["element_recovered"]
    .astype("string")
    .str.strip()
)

q07_remaining = element_df[
    (element_df["query_id"] == "Q07")
    & (~element_df["element_recovered"].isin(["Yes", "No"]))
].copy()

print("Q07 — REMAINING HUMAN ANNOTATION")
print("=" * 80)

print(
    "Remaining element judgments:",
    len(q07_remaining)
)

print(
    "Unique annotation units:",
    q07_remaining["annotation_unit_id"].nunique()
)

print()

for annotation_unit_id, group in q07_remaining.groupby(
    "annotation_unit_id",
    sort=False
):

    first = group.iloc[0]

    print("=" * 80)
    print(annotation_unit_id)

    print("\nQUERY:")
    print(first["query_ar"])

    print("\nEXPECTED ELEMENTS:")

    for _, row in group.iterrows():
        print(
            f'{row["element_id"]}: '
            f'{row["expected_element"]}'
        )

    print("\nRETRIEVED TEXT:")
    print(first["retrieval_text"])

    print()

Q07 — REMAINING HUMAN ANNOTATION
Remaining element judgments: 28
Unique annotation units: 14

AU019

QUERY:
ما أوجه الاختلاف بين انقسام الخلية النباتية والخلية الحيوانية كما يعرضها المنهج؟

EXPECTED ELEMENTS:
E01: يظهر في انقسام الخلية النباتية تكون الصفيحة الوسطى.
E02: يظهر في انقسام الخلية الحيوانية تخصر الغشاء.

RETRIEVED TEXT:
عرض معلم العلوم والحياة على طلبته الشكل الآتي، ساعد الطلبة في تحديد أهم نقاط الاختلاف بين انقسام كل من الخلية الحيوانية والخلية النباتية؟
يوضح الجزء الخاص بالخلية النباتية: الدور الانفصالي، الدور النهائي، انقسام السيتوبلازم، تجمعات بروتينية، أكياس غشائية، وتكون الصفيحة الوسطى.
يوضح الجزء الخاص بالخلية الحيوانية: الدور الانفصالي، الدور النهائي، انقسام السيتوبلازم، المريكز، وتخصر الغشاء.

AU020

QUERY:
ما أوجه الاختلاف بين انقسام الخلية النباتية والخلية الحيوانية كما يعرضها المنهج؟

EXPECTED ELEMENTS:
E01: يظهر في انقسام الخلية النباتية تكون الصفيحة الوسطى.
E02: يظهر في انقسام الخلية الحيوانية تخصر الغشاء.

RETRIEVED TEXT:
تتكون أجسام الكائنات الحية عديدة الخلا

In [35]:
# ============================================================
# CELL 33 — WRITE REVIEWED Q07 JUDGMENTS + VALIDATE
# ============================================================

import pandas as pd
from openpyxl import load_workbook

WORKBOOK_PATH = "/content/retrieval_annotation_workbook.xlsx"

q07_judgments = {
    "AU019": {"E01": "Yes", "E02": "Yes"},
    "AU020": {"E01": "No",  "E02": "No"},
    "AU021": {"E01": "No",  "E02": "No"},
    "AU055": {"E01": "Yes", "E02": "Yes"},
    "AU056": {"E01": "No",  "E02": "No"},
    "AU057": {"E01": "No",  "E02": "No"},
    "AU108": {"E01": "No",  "E02": "No"},
    "AU140": {"E01": "No",  "E02": "No"},
    "AU153": {"E01": "No",  "E02": "No"},
    "AU165": {"E01": "No",  "E02": "No"},
    "AU174": {"E01": "No",  "E02": "No"},
    "AU201": {"E01": "No",  "E02": "Yes"},
    "AU206": {"E01": "No",  "E02": "No"},
    "AU215": {"E01": "No",  "E02": "No"},
}

print("WRITE REVIEWED Q07 JUDGMENTS")
print("=" * 75)

wb = load_workbook(WORKBOOK_PATH)
ws = wb["Element Annotation"]

headers = {cell.value: cell.column for cell in ws[1]}

required = [
    "annotation_unit_id",
    "query_id",
    "element_id",
    "element_recovered"
]

missing = [c for c in required if c not in headers]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

updated = 0
not_found = []
conflicts = []

for au, elements in q07_judgments.items():

    for element_id, judgment in elements.items():

        found = False

        for row in range(2, ws.max_row + 1):

            if (
                ws.cell(row, headers["annotation_unit_id"]).value == au
                and ws.cell(row, headers["query_id"]).value == "Q07"
                and ws.cell(row, headers["element_id"]).value == element_id
            ):
                found = True

                target = ws.cell(
                    row,
                    headers["element_recovered"]
                )

                existing = (
                    str(target.value).strip()
                    if target.value is not None
                    else ""
                )

                if (
                    existing in ["Yes", "No"]
                    and existing != judgment
                ):
                    conflicts.append(
                        (au, element_id, existing, judgment)
                    )

                else:
                    target.value = judgment
                    updated += 1

                break

        if not found:
            not_found.append((au, element_id))

if conflicts:
    print("CONFLICTS:", conflicts)
    raise ValueError(
        "Workbook NOT saved because conflicts were detected."
    )

wb.save(WORKBOOK_PATH)

print("Rows written/confirmed:", updated)
print("Judgments not found:", not_found)

# ============================================================
# VALIDATION
# ============================================================

df = pd.read_excel(
    WORKBOOK_PATH,
    sheet_name="Element Annotation"
)

df["element_recovered"] = (
    df["element_recovered"]
    .astype("string")
    .str.strip()
)

q07 = df[df["query_id"] == "Q07"].copy()

complete = q07["element_recovered"].isin(["Yes", "No"])

errors = []

for au, elements in q07_judgments.items():

    for element_id, expected in elements.items():

        match = q07[
            (q07["annotation_unit_id"] == au)
            & (q07["element_id"] == element_id)
        ]

        if len(match) != 1:
            errors.append(
                f"{au}/{element_id}: found {len(match)} rows"
            )
            continue

        actual = match.iloc[0]["element_recovered"]

        if actual != expected:
            errors.append(
                f"{au}/{element_id}: "
                f"actual={actual}, expected={expected}"
            )

print("\nQ07 VALIDATION")
print("-" * 75)

print("Total element judgments:", len(q07))
print("Completed judgments:", int(complete.sum()))
print("Remaining judgments:", int((~complete).sum()))
print(
    "Unique annotation units:",
    q07["annotation_unit_id"].nunique()
)
print(
    "Yes:",
    int((q07["element_recovered"] == "Yes").sum())
)
print(
    "No:",
    int((q07["element_recovered"] == "No").sum())
)
print("Validation errors:", len(errors))

if errors:
    for error in errors:
        print(" -", error)

if (
    len(q07) == 28
    and complete.sum() == 28
    and q07["annotation_unit_id"].nunique() == 14
    and len(errors) == 0
    and len(not_found) == 0
):
    print("\nQ07 STATUS: PASS — FULLY ANNOTATED")
else:
    print("\nQ07 STATUS: REVIEW REQUIRED")

WRITE REVIEWED Q07 JUDGMENTS
Rows written/confirmed: 28
Judgments not found: []

Q07 VALIDATION
---------------------------------------------------------------------------
Total element judgments: 28
Completed judgments: 28
Remaining judgments: 0
Unique annotation units: 14
Yes: 5
No: 23
Validation errors: 0

Q07 STATUS: PASS — FULLY ANNOTATED


In [36]:
# ============================================================
# CELL 34 — DISPLAY REMAINING Q08 ANNOTATION UNITS
# ============================================================

import pandas as pd

WORKBOOK_PATH = "/content/retrieval_annotation_workbook.xlsx"

element_df = pd.read_excel(
    WORKBOOK_PATH,
    sheet_name="Element Annotation"
)

element_df["element_recovered"] = (
    element_df["element_recovered"]
    .astype("string")
    .str.strip()
)

q08_remaining = element_df[
    (element_df["query_id"] == "Q08")
    & (~element_df["element_recovered"].isin(["Yes", "No"]))
].copy()

print("Q08 — REMAINING HUMAN ANNOTATION")
print("=" * 80)

print(
    "Remaining element judgments:",
    len(q08_remaining)
)

print(
    "Unique annotation units:",
    q08_remaining["annotation_unit_id"].nunique()
)

print()

for annotation_unit_id, group in q08_remaining.groupby(
    "annotation_unit_id",
    sort=False
):

    first = group.iloc[0]

    print("=" * 80)
    print(annotation_unit_id)

    print("\nQUERY:")
    print(first["query_ar"])

    print("\nEXPECTED ELEMENTS:")

    for _, row in group.iterrows():
        print(
            f'{row["element_id"]}: '
            f'{row["expected_element"]}'
        )

    print("\nRETRIEVED TEXT:")
    print(first["retrieval_text"])

    print()

Q08 — REMAINING HUMAN ANNOTATION
Remaining element judgments: 38
Unique annotation units: 19

AU022

QUERY:
ما نتيجة الانقسام المنصف من حيث عدد الخلايا وعدد الكروموسومات؟

EXPECTED ELEMENTS:
E01: ينتج أربع خلايا.
E02: تحتوي كل خلية ناتجة على نصف العدد الأصلي من الكروموسومات في الخلية الأم.

RETRIEVED TEXT:
الدور النهائي: تتجمع الكروموسومات الوليدة في منتصف كل خلية جديدة، ويبدأ الغلاف النووي بالتكون والسيتوبلازم بالانقسام.
انقسام السيتوبلازم: ينقسم السيتوبلازم وتنتج خليتان متماثلتان تحوي كل منهما على نفس عدد كروموسومات الخلية الأم.

AU023

QUERY:
ما نتيجة الانقسام المنصف من حيث عدد الخلايا وعدد الكروموسومات؟

EXPECTED ELEMENTS:
E01: ينتج أربع خلايا.
E02: تحتوي كل خلية ناتجة على نصف العدد الأصلي من الكروموسومات في الخلية الأم.

RETRIEVED TEXT:
ماذا يحدث إذا لم تمر الخلية في الطور البيني؟
من خلال الشكل أعلاه، حدد الأدوار التي يمر بها الانقسام المتساوي؟
ما أهمية المريكزات، وخيوط المغزل في عملية الانقسام المتساوي؟
لماذا تترتب الكروموسومات فرادى في منتصف الخلية في الدور الاستوائي؟
هل الخلية 

In [37]:
# ============================================================
# CELL 35 — WRITE REVIEWED Q08 JUDGMENTS + VALIDATE
# ============================================================

import pandas as pd
from openpyxl import load_workbook

WORKBOOK_PATH = "/content/retrieval_annotation_workbook.xlsx"

q08_judgments = {
    "AU022": {"E01": "No",  "E02": "No"},
    "AU023": {"E01": "No",  "E02": "No"},
    "AU024": {"E01": "No",  "E02": "No"},
    "AU058": {"E01": "No",  "E02": "No"},
    "AU059": {"E01": "No",  "E02": "No"},
    "AU060": {"E01": "Yes", "E02": "Yes"},
    "AU085": {"E01": "Yes", "E02": "Yes"},
    "AU086": {"E01": "No",  "E02": "No"},
    "AU109": {"E01": "No",  "E02": "No"},
    "AU110": {"E01": "No",  "E02": "No"},
    "AU128": {"E01": "No",  "E02": "No"},
    "AU141": {"E01": "No",  "E02": "No"},
    "AU154": {"E01": "No",  "E02": "No"},
    "AU166": {"E01": "Yes", "E02": "Yes"},
    "AU167": {"E01": "No",  "E02": "No"},
    "AU175": {"E01": "No",  "E02": "No"},
    "AU186": {"E01": "No",  "E02": "No"},
    "AU212": {"E01": "No",  "E02": "No"},
    "AU217": {"E01": "Yes", "E02": "Yes"},
}

print("WRITE REVIEWED Q08 JUDGMENTS")
print("=" * 75)

wb = load_workbook(WORKBOOK_PATH)
ws = wb["Element Annotation"]

headers = {cell.value: cell.column for cell in ws[1]}

required = [
    "annotation_unit_id",
    "query_id",
    "element_id",
    "element_recovered"
]

missing = [c for c in required if c not in headers]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

updated = 0
not_found = []
conflicts = []

for au, elements in q08_judgments.items():
    for element_id, judgment in elements.items():

        found = False

        for row in range(2, ws.max_row + 1):

            if (
                ws.cell(row, headers["annotation_unit_id"]).value == au
                and ws.cell(row, headers["query_id"]).value == "Q08"
                and ws.cell(row, headers["element_id"]).value == element_id
            ):
                found = True

                target = ws.cell(
                    row,
                    headers["element_recovered"]
                )

                existing = (
                    str(target.value).strip()
                    if target.value is not None
                    else ""
                )

                if (
                    existing in ["Yes", "No"]
                    and existing != judgment
                ):
                    conflicts.append(
                        (au, element_id, existing, judgment)
                    )
                else:
                    target.value = judgment
                    updated += 1

                break

        if not found:
            not_found.append((au, element_id))

if conflicts:
    print("CONFLICTS:", conflicts)
    raise ValueError(
        "Workbook NOT saved because conflicts were detected."
    )

wb.save(WORKBOOK_PATH)

print("Rows written/confirmed:", updated)
print("Judgments not found:", not_found)

# ============================================================
# VALIDATION
# ============================================================

df = pd.read_excel(
    WORKBOOK_PATH,
    sheet_name="Element Annotation"
)

df["element_recovered"] = (
    df["element_recovered"]
    .astype("string")
    .str.strip()
)

q08 = df[df["query_id"] == "Q08"].copy()

complete = q08["element_recovered"].isin(["Yes", "No"])

errors = []

for au, elements in q08_judgments.items():
    for element_id, expected in elements.items():

        match = q08[
            (q08["annotation_unit_id"] == au)
            & (q08["element_id"] == element_id)
        ]

        if len(match) != 1:
            errors.append(
                f"{au}/{element_id}: found {len(match)} rows"
            )
            continue

        actual = match.iloc[0]["element_recovered"]

        if actual != expected:
            errors.append(
                f"{au}/{element_id}: "
                f"actual={actual}, expected={expected}"
            )

print("\nQ08 VALIDATION")
print("-" * 75)

print("Total element judgments:", len(q08))
print("Completed judgments:", int(complete.sum()))
print("Remaining judgments:", int((~complete).sum()))
print(
    "Unique annotation units:",
    q08["annotation_unit_id"].nunique()
)
print(
    "Yes:",
    int((q08["element_recovered"] == "Yes").sum())
)
print(
    "No:",
    int((q08["element_recovered"] == "No").sum())
)
print("Validation errors:", len(errors))

if errors:
    for error in errors:
        print(" -", error)

if (
    len(q08) == 38
    and complete.sum() == 38
    and q08["annotation_unit_id"].nunique() == 19
    and len(errors) == 0
    and len(not_found) == 0
):
    print("\nQ08 STATUS: PASS — FULLY ANNOTATED")
else:
    print("\nQ08 STATUS: REVIEW REQUIRED")

WRITE REVIEWED Q08 JUDGMENTS
Rows written/confirmed: 38
Judgments not found: []

Q08 VALIDATION
---------------------------------------------------------------------------
Total element judgments: 38
Completed judgments: 38
Remaining judgments: 0
Unique annotation units: 19
Yes: 8
No: 30
Validation errors: 0

Q08 STATUS: PASS — FULLY ANNOTATED


In [38]:
# ============================================================
# CELL 36 — DISPLAY REMAINING Q09 ANNOTATION UNITS
# ============================================================

import pandas as pd

WORKBOOK_PATH = "/content/retrieval_annotation_workbook.xlsx"

element_df = pd.read_excel(
    WORKBOOK_PATH,
    sheet_name="Element Annotation"
)

element_df["element_recovered"] = (
    element_df["element_recovered"]
    .astype("string")
    .str.strip()
)

q09_remaining = element_df[
    (element_df["query_id"] == "Q09")
    & (~element_df["element_recovered"].isin(["Yes", "No"]))
].copy()

print("Q09 — REMAINING HUMAN ANNOTATION")
print("=" * 80)

print(
    "Remaining element judgments:",
    len(q09_remaining)
)

print(
    "Unique annotation units:",
    q09_remaining["annotation_unit_id"].nunique()
)

print()

for annotation_unit_id, group in q09_remaining.groupby(
    "annotation_unit_id",
    sort=False
):

    first = group.iloc[0]

    print("=" * 80)
    print(annotation_unit_id)

    print("\nQUERY:")
    print(first["query_ar"])

    print("\nEXPECTED ELEMENTS:")

    for _, row in group.iterrows():
        print(
            f'{row["element_id"]}: '
            f'{row["expected_element"]}'
        )

    print("\nRETRIEVED TEXT:")
    print(first["retrieval_text"])

    print()

Q09 — REMAINING HUMAN ANNOTATION
Remaining element judgments: 45
Unique annotation units: 15

AU025

QUERY:
ما العلاقة بين الخلايا الجسمية والخلايا التناسلية ونوع الانقسام الذي يحدث في كل منها؟

EXPECTED ELEMENTS:
E01: الخلايا الجسمية تنقسم بطريقة الانقسام المتساوي.
E02: الخلايا التناسلية تنقسم بطريقة الانقسام المنصف.
E03: الانقسام المنصف في الخلايا التناسلية ينتج الغاميتات.

RETRIEVED TEXT:
تتكون أجسام الكائنات الحية عديدة الخلايا التي تتكاثر جنسياً من نوعين رئيسين من الخلايا، هما:
الخلايا الجسمية: تشمل خلايا جسم الكائنات الحية الراقية، وتنقسم النواة بطريقة الانقسام المتساوي، وينتج عنه خليتان (ابنتان)، تحتوي كلاهما على العدد نفسه من الكروموسومات.
الخلايا التناسلية (الجنسية المنتجة للغاميتات): توجد في الأعضاء الجنسية الذكرية والأنثوية لدى معظم الحيوانات والنباتات، وتنقسم النواة بطريقة الانقسام المنصف منتجة الغاميتات.

AU026

QUERY:
ما العلاقة بين الخلايا الجسمية والخلايا التناسلية ونوع الانقسام الذي يحدث في كل منها؟

EXPECTED ELEMENTS:
E01: الخلايا الجسمية تنقسم بطريقة الانقسام المتساو

In [39]:
# ============================================================
# CELL 37 — WRITE REVIEWED Q09 JUDGMENTS + VALIDATE
# ============================================================

import pandas as pd
from openpyxl import load_workbook

WORKBOOK_PATH = "/content/retrieval_annotation_workbook.xlsx"

q09_judgments = {
    "AU025": {"E01":"Yes", "E02":"Yes", "E03":"Yes"},
    "AU026": {"E01":"No",  "E02":"No",  "E03":"No"},
    "AU027": {"E01":"No",  "E02":"No",  "E03":"No"},
    "AU061": {"E01":"Yes", "E02":"Yes", "E03":"Yes"},
    "AU062": {"E01":"No",  "E02":"Yes", "E03":"Yes"},
    "AU063": {"E01":"No",  "E02":"No",  "E03":"No"},
    "AU087": {"E01":"No",  "E02":"No",  "E03":"No"},
    "AU111": {"E01":"No",  "E02":"Yes", "E03":"Yes"},
    "AU112": {"E01":"No",  "E02":"No",  "E03":"No"},
    "AU129": {"E01":"No",  "E02":"No",  "E03":"No"},
    "AU142": {"E01":"No",  "E02":"Yes", "E03":"Yes"},
    "AU155": {"E01":"No",  "E02":"Yes", "E03":"Yes"},
    "AU168": {"E01":"No",  "E02":"Yes", "E03":"Yes"},
    "AU176": {"E01":"No",  "E02":"No",  "E03":"No"},
    "AU213": {"E01":"No",  "E02":"No",  "E03":"No"},
}

print("WRITE REVIEWED Q09 JUDGMENTS")
print("=" * 75)

wb = load_workbook(WORKBOOK_PATH)
ws = wb["Element Annotation"]

headers = {cell.value: cell.column for cell in ws[1]}

required = [
    "annotation_unit_id",
    "query_id",
    "element_id",
    "element_recovered"
]

missing = [c for c in required if c not in headers]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

updated = 0
not_found = []
conflicts = []

for au, elements in q09_judgments.items():
    for element_id, judgment in elements.items():

        found = False

        for row in range(2, ws.max_row + 1):

            if (
                ws.cell(row, headers["annotation_unit_id"]).value == au
                and ws.cell(row, headers["query_id"]).value == "Q09"
                and ws.cell(row, headers["element_id"]).value == element_id
            ):
                found = True

                target = ws.cell(
                    row,
                    headers["element_recovered"]
                )

                existing = (
                    str(target.value).strip()
                    if target.value is not None
                    else ""
                )

                if (
                    existing in ["Yes", "No"]
                    and existing != judgment
                ):
                    conflicts.append(
                        (au, element_id, existing, judgment)
                    )
                else:
                    target.value = judgment
                    updated += 1

                break

        if not found:
            not_found.append((au, element_id))

if conflicts:
    print("CONFLICTS:", conflicts)
    raise ValueError(
        "Workbook NOT saved because conflicts were detected."
    )

wb.save(WORKBOOK_PATH)

print("Rows written/confirmed:", updated)
print("Judgments not found:", not_found)

# ============================================================
# VALIDATION
# ============================================================

df = pd.read_excel(
    WORKBOOK_PATH,
    sheet_name="Element Annotation"
)

df["element_recovered"] = (
    df["element_recovered"]
    .astype("string")
    .str.strip()
)

q09 = df[df["query_id"] == "Q09"].copy()

complete = q09["element_recovered"].isin(["Yes", "No"])

errors = []

for au, elements in q09_judgments.items():
    for element_id, expected in elements.items():

        match = q09[
            (q09["annotation_unit_id"] == au)
            & (q09["element_id"] == element_id)
        ]

        if len(match) != 1:
            errors.append(
                f"{au}/{element_id}: found {len(match)} rows"
            )
            continue

        actual = match.iloc[0]["element_recovered"]

        if actual != expected:
            errors.append(
                f"{au}/{element_id}: "
                f"actual={actual}, expected={expected}"
            )

print("\nQ09 VALIDATION")
print("-" * 75)

print("Total element judgments:", len(q09))
print("Completed judgments:", int(complete.sum()))
print("Remaining judgments:", int((~complete).sum()))
print(
    "Unique annotation units:",
    q09["annotation_unit_id"].nunique()
)
print(
    "Yes:",
    int((q09["element_recovered"] == "Yes").sum())
)
print(
    "No:",
    int((q09["element_recovered"] == "No").sum())
)
print("Validation errors:", len(errors))

if errors:
    for error in errors:
        print(" -", error)

if (
    len(q09) == 45
    and complete.sum() == 45
    and q09["annotation_unit_id"].nunique() == 15
    and len(errors) == 0
    and len(not_found) == 0
):
    print("\nQ09 STATUS: PASS — FULLY ANNOTATED")
else:
    print("\nQ09 STATUS: REVIEW REQUIRED")

WRITE REVIEWED Q09 JUDGMENTS
Rows written/confirmed: 45
Judgments not found: []

Q09 VALIDATION
---------------------------------------------------------------------------
Total element judgments: 45
Completed judgments: 45
Remaining judgments: 0
Unique annotation units: 15
Yes: 16
No: 29
Validation errors: 0

Q09 STATUS: PASS — FULLY ANNOTATED


In [40]:
# ============================================================
# CELL 38 — DISPLAY REMAINING Q10 ANNOTATION UNITS
# ============================================================

import pandas as pd

WORKBOOK_PATH = "/content/retrieval_annotation_workbook.xlsx"

element_df = pd.read_excel(
    WORKBOOK_PATH,
    sheet_name="Element Annotation"
)

element_df["element_recovered"] = (
    element_df["element_recovered"]
    .astype("string")
    .str.strip()
)

q10_remaining = element_df[
    (element_df["query_id"] == "Q10")
    & (~element_df["element_recovered"].isin(["Yes", "No"]))
].copy()

print("Q10 — REMAINING HUMAN ANNOTATION")
print("=" * 80)

print(
    "Remaining element judgments:",
    len(q10_remaining)
)

print(
    "Unique annotation units:",
    q10_remaining["annotation_unit_id"].nunique()
)

print()

for annotation_unit_id, group in q10_remaining.groupby(
    "annotation_unit_id",
    sort=False
):

    first = group.iloc[0]

    print("=" * 80)
    print(annotation_unit_id)

    print("\nQUERY:")
    print(first["query_ar"])

    print("\nEXPECTED ELEMENTS:")

    for _, row in group.iterrows():
        print(
            f'{row["element_id"]}: '
            f'{row["expected_element"]}'
        )

    print("\nRETRIEVED TEXT:")
    print(first["retrieval_text"])

    print()

Q10 — REMAINING HUMAN ANNOTATION
Remaining element judgments: 80
Unique annotation units: 20

AU030

QUERY:
كيف يمكن أن يؤدي خلل أثناء الانقسام المنصف إلى وجود 47 كروموسومًا؟

EXPECTED ELEMENTS:
E01: يحدث خلل أثناء الانقسام المنصف في الخلايا الجنسية.
E02: قد ينتج غاميت يحتوي على 24 كروموسومًا.
E03: يتحد مع غاميت طبيعي يحتوي على 23 كروموسومًا.
E04: ينتج عن ذلك 47 كروموسومًا.

RETRIEVED TEXT:
ادرس مراحل الانقسام المتساوي للخلية في الشكل أدناه، ثم أجب عن الأسئلة التي تليه.
الدور التمهيدي: تظهر كروموسومات قصيرة وغليظة متضاعفة، يختفي الغلاف النووي، وتختفي النوية.

AU064

QUERY:
كيف يمكن أن يؤدي خلل أثناء الانقسام المنصف إلى وجود 47 كروموسومًا؟

EXPECTED ELEMENTS:
E01: يحدث خلل أثناء الانقسام المنصف في الخلايا الجنسية.
E02: قد ينتج غاميت يحتوي على 24 كروموسومًا.
E03: يتحد مع غاميت طبيعي يحتوي على 23 كروموسومًا.
E04: ينتج عن ذلك 47 كروموسومًا.

RETRIEVED TEXT:
لاحظ خريطة الكروموسومات لشخص مصاب بمتلازمة داون. استنتج سبب الاختلاف، ومكانه.
تظهر خريطة الكروموسومات وجود كروموسوم ثالث ضمن الزوج الك

In [41]:
# ============================================================
# CELL 39 — WRITE REMAINING Q10 JUDGMENTS + VALIDATE FULL Q10
# ============================================================

import pandas as pd
from openpyxl import load_workbook

WORKBOOK_PATH = "/content/retrieval_annotation_workbook.xlsx"

q10_judgments = {
    "AU030": {"E01":"No","E02":"No","E03":"No","E04":"No"},
    "AU064": {"E01":"Yes","E02":"Yes","E03":"Yes","E04":"Yes"},
    "AU065": {"E01":"No","E02":"No","E03":"No","E04":"No"},
    "AU066": {"E01":"No","E02":"No","E03":"No","E04":"No"},
    "AU088": {"E01":"No","E02":"No","E03":"No","E04":"No"},
    "AU089": {"E01":"No","E02":"No","E03":"No","E04":"No"},
    "AU090": {"E01":"Yes","E02":"Yes","E03":"Yes","E04":"Yes"},
    "AU113": {"E01":"No","E02":"No","E03":"No","E04":"No"},
    "AU114": {"E01":"No","E02":"No","E03":"No","E04":"No"},
    "AU115": {"E01":"No","E02":"No","E03":"No","E04":"No"},
    "AU130": {"E01":"No","E02":"No","E03":"No","E04":"No"},
    "AU131": {"E01":"No","E02":"No","E03":"No","E04":"No"},
    "AU143": {"E01":"No","E02":"No","E03":"No","E04":"No"},
    "AU177": {"E01":"No","E02":"No","E03":"No","E04":"No"},
    "AU187": {"E01":"No","E02":"No","E03":"No","E04":"No"},
    "AU193": {"E01":"No","E02":"No","E03":"No","E04":"No"},
    "AU202": {"E01":"Yes","E02":"Yes","E03":"Yes","E04":"Yes"},
    "AU207": {"E01":"No","E02":"No","E03":"No","E04":"No"},
    "AU209": {"E01":"No","E02":"No","E03":"No","E04":"No"},
    "AU216": {"E01":"Yes","E02":"Yes","E03":"Yes","E04":"Yes"},
}

print("WRITE REVIEWED Q10 REMAINING JUDGMENTS")
print("=" * 75)

wb = load_workbook(WORKBOOK_PATH)
ws = wb["Element Annotation"]

headers = {cell.value: cell.column for cell in ws[1]}

updated = 0
not_found = []
conflicts = []

for au, elements in q10_judgments.items():
    for element_id, judgment in elements.items():

        found = False

        for row in range(2, ws.max_row + 1):
            if (
                ws.cell(row, headers["annotation_unit_id"]).value == au
                and ws.cell(row, headers["query_id"]).value == "Q10"
                and ws.cell(row, headers["element_id"]).value == element_id
            ):
                found = True

                target = ws.cell(
                    row,
                    headers["element_recovered"]
                )

                existing = (
                    str(target.value).strip()
                    if target.value is not None
                    else ""
                )

                if (
                    existing in ["Yes", "No"]
                    and existing != judgment
                ):
                    conflicts.append(
                        (au, element_id, existing, judgment)
                    )
                else:
                    target.value = judgment
                    updated += 1

                break

        if not found:
            not_found.append((au, element_id))

if conflicts:
    print("CONFLICTS:", conflicts)
    raise ValueError(
        "Workbook NOT saved because conflicts were detected."
    )

wb.save(WORKBOOK_PATH)

print("Rows written/confirmed:", updated)
print("Judgments not found:", not_found)

# ============================================================
# FULL Q10 VALIDATION
# ============================================================

df = pd.read_excel(
    WORKBOOK_PATH,
    sheet_name="Element Annotation"
)

df["element_recovered"] = (
    df["element_recovered"]
    .astype("string")
    .str.strip()
)

q10 = df[df["query_id"] == "Q10"].copy()

complete = q10["element_recovered"].isin(["Yes", "No"])

print("\nQ10 FULL VALIDATION")
print("-" * 75)

print("Total element judgments:", len(q10))
print("Completed judgments:", int(complete.sum()))
print("Remaining judgments:", int((~complete).sum()))
print(
    "Unique annotation units:",
    q10["annotation_unit_id"].nunique()
)
print(
    "Yes:",
    int((q10["element_recovered"] == "Yes").sum())
)
print(
    "No:",
    int((q10["element_recovered"] == "No").sum())
)

# Verify only the newly reviewed units
errors = []

for au, elements in q10_judgments.items():
    for element_id, expected in elements.items():

        match = q10[
            (q10["annotation_unit_id"] == au)
            & (q10["element_id"] == element_id)
        ]

        if len(match) != 1:
            errors.append(
                f"{au}/{element_id}: found {len(match)} rows"
            )
            continue

        actual = match.iloc[0]["element_recovered"]

        if actual != expected:
            errors.append(
                f"{au}/{element_id}: actual={actual}, expected={expected}"
            )

print("Validation errors:", len(errors))

if errors:
    for error in errors:
        print(" -", error)

if (
    complete.sum() == len(q10)
    and len(errors) == 0
    and len(not_found) == 0
):
    print("\nQ10 STATUS: PASS — FULLY ANNOTATED")
else:
    print("\nQ10 STATUS: REVIEW REQUIRED")

WRITE REVIEWED Q10 REMAINING JUDGMENTS
Rows written/confirmed: 80
Judgments not found: []

Q10 FULL VALIDATION
---------------------------------------------------------------------------
Total element judgments: 88
Completed judgments: 88
Remaining judgments: 0
Unique annotation units: 22
Yes: 20
No: 68
Validation errors: 0

Q10 STATUS: PASS — FULLY ANNOTATED


In [42]:
# ============================================================
# CELL 40 — DISPLAY REMAINING Q11 ANNOTATION UNITS
# ============================================================

import pandas as pd

WORKBOOK_PATH = "/content/retrieval_annotation_workbook.xlsx"

element_df = pd.read_excel(
    WORKBOOK_PATH,
    sheet_name="Element Annotation"
)

element_df["element_recovered"] = (
    element_df["element_recovered"]
    .astype("string")
    .str.strip()
)

q11_remaining = element_df[
    (element_df["query_id"] == "Q11")
    & (~element_df["element_recovered"].isin(["Yes", "No"]))
].copy()

print("Q11 — REMAINING HUMAN ANNOTATION")
print("=" * 80)

print(
    "Remaining element judgments:",
    len(q11_remaining)
)

print(
    "Unique annotation units:",
    q11_remaining["annotation_unit_id"].nunique()
)

print()

for annotation_unit_id, group in q11_remaining.groupby(
    "annotation_unit_id",
    sort=False
):

    first = group.iloc[0]

    print("=" * 80)
    print(annotation_unit_id)

    print("\nQUERY:")
    print(first["query_ar"])

    print("\nEXPECTED ELEMENTS:")

    for _, row in group.iterrows():
        print(
            f'{row["element_id"]}: '
            f'{row["expected_element"]}'
        )

    print("\nRETRIEVED TEXT:")
    print(first["retrieval_text"])

    print()

Q11 — REMAINING HUMAN ANNOTATION
Remaining element judgments: 68
Unique annotation units: 17

AU033

QUERY:
ما الفرق بين نواتج الانقسام المتساوي والانقسام المنصف من حيث عدد الخلايا وعدد الكروموسومات؟

EXPECTED ELEMENTS:
E01: الانقسام المتساوي ينتج خليتين.
E02: الخليتان الناتجتان عن الانقسام المتساوي تحتويان على نفس عدد كروموسومات الخلية الأم.
E03: الانقسام المنصف ينتج أربع خلايا.
E04: كل خلية ناتجة عن الانقسام المنصف تحتوي على نصف العدد الأصلي من كروموسومات الخلية الأم.

RETRIEVED TEXT:
ادرس مراحل الانقسام المتساوي للخلية في الشكل أدناه، ثم أجب عن الأسئلة التي تليه.
الدور التمهيدي: تظهر كروموسومات قصيرة وغليظة متضاعفة، يختفي الغلاف النووي، وتختفي النوية.

AU067

QUERY:
ما الفرق بين نواتج الانقسام المتساوي والانقسام المنصف من حيث عدد الخلايا وعدد الكروموسومات؟

EXPECTED ELEMENTS:
E01: الانقسام المتساوي ينتج خليتين.
E02: الخليتان الناتجتان عن الانقسام المتساوي تحتويان على نفس عدد كروموسومات الخلية الأم.
E03: الانقسام المنصف ينتج أربع خلايا.
E04: كل خلية ناتجة عن الانقسام المنصف تحتوي على

In [43]:
# ============================================================
# CELL 41 — WRITE REMAINING Q11 JUDGMENTS + VALIDATE FULL Q11
# ============================================================

import pandas as pd
from openpyxl import load_workbook

WORKBOOK_PATH = "/content/retrieval_annotation_workbook.xlsx"

q11_judgments = {
    "AU033": {"E01":"No","E02":"No","E03":"No","E04":"No"},
    "AU067": {"E01":"Yes","E02":"Yes","E03":"No","E04":"No"},
    "AU068": {"E01":"No","E02":"No","E03":"No","E04":"No"},
    "AU069": {"E01":"Yes","E02":"Yes","E03":"No","E04":"No"},
    "AU091": {"E01":"No","E02":"No","E03":"Yes","E04":"Yes"},
    "AU092": {"E01":"Yes","E02":"Yes","E03":"No","E04":"No"},
    "AU116": {"E01":"Yes","E02":"Yes","E03":"No","E04":"No"},
    "AU117": {"E01":"Yes","E02":"Yes","E03":"No","E04":"No"},
    "AU132": {"E01":"Yes","E02":"Yes","E03":"No","E04":"No"},
    "AU144": {"E01":"No","E02":"No","E03":"Yes","E04":"Yes"},
    "AU156": {"E01":"No","E02":"No","E03":"No","E04":"No"},
    "AU178": {"E01":"No","E02":"No","E03":"No","E04":"No"},
    "AU188": {"E01":"No","E02":"No","E03":"No","E04":"No"},
    "AU210": {"E01":"No","E02":"No","E03":"Yes","E04":"Yes"},
    "AU214": {"E01":"No","E02":"No","E03":"No","E04":"No"},
    "AU218": {"E01":"Yes","E02":"Yes","E03":"No","E04":"No"},
    "AU219": {"E01":"No","E02":"No","E03":"Yes","E04":"Yes"},
}

print("WRITE REVIEWED Q11 REMAINING JUDGMENTS")
print("=" * 75)

wb = load_workbook(WORKBOOK_PATH)
ws = wb["Element Annotation"]

headers = {cell.value: cell.column for cell in ws[1]}

updated = 0
not_found = []
conflicts = []

for au, elements in q11_judgments.items():
    for element_id, judgment in elements.items():

        found = False

        for row in range(2, ws.max_row + 1):
            if (
                ws.cell(row, headers["annotation_unit_id"]).value == au
                and ws.cell(row, headers["query_id"]).value == "Q11"
                and ws.cell(row, headers["element_id"]).value == element_id
            ):
                found = True

                target = ws.cell(
                    row,
                    headers["element_recovered"]
                )

                existing = (
                    str(target.value).strip()
                    if target.value is not None
                    else ""
                )

                if (
                    existing in ["Yes", "No"]
                    and existing != judgment
                ):
                    conflicts.append(
                        (au, element_id, existing, judgment)
                    )
                else:
                    target.value = judgment
                    updated += 1

                break

        if not found:
            not_found.append((au, element_id))

if conflicts:
    print("CONFLICTS:", conflicts)
    raise ValueError(
        "Workbook NOT saved because conflicts were detected."
    )

wb.save(WORKBOOK_PATH)

print("Rows written/confirmed:", updated)
print("Judgments not found:", not_found)

# ============================================================
# FULL Q11 VALIDATION
# ============================================================

df = pd.read_excel(
    WORKBOOK_PATH,
    sheet_name="Element Annotation"
)

df["element_recovered"] = (
    df["element_recovered"]
    .astype("string")
    .str.strip()
)

q11 = df[df["query_id"] == "Q11"].copy()

complete = q11["element_recovered"].isin(["Yes", "No"])

errors = []

for au, elements in q11_judgments.items():
    for element_id, expected in elements.items():

        match = q11[
            (q11["annotation_unit_id"] == au)
            & (q11["element_id"] == element_id)
        ]

        if len(match) != 1:
            errors.append(
                f"{au}/{element_id}: found {len(match)} rows"
            )
            continue

        actual = match.iloc[0]["element_recovered"]

        if actual != expected:
            errors.append(
                f"{au}/{element_id}: actual={actual}, expected={expected}"
            )

print("\nQ11 FULL VALIDATION")
print("-" * 75)

print("Total element judgments:", len(q11))
print("Completed judgments:", int(complete.sum()))
print("Remaining judgments:", int((~complete).sum()))
print(
    "Unique annotation units:",
    q11["annotation_unit_id"].nunique()
)
print(
    "Yes:",
    int((q11["element_recovered"] == "Yes").sum())
)
print(
    "No:",
    int((q11["element_recovered"] == "No").sum())
)
print("Validation errors:", len(errors))

if errors:
    for error in errors:
        print(" -", error)

if (
    complete.sum() == len(q11)
    and len(errors) == 0
    and len(not_found) == 0
):
    print("\nQ11 STATUS: PASS — FULLY ANNOTATED")
else:
    print("\nQ11 STATUS: REVIEW REQUIRED")

WRITE REVIEWED Q11 REMAINING JUDGMENTS
Rows written/confirmed: 68
Judgments not found: []

Q11 FULL VALIDATION
---------------------------------------------------------------------------
Total element judgments: 76
Completed judgments: 76
Remaining judgments: 0
Unique annotation units: 19
Yes: 24
No: 52
Validation errors: 0

Q11 STATUS: PASS — FULLY ANNOTATED


In [44]:
# ============================================================
# CELL 42 — DISPLAY REMAINING Q12 ANNOTATION UNITS
# ============================================================

import pandas as pd

WORKBOOK_PATH = "/content/retrieval_annotation_workbook.xlsx"

element_df = pd.read_excel(
    WORKBOOK_PATH,
    sheet_name="Element Annotation"
)

element_df["element_recovered"] = (
    element_df["element_recovered"]
    .astype("string")
    .str.strip()
)

q12_remaining = element_df[
    (element_df["query_id"] == "Q12")
    & (~element_df["element_recovered"].isin(["Yes", "No"]))
].copy()

print("Q12 — REMAINING HUMAN ANNOTATION")
print("=" * 80)

print(
    "Remaining element judgments:",
    len(q12_remaining)
)

print(
    "Unique annotation units:",
    q12_remaining["annotation_unit_id"].nunique()
)

print()

for annotation_unit_id, group in q12_remaining.groupby(
    "annotation_unit_id",
    sort=False
):

    first = group.iloc[0]

    print("=" * 80)
    print(annotation_unit_id)

    print("\nQUERY:")
    print(first["query_ar"])

    print("\nEXPECTED ELEMENTS:")

    for _, row in group.iterrows():
        print(
            f'{row["element_id"]}: '
            f'{row["expected_element"]}'
        )

    print("\nRETRIEVED TEXT:")
    print(first["retrieval_text"])

    print()

Q12 — REMAINING HUMAN ANNOTATION
Remaining element judgments: 28
Unique annotation units: 14

AU034

QUERY:
ما عدد الكروموسومات في خلايا الإنسان العادي وما العدد الموجود في متلازمة داون؟

EXPECTED ELEMENTS:
E01: خلايا الإنسان العادي تحتوي على 46 كروموسومًا.
E02: خلايا المصاب بمتلازمة داون تحتوي على 47 كروموسومًا.

RETRIEVED TEXT:
إن عدد الكروموسومات في أنوية خلايا الكائن الحي ثابت، وأي تغير في عددها، نقص أو زيادة، يحدث تغيراً في صفات الكائن الحي وهو ما يسمى بالطفرة ففي كل خلية من
خلايا الإنسان العادي (46) كروموسوماً، ولكن يولد بعض الأطفال وعندهم زيادة كروموسوم فتحتوي كل خلية من خلايا الجسم على (47) كروموسوماً، ويطلق عليهم متلازمة داون.
تظهر على المصاب بمتلازمة داون بعض الصفات المظهرية الآتية: وجود ثنية إضافية في جفن العين، قصر القامة، رؤوسهم منبسطة، جبهتهم بارزة، لسانهم كبير الحجم، ذو أيد وأقدام قصيرة، يعاني معظمهم من صعوبات في التعلم.
هل التقيت شخصاً مصاباً بمتلازمة داون؟
كيف يمكنك مساعدتهم في الاندماج في المجتمع؟
نشاط (7) التعلم بالحياة
تحدث عن بعض حالات متلازمة داون، التي لها دور إي

In [45]:
# ============================================================
# CELL 43 — WRITE REVIEWED Q12 JUDGMENTS + VALIDATE
# ============================================================

import pandas as pd
from openpyxl import load_workbook

WORKBOOK_PATH = "/content/retrieval_annotation_workbook.xlsx"

q12_judgments = {
    "AU034": {"E01":"Yes", "E02":"Yes"},
    "AU035": {"E01":"Yes", "E02":"No"},
    "AU036": {"E01":"No",  "E02":"Yes"},
    "AU070": {"E01":"Yes", "E02":"No"},
    "AU071": {"E01":"No",  "E02":"Yes"},
    "AU072": {"E01":"No",  "E02":"No"},
    "AU093": {"E01":"Yes", "E02":"Yes"},
    "AU094": {"E01":"Yes", "E02":"No"},
    "AU133": {"E01":"No",  "E02":"No"},
    "AU145": {"E01":"Yes", "E02":"Yes"},
    "AU169": {"E01":"Yes", "E02":"Yes"},
    "AU189": {"E01":"Yes", "E02":"Yes"},
    "AU194": {"E01":"Yes", "E02":"Yes"},
    "AU220": {"E01":"No",  "E02":"No"},
}

print("WRITE REVIEWED Q12 JUDGMENTS")
print("=" * 75)

wb = load_workbook(WORKBOOK_PATH)
ws = wb["Element Annotation"]

headers = {cell.value: cell.column for cell in ws[1]}

updated = 0
not_found = []
conflicts = []

for au, elements in q12_judgments.items():
    for element_id, judgment in elements.items():

        found = False

        for row in range(2, ws.max_row + 1):

            if (
                ws.cell(row, headers["annotation_unit_id"]).value == au
                and ws.cell(row, headers["query_id"]).value == "Q12"
                and ws.cell(row, headers["element_id"]).value == element_id
            ):
                found = True

                target = ws.cell(
                    row,
                    headers["element_recovered"]
                )

                existing = (
                    str(target.value).strip()
                    if target.value is not None
                    else ""
                )

                if (
                    existing in ["Yes", "No"]
                    and existing != judgment
                ):
                    conflicts.append(
                        (au, element_id, existing, judgment)
                    )
                else:
                    target.value = judgment
                    updated += 1

                break

        if not found:
            not_found.append((au, element_id))

if conflicts:
    print("CONFLICTS:", conflicts)
    raise ValueError(
        "Workbook NOT saved because conflicts were detected."
    )

wb.save(WORKBOOK_PATH)

print("Rows written/confirmed:", updated)
print("Judgments not found:", not_found)

# ============================================================
# Q12 VALIDATION
# ============================================================

df = pd.read_excel(
    WORKBOOK_PATH,
    sheet_name="Element Annotation"
)

df["element_recovered"] = (
    df["element_recovered"]
    .astype("string")
    .str.strip()
)

q12 = df[df["query_id"] == "Q12"].copy()

complete = q12["element_recovered"].isin(["Yes", "No"])

errors = []

for au, elements in q12_judgments.items():
    for element_id, expected in elements.items():

        match = q12[
            (q12["annotation_unit_id"] == au)
            & (q12["element_id"] == element_id)
        ]

        if len(match) != 1:
            errors.append(
                f"{au}/{element_id}: found {len(match)} rows"
            )
            continue

        actual = match.iloc[0]["element_recovered"]

        if actual != expected:
            errors.append(
                f"{au}/{element_id}: actual={actual}, expected={expected}"
            )

print("\nQ12 VALIDATION")
print("-" * 75)

print("Total element judgments:", len(q12))
print("Completed judgments:", int(complete.sum()))
print("Remaining judgments:", int((~complete).sum()))
print(
    "Unique annotation units:",
    q12["annotation_unit_id"].nunique()
)
print(
    "Yes:",
    int((q12["element_recovered"] == "Yes").sum())
)
print(
    "No:",
    int((q12["element_recovered"] == "No").sum())
)
print("Validation errors:", len(errors))

if errors:
    for error in errors:
        print(" -", error)

if (
    len(q12) == 28
    and complete.sum() == 28
    and q12["annotation_unit_id"].nunique() == 14
    and len(errors) == 0
    and len(not_found) == 0
):
    print("\nQ12 STATUS: PASS — FULLY ANNOTATED")
else:
    print("\nQ12 STATUS: REVIEW REQUIRED")

WRITE REVIEWED Q12 JUDGMENTS
Rows written/confirmed: 28
Judgments not found: []

Q12 VALIDATION
---------------------------------------------------------------------------
Total element judgments: 28
Completed judgments: 28
Remaining judgments: 0
Unique annotation units: 14
Yes: 17
No: 11
Validation errors: 0

Q12 STATUS: PASS — FULLY ANNOTATED


In [46]:
# ============================================================
# CELL 44 — FINAL ANNOTATION INTEGRITY CHECK
# ============================================================

import pandas as pd

WORKBOOK_PATH = "/content/retrieval_annotation_workbook.xlsx"

df = pd.read_excel(
    WORKBOOK_PATH,
    sheet_name="Element Annotation"
)

# Normalize annotation values
df["element_recovered"] = (
    df["element_recovered"]
    .astype("string")
    .str.strip()
)

print("FINAL ANNOTATION INTEGRITY CHECK")
print("=" * 80)

# ------------------------------------------------------------
# 1. Overall counts
# ------------------------------------------------------------

total_rows = len(df)

valid_mask = df["element_recovered"].isin(["Yes", "No"])

completed = int(valid_mask.sum())
remaining = int((~valid_mask).sum())

yes_count = int(
    (df["element_recovered"] == "Yes").sum()
)

no_count = int(
    (df["element_recovered"] == "No").sum()
)

print("\n1. OVERALL COUNTS")
print("-" * 80)

print("Total element judgments:", total_rows)
print("Completed judgments:", completed)
print("Remaining / invalid judgments:", remaining)
print("Yes:", yes_count)
print("No:", no_count)

# ------------------------------------------------------------
# 2. Query coverage
# ------------------------------------------------------------

expected_queries = [
    f"Q{i:02d}"
    for i in range(1, 13)
]

actual_queries = sorted(
    df["query_id"].dropna().unique().tolist()
)

missing_queries = [
    q for q in expected_queries
    if q not in actual_queries
]

unexpected_queries = [
    q for q in actual_queries
    if q not in expected_queries
]

print("\n2. QUERY COVERAGE")
print("-" * 80)

print("Queries found:", actual_queries)
print("Missing queries:", missing_queries)
print("Unexpected queries:", unexpected_queries)

# ------------------------------------------------------------
# 3. Per-query summary
# ------------------------------------------------------------

summary = (
    df.groupby("query_id")
    .agg(
        element_judgments=("element_id", "size"),
        annotation_units=("annotation_unit_id", "nunique"),
        yes_count=(
            "element_recovered",
            lambda s: (s == "Yes").sum()
        ),
        no_count=(
            "element_recovered",
            lambda s: (s == "No").sum()
        ),
        incomplete_count=(
            "element_recovered",
            lambda s: (~s.isin(["Yes", "No"])).sum()
        ),
    )
    .reset_index()
)

print("\n3. PER-QUERY SUMMARY")
print("-" * 80)

print(summary.to_string(index=False))

# ------------------------------------------------------------
# 4. Duplicate annotation keys
# ------------------------------------------------------------

key_cols = [
    "annotation_unit_id",
    "query_id",
    "element_id"
]

duplicate_mask = df.duplicated(
    subset=key_cols,
    keep=False
)

duplicate_rows = df[duplicate_mask].copy()

print("\n4. DUPLICATE ANNOTATION KEYS")
print("-" * 80)

print("Duplicate rows:", len(duplicate_rows))

if len(duplicate_rows) > 0:
    print(
        duplicate_rows[
            key_cols + ["element_recovered"]
        ].to_string(index=False)
    )

# ------------------------------------------------------------
# 5. Missing critical fields
# ------------------------------------------------------------

critical_cols = [
    "annotation_unit_id",
    "query_id",
    "element_id",
    "expected_element",
    "retrieval_text",
    "element_recovered"
]

missing_by_column = {}

for col in critical_cols:
    missing_by_column[col] = int(
        df[col].isna().sum()
    )

print("\n5. MISSING CRITICAL FIELDS")
print("-" * 80)

for col, count in missing_by_column.items():
    print(f"{col}: {count}")

# ------------------------------------------------------------
# 6. Invalid annotation values
# ------------------------------------------------------------

invalid_values = df[
    ~df["element_recovered"].isin(["Yes", "No"])
][
    [
        "annotation_unit_id",
        "query_id",
        "element_id",
        "element_recovered"
    ]
]

print("\n6. INVALID / INCOMPLETE VALUES")
print("-" * 80)

print("Invalid/incomplete rows:", len(invalid_values))

if len(invalid_values) > 0:
    print(invalid_values.to_string(index=False))

# ------------------------------------------------------------
# 7. Unique annotation units
# ------------------------------------------------------------

unique_units = df["annotation_unit_id"].nunique()

print("\n7. ANNOTATION UNITS")
print("-" * 80)

print("Unique annotation units:", unique_units)

# ------------------------------------------------------------
# 8. Final status
# ------------------------------------------------------------

all_critical_complete = all(
    count == 0
    for count in missing_by_column.values()
)

final_pass = (
    total_rows == 674
    and completed == 674
    and remaining == 0
    and len(missing_queries) == 0
    and len(unexpected_queries) == 0
    and len(duplicate_rows) == 0
    and all_critical_complete
    and len(invalid_values) == 0
)

print("\n" + "=" * 80)

if final_pass:
    print(
        "FINAL STATUS: PASS — "
        "ALL 674 ELEMENT JUDGMENTS ARE COMPLETE AND VALID"
    )
else:
    print(
        "FINAL STATUS: REVIEW REQUIRED"
    )

print("=" * 80)

FINAL ANNOTATION INTEGRITY CHECK

1. OVERALL COUNTS
--------------------------------------------------------------------------------
Total element judgments: 674
Completed judgments: 674
Remaining / invalid judgments: 0
Yes: 211
No: 463

2. QUERY COVERAGE
--------------------------------------------------------------------------------
Queries found: ['Q01', 'Q02', 'Q03', 'Q04', 'Q05', 'Q06', 'Q07', 'Q08', 'Q09', 'Q10', 'Q11', 'Q12']
Missing queries: []
Unexpected queries: []

3. PER-QUERY SUMMARY
--------------------------------------------------------------------------------
query_id  element_judgments  annotation_units  yes_count  no_count  incomplete_count
     Q01                 20                20         12         8                 0
     Q02                 40                20          4        36                 0
     Q03                 63                21         16        47                 0
     Q04                 57                19         21        36           

In [47]:
# ============================================================
# CELL 45 — EXPORT FINAL HUMAN ELEMENT ANNOTATIONS
# ============================================================

import os
import pandas as pd

WORKBOOK_PATH = "/content/retrieval_annotation_workbook.xlsx"

OUTPUT_DIR = "/content/retrieval-experiments/evaluation"
OUTPUT_PATH = os.path.join(
    OUTPUT_DIR,
    "final_human_element_annotations.csv"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# 1. Load reviewed element annotations
# ------------------------------------------------------------

df = pd.read_excel(
    WORKBOOK_PATH,
    sheet_name="Element Annotation"
)

df["element_recovered"] = (
    df["element_recovered"]
    .astype("string")
    .str.strip()
)

# ------------------------------------------------------------
# 2. Final safety checks before export
# ------------------------------------------------------------

valid_mask = df["element_recovered"].isin(["Yes", "No"])

assert len(df) == 674, (
    f"Expected 674 judgments, found {len(df)}"
)

assert valid_mask.all(), (
    "Export stopped: incomplete or invalid annotations found."
)

assert not df.duplicated(
    subset=[
        "annotation_unit_id",
        "query_id",
        "element_id"
    ]
).any(), (
    "Export stopped: duplicate annotation keys found."
)

# ------------------------------------------------------------
# 3. Select fields to preserve
# ------------------------------------------------------------

export_columns = [
    "annotation_unit_id",
    "query_id",
    "query_type",
    "query_ar",
    "element_id",
    "expected_element",
    "retrieval_text",
    "element_recovered",
    "element_annotation_notes",
]

final_annotations = df[export_columns].copy()

# ------------------------------------------------------------
# 4. Export
# ------------------------------------------------------------

final_annotations.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 5. Verify exported file
# ------------------------------------------------------------

check = pd.read_csv(OUTPUT_PATH)

print("FINAL HUMAN ANNOTATION EXPORT")
print("=" * 75)

print("Output file:", OUTPUT_PATH)
print("Rows exported:", len(check))
print(
    "Unique annotation units:",
    check["annotation_unit_id"].nunique()
)
print(
    "Queries:",
    check["query_id"].nunique()
)
print(
    "Yes:",
    (check["element_recovered"] == "Yes").sum()
)
print(
    "No:",
    (check["element_recovered"] == "No").sum()
)

if (
    len(check) == 674
    and check["annotation_unit_id"].nunique() == 220
    and check["query_id"].nunique() == 12
    and (check["element_recovered"] == "Yes").sum() == 211
    and (check["element_recovered"] == "No").sum() == 463
):
    print(
        "\nEXPORT STATUS: PASS — "
        "FINAL HUMAN ANNOTATIONS SAVED SUCCESSFULLY"
    )
else:
    print("\nEXPORT STATUS: REVIEW REQUIRED")

FINAL HUMAN ANNOTATION EXPORT
Output file: /content/retrieval-experiments/evaluation/final_human_element_annotations.csv
Rows exported: 674
Unique annotation units: 220
Queries: 12
Yes: 211
No: 463

EXPORT STATUS: PASS — FINAL HUMAN ANNOTATIONS SAVED SUCCESSFULLY


In [50]:
# ============================================================
# CELL 46 — PROPAGATE HUMAN ANNOTATIONS TO RAW RETRIEVAL RECORDS
# ============================================================

import os
import pandas as pd

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

BASE_DIR = "/content/retrieval-experiments"

# Correct location of the raw retrieval results
RAW_PATH = "/content/raw_retrieval_results.csv"

# Final reviewed human annotations created in Cell 45
ANNOTATION_PATH = os.path.join(
    BASE_DIR,
    "evaluation",
    "final_human_element_annotations.csv"
)

# New derived output — raw file remains unchanged
OUTPUT_PATH = os.path.join(
    BASE_DIR,
    "evaluation",
    "annotated_retrieval_results.csv"
)

# ------------------------------------------------------------
# 1. CHECK INPUT FILES
# ------------------------------------------------------------

assert os.path.exists(RAW_PATH), (
    f"Raw retrieval file not found: {RAW_PATH}"
)

assert os.path.exists(ANNOTATION_PATH), (
    f"Annotation file not found: {ANNOTATION_PATH}"
)

# ------------------------------------------------------------
# 2. LOAD FILES
# ------------------------------------------------------------

raw = pd.read_csv(RAW_PATH)
ann = pd.read_csv(ANNOTATION_PATH)

ann["element_recovered"] = (
    ann["element_recovered"]
    .astype("string")
    .str.strip()
)

print("PROPAGATE HUMAN ANNOTATIONS")
print("=" * 80)

print("Raw retrieval records:", len(raw))
print("Element annotation rows:", len(ann))

# ------------------------------------------------------------
# 3. SAFETY CHECKS
# ------------------------------------------------------------

assert len(raw) == 648, (
    f"Expected 648 raw retrieval records, found {len(raw)}"
)

assert len(ann) == 674, (
    f"Expected 674 element judgments, found {len(ann)}"
)

assert ann["element_recovered"].isin(
    ["Yes", "No"]
).all(), (
    "Invalid or incomplete element annotation values found."
)

required_raw_columns = [
    "query_id",
    "retrieval_text"
]

required_annotation_columns = [
    "query_id",
    "retrieval_text",
    "element_id",
    "element_recovered"
]

for col in required_raw_columns:
    assert col in raw.columns, (
        f"Missing required raw-results column: {col}"
    )

for col in required_annotation_columns:
    assert col in ann.columns, (
        f"Missing required annotation column: {col}"
    )

# ------------------------------------------------------------
# 4. BUILD ONE ANNOTATION RECORD PER UNIQUE
#    (query_id, retrieval_text)
# ------------------------------------------------------------

def aggregate_annotation(group):

    expected_count = len(group)

    recovered_count = int(
        (group["element_recovered"] == "Yes").sum()
    )

    recovered_ids = (
        group.loc[
            group["element_recovered"] == "Yes",
            "element_id"
        ]
        .astype(str)
        .tolist()
    )

    missing_ids = (
        group.loc[
            group["element_recovered"] == "No",
            "element_id"
        ]
        .astype(str)
        .tolist()
    )

    # Derived relevance follows the frozen annotation rules
    if recovered_count == expected_count:
        relevance = "Relevant"

    elif recovered_count == 0:
        relevance = "Not Relevant"

    else:
        relevance = "Partially Relevant"

    return pd.Series({
        "expected_element_count": expected_count,
        "recovered_element_count": recovered_count,
        "recovered_element_ids": "|".join(recovered_ids),
        "missing_element_ids": "|".join(missing_ids),
        "derived_relevance": relevance,
    })


annotation_units = (
    ann.groupby(
        ["query_id", "retrieval_text"],
        sort=False,
        dropna=False
    )
    .apply(
        aggregate_annotation,
        include_groups=False
    )
    .reset_index()
)

print(
    "Unique annotation units reconstructed:",
    len(annotation_units)
)

assert len(annotation_units) == 220, (
    f"Expected 220 unique annotation units, "
    f"found {len(annotation_units)}"
)

# ------------------------------------------------------------
# 5. CHECK THAT ANNOTATION UNIT KEYS ARE UNIQUE
# ------------------------------------------------------------

duplicate_annotation_units = (
    annotation_units.duplicated(
        subset=["query_id", "retrieval_text"],
        keep=False
    )
)

assert not duplicate_annotation_units.any(), (
    "Duplicate annotation-unit keys found."
)

# ------------------------------------------------------------
# 6. PROPAGATE ANNOTATIONS TO ALL 648 RAW RECORDS
# ------------------------------------------------------------

annotated_raw = raw.merge(
    annotation_units,
    on=[
        "query_id",
        "retrieval_text"
    ],
    how="left",
    validate="many_to_one"
)

# ------------------------------------------------------------
# 7. VALIDATE PROPAGATION
# ------------------------------------------------------------

missing_propagation = annotated_raw[
    annotated_raw["derived_relevance"].isna()
].copy()

print(
    "Records without propagated annotation:",
    len(missing_propagation)
)

assert len(annotated_raw) == 648, (
    f"Row count changed during propagation: "
    f"{len(annotated_raw)}"
)

assert len(missing_propagation) == 0, (
    "Some raw retrieval records did not receive "
    "human-derived annotations."
)

# Validate recovered counts
assert (
    annotated_raw["recovered_element_count"]
    <= annotated_raw["expected_element_count"]
).all(), (
    "Recovered element count exceeds expected element count."
)

# ------------------------------------------------------------
# 8. SAVE DERIVED ANNOTATED RETRIEVAL RESULTS
# ------------------------------------------------------------

os.makedirs(
    os.path.dirname(OUTPUT_PATH),
    exist_ok=True
)

annotated_raw.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 9. RELOAD SAVED FILE AND VERIFY
# ------------------------------------------------------------

check = pd.read_csv(OUTPUT_PATH)

print("\nPROPAGATION SUMMARY")
print("-" * 80)

print("Output file:", OUTPUT_PATH)
print("Rows:", len(check))

print("\nDerived relevance distribution:")

print(
    check["derived_relevance"]
    .value_counts()
    .to_string()
)

unique_query_text_pairs = (
    check[
        ["query_id", "retrieval_text"]
    ]
    .drop_duplicates()
    .shape[0]
)

print(
    "\nUnique query-text pairs:",
    unique_query_text_pairs
)

print(
    "Queries represented:",
    check["query_id"].nunique()
)

# ------------------------------------------------------------
# 10. FINAL STATUS
# ------------------------------------------------------------

final_pass = (
    len(check) == 648
    and check["derived_relevance"].notna().all()
    and unique_query_text_pairs == 220
    and check["query_id"].nunique() == 12
)

print("\n" + "=" * 80)

if final_pass:
    print(
        "PROPAGATION STATUS: PASS — "
        "ALL 648 RAW RETRIEVAL RECORDS ANNOTATED"
    )
else:
    print(
        "PROPAGATION STATUS: REVIEW REQUIRED"
    )

print("=" * 80)

PROPAGATE HUMAN ANNOTATIONS
Raw retrieval records: 648
Element annotation rows: 674
Unique annotation units reconstructed: 220
Records without propagated annotation: 0

PROPAGATION SUMMARY
--------------------------------------------------------------------------------
Output file: /content/retrieval-experiments/evaluation/annotated_retrieval_results.csv
Rows: 648

Derived relevance distribution:
derived_relevance
Not Relevant          337
Relevant              189
Partially Relevant    122

Unique query-text pairs: 220
Queries represented: 12

PROPAGATION STATUS: PASS — ALL 648 RAW RETRIEVAL RECORDS ANNOTATED


In [51]:
# ============================================================
# CELL 47 — VALIDATE ANNOTATED RETRIEVAL STRUCTURE
#           BEFORE METRIC CALCULATION
# ============================================================

import os
import pandas as pd

ANNOTATED_PATH = (
    "/content/retrieval-experiments/evaluation/"
    "annotated_retrieval_results.csv"
)

df = pd.read_csv(ANNOTATED_PATH)

print("PRE-METRIC STRUCTURE VALIDATION")
print("=" * 80)

# ------------------------------------------------------------
# 1. BASIC STRUCTURE
# ------------------------------------------------------------

print("\n1. BASIC COUNTS")
print("-" * 80)

print("Rows:", len(df))
print("Queries:", df["query_id"].nunique())

# Show columns so we verify the actual raw-result schema
print("\nColumns:")
for col in df.columns:
    print(" -", col)

# ------------------------------------------------------------
# 2. IDENTIFY EXPERIMENTAL CONDITION COLUMNS
# ------------------------------------------------------------

candidate_condition_columns = [
    "chunk_config",
    "chunk_configuration",
    "configuration",
    "overlap",
    "overlap_setting",
    "embedding_model",
    "embedding_id",
    "model_id",
]

present_condition_columns = [
    col for col in candidate_condition_columns
    if col in df.columns
]

print("\n2. CONDITION-RELATED COLUMNS FOUND")
print("-" * 80)

print(present_condition_columns)

for col in present_condition_columns:
    print(f"\n{col}:")
    print(
        df[col]
        .value_counts(dropna=False)
        .to_string()
    )

# ------------------------------------------------------------
# 3. FIND RANK COLUMN
# ------------------------------------------------------------

candidate_rank_columns = [
    "rank",
    "retrieval_rank",
    "result_rank"
]

rank_columns_found = [
    col for col in candidate_rank_columns
    if col in df.columns
]

print("\n3. RANK COLUMN")
print("-" * 80)

print("Rank columns found:", rank_columns_found)

if len(rank_columns_found) == 1:

    rank_col = rank_columns_found[0]

    print("\nRank distribution:")
    print(
        df[rank_col]
        .value_counts()
        .sort_index()
        .to_string()
    )

else:
    rank_col = None

# ------------------------------------------------------------
# 4. CHECK EXPECTED ELEMENT COUNTS BY QUERY
# ------------------------------------------------------------

expected_counts = (
    df.groupby("query_id")["expected_element_count"]
    .unique()
)

print("\n4. EXPECTED ELEMENT COUNTS BY QUERY")
print("-" * 80)

for query_id, values in expected_counts.items():
    print(
        query_id,
        "->",
        list(values)
    )

# ------------------------------------------------------------
# 5. CHECK RELEVANCE DISTRIBUTION
# ------------------------------------------------------------

print("\n5. DERIVED RELEVANCE")
print("-" * 80)

print(
    df["derived_relevance"]
    .value_counts()
    .to_string()
)

# ------------------------------------------------------------
# 6. CHECK RECOVERED ELEMENT COUNT RANGE
# ------------------------------------------------------------

print("\n6. RECOVERED ELEMENT COUNT RANGE")
print("-" * 80)

print(
    df[
        [
            "expected_element_count",
            "recovered_element_count"
        ]
    ]
    .describe()
    .to_string()
)

invalid_counts = df[
    (df["recovered_element_count"] < 0)
    |
    (
        df["recovered_element_count"]
        > df["expected_element_count"]
    )
]

print(
    "\nInvalid recovered-element counts:",
    len(invalid_counts)
)

# ------------------------------------------------------------
# 7. FINAL PRE-METRIC STATUS
# ------------------------------------------------------------

basic_pass = (
    len(df) == 648
    and df["query_id"].nunique() == 12
    and df["derived_relevance"].notna().all()
    and len(invalid_counts) == 0
    and len(rank_columns_found) == 1
)

print("\n" + "=" * 80)

if basic_pass:
    print(
        "PRE-METRIC STATUS: PASS — "
        "ANNOTATED DATA STRUCTURE READY FOR METRIC CALCULATION"
    )
else:
    print(
        "PRE-METRIC STATUS: REVIEW REQUIRED"
    )

print("=" * 80)

PRE-METRIC STRUCTURE VALIDATION

1. BASIC COUNTS
--------------------------------------------------------------------------------
Rows: 648
Queries: 12

Columns:
 - model_id
 - model_name
 - configuration
 - overlap_setting
 - query_id
 - query_type
 - query_ar
 - rank
 - chunk_id
 - chunk_title
 - similarity_score
 - core_source_unit_ids
 - overlap_source_unit_ids
 - retrieval_source_unit_ids
 - retrieval_text
 - expected_element_count
 - recovered_element_count
 - recovered_element_ids
 - missing_element_ids
 - derived_relevance

2. CONDITION-RELATED COLUMNS FOUND
--------------------------------------------------------------------------------
['configuration', 'overlap_setting', 'model_id']

configuration:
configuration
A_Fine        216
B_Baseline    216
C_Broad       216

overlap_setting:
overlap_setting
O0    324
O1    324

model_id:
model_id
E1    216
E2    216
E3    216

3. RANK COLUMN
--------------------------------------------------------------------------------
Rank columns

In [52]:
# ============================================================
# CELL 48 — CALCULATE QUERY-CONDITION RETRIEVAL METRICS
# ============================================================

import os
import pandas as pd

ANNOTATED_PATH = (
    "/content/retrieval-experiments/evaluation/"
    "annotated_retrieval_results.csv"
)

OUTPUT_PATH = (
    "/content/retrieval-experiments/evaluation/"
    "query_condition_metrics.csv"
)

df = pd.read_csv(ANNOTATED_PATH)

print("QUERY-CONDITION RETRIEVAL METRICS")
print("=" * 80)

# ------------------------------------------------------------
# 1. NORMALIZE RECOVERED ELEMENT IDs
# ------------------------------------------------------------

def parse_element_ids(value):
    if pd.isna(value):
        return set()

    value = str(value).strip()

    if value == "":
        return set()

    return {
        x.strip()
        for x in value.split("|")
        if x.strip()
    }


df["_recovered_set"] = (
    df["recovered_element_ids"]
    .apply(parse_element_ids)
)

# ------------------------------------------------------------
# 2. DEFINE ONE EXPERIMENTAL RUN
# ------------------------------------------------------------

condition_cols = [
    "configuration",
    "overlap_setting",
    "model_id"
]

run_cols = condition_cols + [
    "query_id"
]

# Each run must contain Top-3 retrieval results
run_sizes = (
    df.groupby(run_cols)
    .size()
)

print("\n1. RUN STRUCTURE")
print("-" * 80)

print("Query-condition runs:", len(run_sizes))
print("Minimum records per run:", run_sizes.min())
print("Maximum records per run:", run_sizes.max())

assert len(run_sizes) == 216, (
    f"Expected 216 query-condition runs, "
    f"found {len(run_sizes)}"
)

assert (run_sizes == 3).all(), (
    "Every query-condition run must contain exactly 3 ranks."
)

# ------------------------------------------------------------
# 3. CALCULATE METRICS FOR EACH QUERY-CONDITION RUN
# ------------------------------------------------------------

metric_rows = []

for keys, group in df.groupby(
    run_cols,
    sort=False
):

    configuration, overlap_setting, model_id, query_id = keys

    group = group.sort_values("rank")

    # Validate rank structure
    ranks = group["rank"].tolist()

    assert ranks == [1, 2, 3], (
        f"Unexpected ranks for "
        f"{configuration}/{overlap_setting}/"
        f"{model_id}/{query_id}: {ranks}"
    )

    # Expected number of elements must be constant
    expected_counts = (
        group["expected_element_count"]
        .dropna()
        .unique()
    )

    assert len(expected_counts) == 1, (
        f"Inconsistent expected element count "
        f"for {query_id}"
    )

    expected_count = int(expected_counts[0])

    # Expected IDs are E01 ... E0N
    expected_ids = {
        f"E{i:02d}"
        for i in range(1, expected_count + 1)
    }

    # --------------------------------------------------------
    # Rank 1 recovered elements
    # --------------------------------------------------------

    rank1_set = set(
        group.loc[
            group["rank"] == 1,
            "_recovered_set"
        ].iloc[0]
    )

    # --------------------------------------------------------
    # Union across Top-3
    # --------------------------------------------------------

    top3_union = set()

    for recovered_set in group["_recovered_set"]:
        top3_union.update(recovered_set)

    # Safety: no unexpected element IDs
    unexpected_rank1 = rank1_set - expected_ids
    unexpected_top3 = top3_union - expected_ids

    assert not unexpected_rank1, (
        f"Unexpected Rank-1 element IDs: "
        f"{unexpected_rank1}"
    )

    assert not unexpected_top3, (
        f"Unexpected Top-3 element IDs: "
        f"{unexpected_top3}"
    )

    # --------------------------------------------------------
    # Full Hit@1
    # 1 iff Rank 1 recovers ALL expected elements
    # --------------------------------------------------------

    full_hit_at_1 = int(
        expected_ids.issubset(rank1_set)
    )

    # --------------------------------------------------------
    # Full Hit@3
    # 1 iff union of Top-3 recovers ALL expected elements
    # --------------------------------------------------------

    full_hit_at_3 = int(
        expected_ids.issubset(top3_union)
    )

    # --------------------------------------------------------
    # Coverage@3
    # Fraction of expected elements recovered in Top-3 union
    # --------------------------------------------------------

    coverage_at_3 = (
        len(expected_ids.intersection(top3_union))
        / expected_count
    )

    metric_rows.append({
        "configuration": configuration,
        "overlap_setting": overlap_setting,
        "model_id": model_id,
        "query_id": query_id,
        "query_type": group["query_type"].iloc[0],
        "expected_element_count": expected_count,
        "rank1_recovered_count": len(rank1_set),
        "top3_recovered_count": len(top3_union),
        "full_hit_at_1": full_hit_at_1,
        "full_hit_at_3": full_hit_at_3,
        "coverage_at_3": coverage_at_3,
    })


metrics = pd.DataFrame(metric_rows)

# ------------------------------------------------------------
# 4. VALIDATE METRIC TABLE
# ------------------------------------------------------------

print("\n2. METRIC TABLE")
print("-" * 80)

print("Metric rows:", len(metrics))
print(
    "Experimental conditions:",
    metrics[
        condition_cols
    ].drop_duplicates().shape[0]
)

print(
    "Queries per condition:",
    metrics.groupby(condition_cols)["query_id"]
    .nunique()
    .min(),
    "to",
    metrics.groupby(condition_cols)["query_id"]
    .nunique()
    .max()
)

assert len(metrics) == 216, (
    f"Expected 216 metric rows, found {len(metrics)}"
)

assert (
    metrics[condition_cols]
    .drop_duplicates()
    .shape[0]
    == 18
), "Expected 18 experimental conditions."

assert (
    metrics.groupby(condition_cols)["query_id"]
    .nunique()
    == 12
).all(), (
    "Every condition must contain all 12 queries."
)

assert metrics["full_hit_at_1"].isin([0, 1]).all()
assert metrics["full_hit_at_3"].isin([0, 1]).all()

assert metrics["coverage_at_3"].between(
    0, 1
).all()

# Logical consistency:
# Full Hit@1 implies Full Hit@3
assert (
    metrics["full_hit_at_1"]
    <= metrics["full_hit_at_3"]
).all(), (
    "Logical error: Full Hit@1 cannot exceed Full Hit@3."
)

# Full Hit@3 must correspond to complete coverage
assert (
    (
        metrics["full_hit_at_3"] == 1
    )
    ==
    (
        metrics["coverage_at_3"] == 1.0
    )
).all(), (
    "Full Hit@3 and complete Coverage@3 are inconsistent."
)

# ------------------------------------------------------------
# 5. SAVE QUERY-CONDITION METRICS
# ------------------------------------------------------------

metrics.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 6. OVERALL DIAGNOSTIC SUMMARY
# ------------------------------------------------------------

print("\n3. OVERALL DIAGNOSTIC SUMMARY")
print("-" * 80)

print(
    "Full Hit@1 successful runs:",
    int(metrics["full_hit_at_1"].sum()),
    "/",
    len(metrics)
)

print(
    "Full Hit@3 successful runs:",
    int(metrics["full_hit_at_3"].sum()),
    "/",
    len(metrics)
)

print(
    "Mean Coverage@3:",
    round(metrics["coverage_at_3"].mean(), 4)
)

print("\nOutput file:")
print(OUTPUT_PATH)

# ------------------------------------------------------------
# 7. FINAL STATUS
# ------------------------------------------------------------

print("\n" + "=" * 80)

if (
    len(metrics) == 216
    and metrics[condition_cols]
        .drop_duplicates()
        .shape[0] == 18
    and metrics["coverage_at_3"]
        .between(0, 1)
        .all()
):
    print(
        "METRIC CALCULATION STATUS: PASS — "
        "216 QUERY-CONDITION RUNS CALCULATED"
    )
else:
    print(
        "METRIC CALCULATION STATUS: REVIEW REQUIRED"
    )

print("=" * 80)

QUERY-CONDITION RETRIEVAL METRICS

1. RUN STRUCTURE
--------------------------------------------------------------------------------
Query-condition runs: 216
Minimum records per run: 3
Maximum records per run: 3

2. METRIC TABLE
--------------------------------------------------------------------------------
Metric rows: 216
Experimental conditions: 18
Queries per condition: 12 to 12

3. OVERALL DIAGNOSTIC SUMMARY
--------------------------------------------------------------------------------
Full Hit@1 successful runs: 120 / 216
Full Hit@3 successful runs: 175 / 216
Mean Coverage@3: 0.8677

Output file:
/content/retrieval-experiments/evaluation/query_condition_metrics.csv

METRIC CALCULATION STATUS: PASS — 216 QUERY-CONDITION RUNS CALCULATED


In [53]:
# ============================================================
# CELL 49 — CONDITION-LEVEL RETRIEVAL SUMMARY
# ============================================================

import os
import pandas as pd

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

METRICS_PATH = (
    "/content/retrieval-experiments/evaluation/"
    "query_condition_metrics.csv"
)

OUTPUT_PATH = (
    "/content/retrieval-experiments/evaluation/"
    "condition_level_summary.csv"
)

# ------------------------------------------------------------
# 1. LOAD QUERY-CONDITION METRICS
# ------------------------------------------------------------

metrics = pd.read_csv(METRICS_PATH)

print("CONDITION-LEVEL RETRIEVAL SUMMARY")
print("=" * 90)

assert len(metrics) == 216, (
    f"Expected 216 query-condition rows, "
    f"found {len(metrics)}"
)

condition_cols = [
    "configuration",
    "overlap_setting",
    "model_id"
]

# ------------------------------------------------------------
# 2. BUILD CONDITION-LEVEL SUMMARY
# ------------------------------------------------------------

summary = (
    metrics
    .groupby(
        condition_cols,
        as_index=False
    )
    .agg(
        query_count=(
            "query_id",
            "nunique"
        ),
        full_hit_at_1_count=(
            "full_hit_at_1",
            "sum"
        ),
        full_hit_at_1_rate=(
            "full_hit_at_1",
            "mean"
        ),
        full_hit_at_3_count=(
            "full_hit_at_3",
            "sum"
        ),
        full_hit_at_3_rate=(
            "full_hit_at_3",
            "mean"
        ),
        mean_coverage_at_3=(
            "coverage_at_3",
            "mean"
        ),
        min_coverage_at_3=(
            "coverage_at_3",
            "min"
        ),
        max_coverage_at_3=(
            "coverage_at_3",
            "max"
        )
    )
)

# ------------------------------------------------------------
# 3. VALIDATE STRUCTURE
# ------------------------------------------------------------

print("\n1. STRUCTURE CHECK")
print("-" * 90)

print("Conditions:", len(summary))

print(
    "Queries per condition:",
    summary["query_count"].min(),
    "to",
    summary["query_count"].max()
)

assert len(summary) == 18, (
    f"Expected 18 conditions, found {len(summary)}"
)

assert (
    summary["query_count"] == 12
).all(), (
    "Every condition must contain exactly 12 queries."
)

assert (
    summary["full_hit_at_1_count"]
    <= summary["full_hit_at_3_count"]
).all(), (
    "Full Hit@1 count cannot exceed Full Hit@3 count."
)

assert (
    summary["full_hit_at_1_count"]
    <= 12
).all()

assert (
    summary["full_hit_at_3_count"]
    <= 12
).all()

assert (
    summary["mean_coverage_at_3"]
    .between(0, 1)
    .all()
)

# ------------------------------------------------------------
# 4. ADD DISPLAY PERCENTAGES
# ------------------------------------------------------------

summary["full_hit_at_1_percent"] = (
    summary["full_hit_at_1_rate"] * 100
).round(2)

summary["full_hit_at_3_percent"] = (
    summary["full_hit_at_3_rate"] * 100
).round(2)

summary["mean_coverage_at_3_percent"] = (
    summary["mean_coverage_at_3"] * 100
).round(2)

# ------------------------------------------------------------
# 5. ORDER TABLE
# ------------------------------------------------------------

summary = summary[
    [
        "configuration",
        "overlap_setting",
        "model_id",
        "query_count",

        "full_hit_at_1_count",
        "full_hit_at_1_rate",
        "full_hit_at_1_percent",

        "full_hit_at_3_count",
        "full_hit_at_3_rate",
        "full_hit_at_3_percent",

        "mean_coverage_at_3",
        "mean_coverage_at_3_percent",

        "min_coverage_at_3",
        "max_coverage_at_3",
    ]
]

summary = summary.sort_values(
    by=[
        "configuration",
        "overlap_setting",
        "model_id"
    ]
).reset_index(drop=True)

# ------------------------------------------------------------
# 6. SAVE SUMMARY
# ------------------------------------------------------------

summary.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 7. DISPLAY RESULTS
# ------------------------------------------------------------

display_cols = [
    "configuration",
    "overlap_setting",
    "model_id",
    "full_hit_at_1_count",
    "full_hit_at_1_percent",
    "full_hit_at_3_count",
    "full_hit_at_3_percent",
    "mean_coverage_at_3_percent",
    "min_coverage_at_3",
]

print("\n2. RESULTS BY EXPERIMENTAL CONDITION")
print("-" * 90)

print(
    summary[display_cols]
    .to_string(index=False)
)

# ------------------------------------------------------------
# 8. OVERALL CONSISTENCY CHECK
# ------------------------------------------------------------

total_hit1 = int(
    summary["full_hit_at_1_count"].sum()
)

total_hit3 = int(
    summary["full_hit_at_3_count"].sum()
)

overall_mean_coverage = (
    metrics["coverage_at_3"].mean()
)

print("\n3. OVERALL CONSISTENCY CHECK")
print("-" * 90)

print(
    "Full Hit@1 total:",
    total_hit1,
    "/ 216"
)

print(
    "Full Hit@3 total:",
    total_hit3,
    "/ 216"
)

print(
    "Overall Mean Coverage@3:",
    round(overall_mean_coverage, 4)
)

assert total_hit1 == 120, (
    f"Expected Full Hit@1 total = 120, "
    f"found {total_hit1}"
)

assert total_hit3 == 175, (
    f"Expected Full Hit@3 total = 175, "
    f"found {total_hit3}"
)

assert abs(
    overall_mean_coverage - 0.8677
) < 0.0001, (
    "Overall Mean Coverage@3 does not match Cell 48."
)

# ------------------------------------------------------------
# 9. FINAL STATUS
# ------------------------------------------------------------

print("\nOutput file:")
print(OUTPUT_PATH)

print("\n" + "=" * 90)

print(
    "CONDITION SUMMARY STATUS: PASS — "
    "18 EXPERIMENTAL CONDITIONS SUMMARIZED"
)

print("=" * 90)

CONDITION-LEVEL RETRIEVAL SUMMARY

1. STRUCTURE CHECK
------------------------------------------------------------------------------------------
Conditions: 18
Queries per condition: 12 to 12

2. RESULTS BY EXPERIMENTAL CONDITION
------------------------------------------------------------------------------------------
configuration overlap_setting model_id  full_hit_at_1_count  full_hit_at_1_percent  full_hit_at_3_count  full_hit_at_3_percent  mean_coverage_at_3_percent  min_coverage_at_3
       A_Fine              O0       E1                    6                  50.00                    8                  66.67                       76.79           0.000000
       A_Fine              O0       E2                    8                  66.67                    9                  75.00                       80.36           0.000000
       A_Fine              O0       E3                    8                  66.67                    9                  75.00                       80.36   

In [54]:
# ============================================================
# CELL 50 — FACTOR-LEVEL RETRIEVAL ANALYSIS
# ============================================================

import os
import pandas as pd

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

METRICS_PATH = (
    "/content/retrieval-experiments/evaluation/"
    "query_condition_metrics.csv"
)

OUTPUT_DIR = (
    "/content/retrieval-experiments/evaluation"
)

metrics = pd.read_csv(METRICS_PATH)

print("FACTOR-LEVEL RETRIEVAL ANALYSIS")
print("=" * 90)

assert len(metrics) == 216

# ------------------------------------------------------------
# HELPER FUNCTION
# ------------------------------------------------------------

def summarize_factor(df, factor):

    summary = (
        df.groupby(
            factor,
            as_index=False
        )
        .agg(
            run_count=(
                "query_id",
                "size"
            ),
            full_hit_at_1_count=(
                "full_hit_at_1",
                "sum"
            ),
            full_hit_at_1_rate=(
                "full_hit_at_1",
                "mean"
            ),
            full_hit_at_3_count=(
                "full_hit_at_3",
                "sum"
            ),
            full_hit_at_3_rate=(
                "full_hit_at_3",
                "mean"
            ),
            mean_coverage_at_3=(
                "coverage_at_3",
                "mean"
            ),
            min_coverage_at_3=(
                "coverage_at_3",
                "min"
            )
        )
    )

    summary["full_hit_at_1_percent"] = (
        summary["full_hit_at_1_rate"] * 100
    ).round(2)

    summary["full_hit_at_3_percent"] = (
        summary["full_hit_at_3_rate"] * 100
    ).round(2)

    summary["mean_coverage_at_3_percent"] = (
        summary["mean_coverage_at_3"] * 100
    ).round(2)

    return summary


# ------------------------------------------------------------
# 1. CHUNK CONFIGURATION
# ------------------------------------------------------------

config_summary = summarize_factor(
    metrics,
    "configuration"
)

print("\n1. CHUNK CONFIGURATION")
print("-" * 90)

print(
    config_summary[
        [
            "configuration",
            "run_count",
            "full_hit_at_1_count",
            "full_hit_at_1_percent",
            "full_hit_at_3_count",
            "full_hit_at_3_percent",
            "mean_coverage_at_3_percent",
            "min_coverage_at_3",
        ]
    ].to_string(index=False)
)

config_path = os.path.join(
    OUTPUT_DIR,
    "factor_summary_configuration.csv"
)

config_summary.to_csv(
    config_path,
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 2. OVERLAP SETTING
# ------------------------------------------------------------

overlap_summary = summarize_factor(
    metrics,
    "overlap_setting"
)

print("\n2. OVERLAP SETTING")
print("-" * 90)

print(
    overlap_summary[
        [
            "overlap_setting",
            "run_count",
            "full_hit_at_1_count",
            "full_hit_at_1_percent",
            "full_hit_at_3_count",
            "full_hit_at_3_percent",
            "mean_coverage_at_3_percent",
            "min_coverage_at_3",
        ]
    ].to_string(index=False)
)

overlap_path = os.path.join(
    OUTPUT_DIR,
    "factor_summary_overlap.csv"
)

overlap_summary.to_csv(
    overlap_path,
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 3. EMBEDDING MODEL
# ------------------------------------------------------------

model_summary = summarize_factor(
    metrics,
    "model_id"
)

print("\n3. EMBEDDING MODEL")
print("-" * 90)

print(
    model_summary[
        [
            "model_id",
            "run_count",
            "full_hit_at_1_count",
            "full_hit_at_1_percent",
            "full_hit_at_3_count",
            "full_hit_at_3_percent",
            "mean_coverage_at_3_percent",
            "min_coverage_at_3",
        ]
    ].to_string(index=False)
)

model_path = os.path.join(
    OUTPUT_DIR,
    "factor_summary_embedding_model.csv"
)

model_summary.to_csv(
    model_path,
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# 4. INTERNAL CONSISTENCY CHECKS
# ------------------------------------------------------------

print("\n4. CONSISTENCY CHECKS")
print("-" * 90)

print(
    "Configuration run total:",
    config_summary["run_count"].sum()
)

print(
    "Overlap run total:",
    overlap_summary["run_count"].sum()
)

print(
    "Model run total:",
    model_summary["run_count"].sum()
)

assert config_summary["run_count"].sum() == 216
assert overlap_summary["run_count"].sum() == 216
assert model_summary["run_count"].sum() == 216

assert (
    config_summary["full_hit_at_1_count"].sum()
    == 120
)

assert (
    config_summary["full_hit_at_3_count"].sum()
    == 175
)

assert (
    overlap_summary["full_hit_at_1_count"].sum()
    == 120
)

assert (
    overlap_summary["full_hit_at_3_count"].sum()
    == 175
)

assert (
    model_summary["full_hit_at_1_count"].sum()
    == 120
)

assert (
    model_summary["full_hit_at_3_count"].sum()
    == 175
)

# ------------------------------------------------------------
# 5. OUTPUT FILES
# ------------------------------------------------------------

print("\n5. OUTPUT FILES")
print("-" * 90)

print(config_path)
print(overlap_path)
print(model_path)

print("\n" + "=" * 90)

print(
    "FACTOR ANALYSIS STATUS: PASS — "
    "CONFIGURATION, OVERLAP, AND EMBEDDING MODEL SUMMARIZED"
)

print("=" * 90)

FACTOR-LEVEL RETRIEVAL ANALYSIS

1. CHUNK CONFIGURATION
------------------------------------------------------------------------------------------
configuration  run_count  full_hit_at_1_count  full_hit_at_1_percent  full_hit_at_3_count  full_hit_at_3_percent  mean_coverage_at_3_percent  min_coverage_at_3
       A_Fine         72                   40                  55.56                   55                  76.39                       84.95                0.0
   B_Baseline         72                   42                  58.33                   60                  83.33                       88.19                0.0
      C_Broad         72                   38                  52.78                   60                  83.33                       87.17                0.0

2. OVERLAP SETTING
------------------------------------------------------------------------------------------
overlap_setting  run_count  full_hit_at_1_count  full_hit_at_1_percent  full_hit_at_3_count  full_hit_

In [55]:
# ============================================================
# CELL 51 — QUERY-LEVEL RETRIEVAL ANALYSIS
# ============================================================

import os
import pandas as pd

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

METRICS_PATH = (
    "/content/retrieval-experiments/evaluation/"
    "query_condition_metrics.csv"
)

OUTPUT_PATH = (
    "/content/retrieval-experiments/evaluation/"
    "query_level_summary.csv"
)

metrics = pd.read_csv(METRICS_PATH)

print("QUERY-LEVEL RETRIEVAL ANALYSIS")
print("=" * 95)

assert len(metrics) == 216, (
    f"Expected 216 metric rows, found {len(metrics)}"
)

# ------------------------------------------------------------
# 1. VALIDATE QUERY STRUCTURE
# ------------------------------------------------------------

query_counts = (
    metrics.groupby("query_id")
    .size()
)

print("\n1. QUERY STRUCTURE")
print("-" * 95)

print("Queries:", metrics["query_id"].nunique())
print(
    "Runs per query:",
    query_counts.min(),
    "to",
    query_counts.max()
)

assert metrics["query_id"].nunique() == 12
assert (query_counts == 18).all()

# ------------------------------------------------------------
# 2. BUILD QUERY-LEVEL SUMMARY
# ------------------------------------------------------------

query_summary = (
    metrics
    .groupby(
        [
            "query_id",
            "query_type",
            "expected_element_count"
        ],
        as_index=False
    )
    .agg(
        run_count=(
            "query_id",
            "size"
        ),
        full_hit_at_1_count=(
            "full_hit_at_1",
            "sum"
        ),
        full_hit_at_1_rate=(
            "full_hit_at_1",
            "mean"
        ),
        full_hit_at_3_count=(
            "full_hit_at_3",
            "sum"
        ),
        full_hit_at_3_rate=(
            "full_hit_at_3",
            "mean"
        ),
        mean_coverage_at_3=(
            "coverage_at_3",
            "mean"
        ),
        min_coverage_at_3=(
            "coverage_at_3",
            "min"
        ),
        max_coverage_at_3=(
            "coverage_at_3",
            "max"
        )
    )
)

# ------------------------------------------------------------
# 3. ADD DISPLAY PERCENTAGES
# ------------------------------------------------------------

query_summary["full_hit_at_1_percent"] = (
    query_summary["full_hit_at_1_rate"] * 100
).round(2)

query_summary["full_hit_at_3_percent"] = (
    query_summary["full_hit_at_3_rate"] * 100
).round(2)

query_summary["mean_coverage_at_3_percent"] = (
    query_summary["mean_coverage_at_3"] * 100
).round(2)

# ------------------------------------------------------------
# 4. ORDER BY QUERY ID
# ------------------------------------------------------------

query_summary = (
    query_summary
    .sort_values("query_id")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 5. VALIDATE
# ------------------------------------------------------------

assert len(query_summary) == 12

assert (
    query_summary["run_count"] == 18
).all()

assert (
    query_summary["full_hit_at_1_count"]
    <= query_summary["full_hit_at_3_count"]
).all()

assert (
    query_summary["mean_coverage_at_3"]
    .between(0, 1)
    .all()
)

# ------------------------------------------------------------
# 6. SAVE
# ------------------------------------------------------------

query_summary.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 7. DISPLAY QUERY-LEVEL RESULTS
# ------------------------------------------------------------

display_cols = [
    "query_id",
    "query_type",
    "expected_element_count",
    "full_hit_at_1_count",
    "full_hit_at_1_percent",
    "full_hit_at_3_count",
    "full_hit_at_3_percent",
    "mean_coverage_at_3_percent",
    "min_coverage_at_3",
    "max_coverage_at_3",
]

print("\n2. RESULTS BY QUERY")
print("-" * 95)

print(
    query_summary[display_cols]
    .to_string(index=False)
)

# ------------------------------------------------------------
# 8. CHECK TOTALS AGAINST CELLS 48–50
# ------------------------------------------------------------

total_hit1 = int(
    query_summary["full_hit_at_1_count"].sum()
)

total_hit3 = int(
    query_summary["full_hit_at_3_count"].sum()
)

overall_coverage = (
    metrics["coverage_at_3"].mean()
)

print("\n3. CONSISTENCY CHECK")
print("-" * 95)

print(
    "Full Hit@1 total:",
    total_hit1,
    "/ 216"
)

print(
    "Full Hit@3 total:",
    total_hit3,
    "/ 216"
)

print(
    "Overall Mean Coverage@3:",
    round(overall_coverage, 4)
)

assert total_hit1 == 120
assert total_hit3 == 175

assert abs(
    overall_coverage - 0.8677
) < 0.0001

# ------------------------------------------------------------
# 9. IDENTIFY INCOMPLETE TOP-3 RUNS
#    Diagnostic only — not a ranking
# ------------------------------------------------------------

incomplete = metrics[
    metrics["full_hit_at_3"] == 0
].copy()

print("\n4. INCOMPLETE TOP-3 RETRIEVALS")
print("-" * 95)

print(
    "Incomplete query-condition runs:",
    len(incomplete),
    "/ 216"
)

incomplete_by_query = (
    incomplete
    .groupby("query_id")
    .size()
    .reindex(
        query_summary["query_id"],
        fill_value=0
    )
)

print("\nIncomplete runs by query:")

print(
    incomplete_by_query.to_string()
)

assert len(incomplete) == 41

# ------------------------------------------------------------
# 10. FINAL STATUS
# ------------------------------------------------------------

print("\nOutput file:")
print(OUTPUT_PATH)

print("\n" + "=" * 95)

print(
    "QUERY ANALYSIS STATUS: PASS — "
    "12 QUERIES SUMMARIZED ACROSS 18 CONDITIONS"
)

print("=" * 95)

QUERY-LEVEL RETRIEVAL ANALYSIS

1. QUERY STRUCTURE
-----------------------------------------------------------------------------------------------
Queries: 12
Runs per query: 18 to 18

2. RESULTS BY QUERY
-----------------------------------------------------------------------------------------------
query_id               query_type  expected_element_count  full_hit_at_1_count  full_hit_at_1_percent  full_hit_at_3_count  full_hit_at_3_percent  mean_coverage_at_3_percent  min_coverage_at_3  max_coverage_at_3
     Q01            Direct/Narrow                       1                   16                  88.89                   18                 100.00                      100.00           1.000000                1.0
     Q02            Direct/Narrow                       2                   17                  94.44                   18                 100.00                      100.00           1.000000                1.0
     Q03        Direct/Relational                       3      

In [56]:
# ============================================================
# CELL 52 — FULL HIT@3 DIAGNOSTIC MATRIX
# ============================================================

import os
import pandas as pd

METRICS_PATH = (
    "/content/retrieval-experiments/evaluation/"
    "query_condition_metrics.csv"
)

OUTPUT_PATH = (
    "/content/retrieval-experiments/evaluation/"
    "full_hit_at_3_diagnostic_matrix.csv"
)

metrics = pd.read_csv(METRICS_PATH)

print("FULL HIT@3 DIAGNOSTIC MATRIX")
print("=" * 105)

assert len(metrics) == 216

# ------------------------------------------------------------
# 1. CREATE CONDITION LABEL
# ------------------------------------------------------------

metrics["condition"] = (
    metrics["configuration"].astype(str)
    + " | "
    + metrics["overlap_setting"].astype(str)
    + " | "
    + metrics["model_id"].astype(str)
)

# ------------------------------------------------------------
# 2. VALIDATE 18 CONDITIONS
# ------------------------------------------------------------

conditions = (
    metrics[
        [
            "configuration",
            "overlap_setting",
            "model_id",
            "condition"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "configuration",
            "overlap_setting",
            "model_id"
        ]
    )
)

print("\n1. CONDITION STRUCTURE")
print("-" * 105)

print("Conditions:", len(conditions))

assert len(conditions) == 18

# ------------------------------------------------------------
# 3. BUILD FULL HIT@3 MATRIX
# ------------------------------------------------------------

hit3_matrix = (
    metrics
    .pivot(
        index="condition",
        columns="query_id",
        values="full_hit_at_3"
    )
)

query_order = [
    f"Q{i:02d}"
    for i in range(1, 13)
]

hit3_matrix = hit3_matrix[
    query_order
]

# Put conditions in controlled order
condition_order = conditions["condition"].tolist()

hit3_matrix = hit3_matrix.reindex(
    condition_order
)

# ------------------------------------------------------------
# 4. ADD CONDITION TOTAL
# ------------------------------------------------------------

hit3_matrix["Total_Hit3"] = (
    hit3_matrix[query_order]
    .sum(axis=1)
)

# ------------------------------------------------------------
# 5. VALIDATE MATRIX
# ------------------------------------------------------------

assert hit3_matrix.shape == (18, 13)

assert (
    hit3_matrix[query_order]
    .isin([0, 1])
    .all()
    .all()
)

assert (
    hit3_matrix["Total_Hit3"].sum()
    == 175
)

# ------------------------------------------------------------
# 6. DISPLAY MATRIX
# ------------------------------------------------------------

print("\n2. FULL HIT@3 BY CONDITION AND QUERY")
print("-" * 105)

print(
    hit3_matrix
    .to_string()
)

# ------------------------------------------------------------
# 7. FAILURE COUNTS BY CONDITION
# ------------------------------------------------------------

failure_by_condition = (
    12 - hit3_matrix["Total_Hit3"]
)

print("\n3. INCOMPLETE TOP-3 RUNS BY CONDITION")
print("-" * 105)

for condition, failures in failure_by_condition.items():
    print(
        f"{condition}: {int(failures)} incomplete run(s)"
    )

print(
    "\nTotal incomplete runs:",
    int(failure_by_condition.sum())
)

assert int(failure_by_condition.sum()) == 41

# ------------------------------------------------------------
# 8. FAILURE COUNTS BY QUERY
# ------------------------------------------------------------

failure_by_query = (
    18 - hit3_matrix[query_order].sum(axis=0)
)

print("\n4. INCOMPLETE TOP-3 RUNS BY QUERY")
print("-" * 105)

print(
    failure_by_query.to_string()
)

assert int(failure_by_query.sum()) == 41

# ------------------------------------------------------------
# 9. SAVE MATRIX
# ------------------------------------------------------------

output = (
    hit3_matrix
    .reset_index()
)

output.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("\n5. OUTPUT FILE")
print("-" * 105)

print(OUTPUT_PATH)

# ------------------------------------------------------------
# 10. FINAL STATUS
# ------------------------------------------------------------

print("\n" + "=" * 105)

print(
    "DIAGNOSTIC MATRIX STATUS: PASS — "
    "FULL HIT@3 PATTERN MAPPED ACROSS "
    "18 CONDITIONS × 12 QUERIES"
)

print("=" * 105)

FULL HIT@3 DIAGNOSTIC MATRIX

1. CONDITION STRUCTURE
---------------------------------------------------------------------------------------------------------
Conditions: 18

2. FULL HIT@3 BY CONDITION AND QUERY
---------------------------------------------------------------------------------------------------------
query_id              Q01  Q02  Q03  Q04  Q05  Q06  Q07  Q08  Q09  Q10  Q11  Q12  Total_Hit3
condition                                                                                   
A_Fine | O0 | E1        1    1    1    0    1    0    1    0    1    1    0    1           8
A_Fine | O0 | E2        1    1    0    1    1    0    1    1    1    1    0    1           9
A_Fine | O0 | E3        1    1    0    1    1    0    1    1    1    1    0    1           9
A_Fine | O1 | E1        1    1    1    1    1    0    1    1    1    1    0    1          10
A_Fine | O1 | E2        1    1    0    1    1    0    1    1    1    1    1    1          10
A_Fine | O1 | E3        1    1 

In [57]:
# ============================================================
# CELL 53 — SELECTION EVIDENCE TABLE
# ============================================================

import os
import pandas as pd

METRICS_PATH = (
    "/content/retrieval-experiments/evaluation/"
    "query_condition_metrics.csv"
)

OUTPUT_PATH = (
    "/content/retrieval-experiments/evaluation/"
    "selection_evidence_table.csv"
)

metrics = pd.read_csv(METRICS_PATH)

print("SELECTION EVIDENCE TABLE")
print("=" * 100)

condition_cols = [
    "configuration",
    "overlap_setting",
    "model_id"
]

# ------------------------------------------------------------
# 1. BUILD CONDITION EVIDENCE
# ------------------------------------------------------------

selection = (
    metrics
    .groupby(
        condition_cols,
        as_index=False
    )
    .agg(
        query_count=(
            "query_id",
            "size"
        ),

        full_hit_at_1_count=(
            "full_hit_at_1",
            "sum"
        ),

        full_hit_at_3_count=(
            "full_hit_at_3",
            "sum"
        ),

        mean_coverage_at_3=(
            "coverage_at_3",
            "mean"
        ),

        min_coverage_at_3=(
            "coverage_at_3",
            "min"
        )
    )
)

# ------------------------------------------------------------
# 2. DERIVE DIAGNOSTIC COUNTS
# ------------------------------------------------------------

selection["incomplete_top3_count"] = (
    selection["query_count"]
    - selection["full_hit_at_3_count"]
)

selection["full_hit_at_1_percent"] = (
    selection["full_hit_at_1_count"]
    / selection["query_count"]
    * 100
).round(2)

selection["full_hit_at_3_percent"] = (
    selection["full_hit_at_3_count"]
    / selection["query_count"]
    * 100
).round(2)

selection["mean_coverage_at_3_percent"] = (
    selection["mean_coverage_at_3"]
    * 100
).round(2)

selection["min_coverage_at_3_percent"] = (
    selection["min_coverage_at_3"]
    * 100
).round(2)

# ------------------------------------------------------------
# 3. ADD CONDITION LABEL
# ------------------------------------------------------------

selection["condition"] = (
    selection["configuration"]
    + " | "
    + selection["overlap_setting"]
    + " | "
    + selection["model_id"]
)

# ------------------------------------------------------------
# 4. CONTROLLED DISPLAY ORDER
#
# This is NOT a weighted score.
# Sorting is only for inspection:
#   1) Full Hit@3
#   2) Mean Coverage@3
#   3) Full Hit@1
# ------------------------------------------------------------

selection_display = (
    selection
    .sort_values(
        by=[
            "full_hit_at_3_count",
            "mean_coverage_at_3",
            "full_hit_at_1_count"
        ],
        ascending=[
            False,
            False,
            False
        ]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 5. VALIDATION
# ------------------------------------------------------------

assert len(selection) == 18

assert (
    selection["query_count"] == 12
).all()

assert (
    selection["full_hit_at_1_count"]
    <= selection["full_hit_at_3_count"]
).all()

assert (
    selection["incomplete_top3_count"]
    ==
    12 - selection["full_hit_at_3_count"]
).all()

assert (
    selection["mean_coverage_at_3"]
    .between(0, 1)
    .all()
)

# ------------------------------------------------------------
# 6. DISPLAY
# ------------------------------------------------------------

display_cols = [
    "condition",
    "full_hit_at_1_count",
    "full_hit_at_1_percent",
    "full_hit_at_3_count",
    "full_hit_at_3_percent",
    "mean_coverage_at_3_percent",
    "min_coverage_at_3_percent",
    "incomplete_top3_count",
]

print("\n1. CONDITION EVIDENCE")
print("-" * 100)

print(
    selection_display[display_cols]
    .to_string(index=False)
)

# ------------------------------------------------------------
# 7. CHECK COMPLETE TOP-3 CONDITIONS
# ------------------------------------------------------------

complete_conditions = selection[
    selection["full_hit_at_3_count"] == 12
].copy()

print("\n2. CONDITIONS WITH COMPLETE FULL HIT@3")
print("-" * 100)

if len(complete_conditions) == 0:
    print("None")
else:
    print(
        complete_conditions[display_cols]
        .to_string(index=False)
    )

# ------------------------------------------------------------
# 8. CHECK PERFECT MEAN COVERAGE
# ------------------------------------------------------------

perfect_coverage = selection[
    selection["mean_coverage_at_3"] == 1.0
].copy()

print("\n3. CONDITIONS WITH 100% MEAN COVERAGE@3")
print("-" * 100)

if len(perfect_coverage) == 0:
    print("None")
else:
    print(
        perfect_coverage[display_cols]
        .to_string(index=False)
    )

# ------------------------------------------------------------
# 9. SAVE
# ------------------------------------------------------------

selection_display.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("\n4. OUTPUT FILE")
print("-" * 100)

print(OUTPUT_PATH)

# ------------------------------------------------------------
# 10. FINAL STATUS
# ------------------------------------------------------------

print("\n" + "=" * 100)

print(
    "SELECTION EVIDENCE STATUS: PASS — "
    "18 CONDITIONS COMPARED WITHOUT A WEIGHTED COMPOSITE SCORE"
)

print("=" * 100)

SELECTION EVIDENCE TABLE

1. CONDITION EVIDENCE
----------------------------------------------------------------------------------------------------
           condition  full_hit_at_1_count  full_hit_at_1_percent  full_hit_at_3_count  full_hit_at_3_percent  mean_coverage_at_3_percent  min_coverage_at_3_percent  incomplete_top3_count
B_Baseline | O0 | E1                    7                  58.33                   12                 100.00                      100.00                     100.00                      0
B_Baseline | O0 | E3                    9                  75.00                   11                  91.67                       97.22                      66.67                      1
   C_Broad | O0 | E3                    7                  58.33                   11                  91.67                       97.22                      66.67                      1
   C_Broad | O1 | E3                    7                  58.33                   11                  

In [58]:
# ============================================================
# CELL 54 — FINAL SELECTED-CONDITION VERIFICATION
# ============================================================

import pandas as pd

METRICS_PATH = (
    "/content/retrieval-experiments/evaluation/"
    "query_condition_metrics.csv"
)

metrics = pd.read_csv(METRICS_PATH)

print("FINAL SELECTED-CONDITION VERIFICATION")
print("=" * 90)

# ------------------------------------------------------------
# Selected experimental condition
# ------------------------------------------------------------

selected = metrics[
    (metrics["configuration"] == "B_Baseline")
    & (metrics["overlap_setting"] == "O0")
    & (metrics["model_id"] == "E1")
].copy()

selected = (
    selected
    .sort_values("query_id")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 1. STRUCTURE CHECK
# ------------------------------------------------------------

print("\n1. STRUCTURE CHECK")
print("-" * 90)

print("Rows:", len(selected))
print(
    "Queries:",
    selected["query_id"].nunique()
)

assert len(selected) == 12
assert selected["query_id"].nunique() == 12

# ------------------------------------------------------------
# 2. QUERY-BY-QUERY RESULTS
# ------------------------------------------------------------

display_cols = [
    "query_id",
    "query_type",
    "expected_element_count",
    "rank1_recovered_count",
    "top3_recovered_count",
    "full_hit_at_1",
    "full_hit_at_3",
    "coverage_at_3",
]

print("\n2. QUERY-BY-QUERY RESULTS")
print("-" * 90)

print(
    selected[display_cols]
    .to_string(index=False)
)

# ------------------------------------------------------------
# 3. FINAL VERIFICATION
# ------------------------------------------------------------

hit1_count = int(
    selected["full_hit_at_1"].sum()
)

hit3_count = int(
    selected["full_hit_at_3"].sum()
)

mean_coverage = (
    selected["coverage_at_3"].mean()
)

min_coverage = (
    selected["coverage_at_3"].min()
)

print("\n3. FINAL VERIFICATION")
print("-" * 90)

print(
    "Full Hit@1:",
    hit1_count,
    "/ 12"
)

print(
    "Full Hit@3:",
    hit3_count,
    "/ 12"
)

print(
    "Mean Coverage@3:",
    round(mean_coverage, 4)
)

print(
    "Minimum Coverage@3:",
    round(min_coverage, 4)
)

# ------------------------------------------------------------
# 4. ASSERT EXPECTED SELECTION EVIDENCE
# ------------------------------------------------------------

assert hit1_count == 7
assert hit3_count == 12
assert mean_coverage == 1.0
assert min_coverage == 1.0

assert (
    selected["top3_recovered_count"]
    == selected["expected_element_count"]
).all()

print("\n" + "=" * 90)

print(
    "SELECTED CONDITION VERIFICATION: PASS — "
    "B_BASELINE + O0 + E1 RECOVERED ALL EXPECTED "
    "ELEMENTS WITHIN TOP-3 FOR ALL 12 QUERIES"
)

print("=" * 90)

FINAL SELECTED-CONDITION VERIFICATION

1. STRUCTURE CHECK
------------------------------------------------------------------------------------------
Rows: 12
Queries: 12

2. QUERY-BY-QUERY RESULTS
------------------------------------------------------------------------------------------
query_id               query_type  expected_element_count  rank1_recovered_count  top3_recovered_count  full_hit_at_1  full_hit_at_3  coverage_at_3
     Q01            Direct/Narrow                       1                      1                     1              1              1            1.0
     Q02            Direct/Narrow                       2                      2                     2              1              1            1.0
     Q03        Direct/Relational                       3                      3                     3              1              1            1.0
     Q04            Direct/Narrow                       3                      0                     3              0   

In [59]:
# ============================================================
# CELL 55 — CHECK RETRIEVAL EXPERIMENT ARTIFACTS
# ============================================================

import os

EVALUATION_DIR = (
    "/content/retrieval-experiments/evaluation"
)

print("RETRIEVAL EVALUATION ARTIFACTS")
print("=" * 80)

if not os.path.exists(EVALUATION_DIR):
    print("ERROR: Evaluation directory does not exist:")
    print(EVALUATION_DIR)

else:
    files = sorted(os.listdir(EVALUATION_DIR))

    print("Directory:")
    print(EVALUATION_DIR)

    print("\nFiles found:", len(files))
    print("-" * 80)

    for i, filename in enumerate(files, start=1):

        path = os.path.join(
            EVALUATION_DIR,
            filename
        )

        size = os.path.getsize(path)

        print(
            f"{i:02d}. {filename} "
            f"({size:,} bytes)"
        )

print("\n" + "=" * 80)

RETRIEVAL EVALUATION ARTIFACTS
Directory:
/content/retrieval-experiments/evaluation

Files found: 10
--------------------------------------------------------------------------------
01. annotated_retrieval_results.csv (960,896 bytes)
02. condition_level_summary.csv (1,967 bytes)
03. factor_summary_configuration.csv (502 bytes)
04. factor_summary_embedding_model.csv (466 bytes)
05. factor_summary_overlap.csv (382 bytes)
06. final_human_element_annotations.csv (884,229 bytes)
07. full_hit_at_3_diagnostic_matrix.csv (886 bytes)
08. query_condition_metrics.csv (11,060 bytes)
09. query_level_summary.csv (1,496 bytes)
10. selection_evidence_table.csv (1,841 bytes)



In [60]:
# ============================================================
# CELL 56 — INSPECT RETRIEVAL EXPERIMENTS DIRECTORY STRUCTURE
# ============================================================

import os

ROOT_DIR = "/content/retrieval-experiments"

print("RETRIEVAL EXPERIMENTS — DIRECTORY STRUCTURE")
print("=" * 90)

if not os.path.exists(ROOT_DIR):
    print("ERROR: Directory does not exist:")
    print(ROOT_DIR)

else:
    for root, dirs, files in os.walk(ROOT_DIR):

        # Ignore notebook/system cache folders
        dirs[:] = [
            d for d in dirs
            if d not in {
                ".ipynb_checkpoints",
                "__pycache__",
                ".git"
            }
        ]

        level = root.replace(
            ROOT_DIR, ""
        ).count(os.sep)

        indent = "    " * level

        folder_name = (
            os.path.basename(root)
            if root != ROOT_DIR
            else "retrieval-experiments"
        )

        print(f"{indent}📁 {folder_name}/")

        subindent = "    " * (level + 1)

        for filename in sorted(files):
            path = os.path.join(root, filename)
            size = os.path.getsize(path)

            print(
                f"{subindent}📄 {filename} "
                f"({size:,} bytes)"
            )

print("\n" + "=" * 90)
print("STRUCTURE INSPECTION COMPLETE")
print("=" * 90)

RETRIEVAL EXPERIMENTS — DIRECTORY STRUCTURE
📁 retrieval-experiments/
    📁 evaluation/
        📄 annotated_retrieval_results.csv (960,896 bytes)
        📄 condition_level_summary.csv (1,967 bytes)
        📄 factor_summary_configuration.csv (502 bytes)
        📄 factor_summary_embedding_model.csv (466 bytes)
        📄 factor_summary_overlap.csv (382 bytes)
        📄 final_human_element_annotations.csv (884,229 bytes)
        📄 full_hit_at_3_diagnostic_matrix.csv (886 bytes)
        📄 query_condition_metrics.csv (11,060 bytes)
        📄 query_level_summary.csv (1,496 bytes)
        📄 selection_evidence_table.csv (1,841 bytes)

STRUCTURE INSPECTION COMPLETE


In [61]:
# ============================================================
# CELL 57 — INSPECT RELEVANT ROOT ARTIFACTS
# ============================================================

import os

ROOT = "/content"

relevant_names = [
    "raw_retrieval_results.csv",
    "retrieval_run_summary.csv",
    "retrieval_queries.json",
    "retrieval_expected_elements.json",
    "retrieval_annotation_dataset.csv",
    "retrieval_annotation_workbook.xlsx",
    "element_level_annotation.csv",
    "unique_annotation_units.csv",
    "02_embedding_and_retrieval_experiments.ipynb",
]

print("ROOT RETRIEVAL ARTIFACT CHECK")
print("=" * 85)

found = []
missing = []

for filename in relevant_names:

    path = os.path.join(ROOT, filename)

    if os.path.exists(path):

        size = os.path.getsize(path)

        found.append(filename)

        print(
            f"FOUND   {filename} "
            f"({size:,} bytes)"
        )

    else:

        missing.append(filename)

        print(
            f"MISSING {filename}"
        )

print("\n" + "-" * 85)

print("Found:", len(found))
print("Missing:", len(missing))

print("\nFOUND FILES:")
for filename in found:
    print(" -", filename)

if missing:
    print("\nNOT FOUND IN /content:")
    for filename in missing:
        print(" -", filename)

print("\n" + "=" * 85)
print("ROOT ARTIFACT CHECK COMPLETE")
print("=" * 85)

ROOT RETRIEVAL ARTIFACT CHECK
FOUND   raw_retrieval_results.csv (941,505 bytes)
FOUND   retrieval_run_summary.csv (1,310 bytes)
FOUND   retrieval_queries.json (6,848 bytes)
FOUND   retrieval_expected_elements.json (9,232 bytes)
FOUND   retrieval_annotation_dataset.csv (1,089,644 bytes)
FOUND   retrieval_annotation_workbook.xlsx (202,578 bytes)
FOUND   element_level_annotation.csv (882,670 bytes)
FOUND   unique_annotation_units.csv (328,044 bytes)
MISSING 02_embedding_and_retrieval_experiments.ipynb

-------------------------------------------------------------------------------------
Found: 8
Missing: 1

FOUND FILES:
 - raw_retrieval_results.csv
 - retrieval_run_summary.csv
 - retrieval_queries.json
 - retrieval_expected_elements.json
 - retrieval_annotation_dataset.csv
 - retrieval_annotation_workbook.xlsx
 - element_level_annotation.csv
 - unique_annotation_units.csv

NOT FOUND IN /content:
 - 02_embedding_and_retrieval_experiments.ipynb

ROOT ARTIFACT CHECK COMPLETE


In [62]:
# ============================================================
# CELL 58 — ORGANIZE FINAL RETRIEVAL EXPERIMENT ARTIFACTS
# ============================================================

import os
import shutil

BASE = "/content/retrieval-experiments"

QUERIES_DIR = os.path.join(BASE, "queries")
RESULTS_DIR = os.path.join(BASE, "results")
EVALUATION_DIR = os.path.join(BASE, "evaluation")

# ------------------------------------------------------------
# 1. CREATE DIRECTORIES
# ------------------------------------------------------------

for directory in [
    QUERIES_DIR,
    RESULTS_DIR,
    EVALUATION_DIR
]:
    os.makedirs(directory, exist_ok=True)

# ------------------------------------------------------------
# 2. FILE MAPPING
# ------------------------------------------------------------

file_mapping = {

    # Frozen evaluation queries
    "/content/retrieval_queries.json":
        os.path.join(
            QUERIES_DIR,
            "retrieval_queries.json"
        ),

    "/content/retrieval_expected_elements.json":
        os.path.join(
            QUERIES_DIR,
            "retrieval_expected_elements.json"
        ),

    # Immutable raw retrieval outputs
    "/content/raw_retrieval_results.csv":
        os.path.join(
            RESULTS_DIR,
            "raw_retrieval_results.csv"
        ),

    "/content/retrieval_run_summary.csv":
        os.path.join(
            RESULTS_DIR,
            "retrieval_run_summary.csv"
        ),

    # Annotation/audit artifacts
    "/content/retrieval_annotation_dataset.csv":
        os.path.join(
            EVALUATION_DIR,
            "retrieval_annotation_dataset.csv"
        ),

    "/content/retrieval_annotation_workbook.xlsx":
        os.path.join(
            EVALUATION_DIR,
            "retrieval_annotation_workbook.xlsx"
        ),
}

# ------------------------------------------------------------
# 3. COPY FILES
# ------------------------------------------------------------

print("ORGANIZING RETRIEVAL EXPERIMENT ARTIFACTS")
print("=" * 85)

copied = 0

for source, destination in file_mapping.items():

    if not os.path.exists(source):
        print(
            "MISSING:",
            source
        )
        continue

    shutil.copy2(
        source,
        destination
    )

    copied += 1

    print(
        "COPIED:",
        os.path.basename(source)
    )

    print(
        "     ->",
        destination
    )

# ------------------------------------------------------------
# 4. VERIFY FINAL REQUIRED ARTIFACTS
# ------------------------------------------------------------

required_files = [

    # Queries
    os.path.join(
        QUERIES_DIR,
        "retrieval_queries.json"
    ),

    os.path.join(
        QUERIES_DIR,
        "retrieval_expected_elements.json"
    ),

    # Raw results
    os.path.join(
        RESULTS_DIR,
        "raw_retrieval_results.csv"
    ),

    os.path.join(
        RESULTS_DIR,
        "retrieval_run_summary.csv"
    ),

    # Annotation
    os.path.join(
        EVALUATION_DIR,
        "retrieval_annotation_dataset.csv"
    ),

    os.path.join(
        EVALUATION_DIR,
        "retrieval_annotation_workbook.xlsx"
    ),

    os.path.join(
        EVALUATION_DIR,
        "final_human_element_annotations.csv"
    ),

    # Derived evaluation
    os.path.join(
        EVALUATION_DIR,
        "annotated_retrieval_results.csv"
    ),

    os.path.join(
        EVALUATION_DIR,
        "query_condition_metrics.csv"
    ),

    os.path.join(
        EVALUATION_DIR,
        "condition_level_summary.csv"
    ),

    os.path.join(
        EVALUATION_DIR,
        "factor_summary_configuration.csv"
    ),

    os.path.join(
        EVALUATION_DIR,
        "factor_summary_overlap.csv"
    ),

    os.path.join(
        EVALUATION_DIR,
        "factor_summary_embedding_model.csv"
    ),

    os.path.join(
        EVALUATION_DIR,
        "query_level_summary.csv"
    ),

    os.path.join(
        EVALUATION_DIR,
        "full_hit_at_3_diagnostic_matrix.csv"
    ),

    os.path.join(
        EVALUATION_DIR,
        "selection_evidence_table.csv"
    ),
]

print("\nFINAL ARTIFACT CHECK")
print("-" * 85)

missing = []

for path in required_files:

    relative = os.path.relpath(
        path,
        BASE
    )

    if os.path.exists(path):

        size = os.path.getsize(path)

        print(
            f"PASS  {relative} "
            f"({size:,} bytes)"
        )

    else:

        missing.append(relative)

        print(
            f"FAIL  {relative}"
        )

# ------------------------------------------------------------
# 5. FINAL STATUS
# ------------------------------------------------------------

print("\n" + "=" * 85)

print(
    "Files copied:",
    copied,
    "/",
    len(file_mapping)
)

if not missing:

    print(
        "ARTIFACT ORGANIZATION STATUS: PASS — "
        "ALL REQUIRED RETRIEVAL ARTIFACTS PRESENT"
    )

else:

    print(
        "ARTIFACT ORGANIZATION STATUS: REVIEW REQUIRED"
    )

    print("\nMissing:")
    for item in missing:
        print(" -", item)

print("=" * 85)

ORGANIZING RETRIEVAL EXPERIMENT ARTIFACTS
COPIED: retrieval_queries.json
     -> /content/retrieval-experiments/queries/retrieval_queries.json
COPIED: retrieval_expected_elements.json
     -> /content/retrieval-experiments/queries/retrieval_expected_elements.json
COPIED: raw_retrieval_results.csv
     -> /content/retrieval-experiments/results/raw_retrieval_results.csv
COPIED: retrieval_run_summary.csv
     -> /content/retrieval-experiments/results/retrieval_run_summary.csv
COPIED: retrieval_annotation_dataset.csv
     -> /content/retrieval-experiments/evaluation/retrieval_annotation_dataset.csv
COPIED: retrieval_annotation_workbook.xlsx
     -> /content/retrieval-experiments/evaluation/retrieval_annotation_workbook.xlsx

FINAL ARTIFACT CHECK
-------------------------------------------------------------------------------------
PASS  queries/retrieval_queries.json (6,848 bytes)
PASS  queries/retrieval_expected_elements.json (9,232 bytes)
PASS  results/raw_retrieval_results.csv (941,505 b